### Роль RDKit в работе

**RDKit** используется как инструмент для работы с химической структурой молекул. В нашем проекте он выполняет три основные задачи:

1. **Проверка SMILES** — определяет, соответствует ли строка корректной химической структуре.
2. **Работа со структурой молекулы** — позволяет преобразовать SMILES в объект молекулы и анализировать его.
3. **Проверка результатов VAE** — после генерации новых SMILES с помощью VAE RDKit позволит определить, какие из них являются валидными молекулами.

Таким образом, **VAE отвечает за генерацию новых молекулярных структур, а RDKit — за их химическую обработку и проверку**.


In [ ]:
from datasets import load_dataset

ds = load_dataset("edmanft/zinc250k")

### Описание данных датасета ZINC250K

Датасет содержит около 250 тысяч молекул. Для каждой молекулы представлены структурные представления и несколько рассчитанных молекулярных свойств.

* **`index`** — порядковый индекс записи в датасете. Не является химическим свойством молекулы и не используется в качестве входного признака для VAE.

* **`smiles`** — структурное представление молекулы в формате **SMILES (Simplified Molecular Input Line Entry System)**. Молекула записывается в виде строки символов, например `CCO`. В рамках данной работы `SMILES` является основным представлением молекул, которое будет использоваться для обучения генеративной модели.

* **`selfies`** — представление молекулы в формате **SELFIES (Self-Referencing Embedded Strings)**. Это альтернативный способ кодирования молекулярной структуры, разработанный с учётом получения синтаксически корректных молекулярных строк при генерации. В данной работе этот столбец может использоваться для сравнения с представлением SMILES.

* **`logP`** — коэффициент распределения вещества между органической и водной фазами в логарифмической шкале. Используется как характеристика **липофильности (гидрофобности)** молекулы. Чем выше значение `logP`, тем в среднем более липофильной является молекула.

* **`qed`** — количественная оценка **drug-likeness**, то есть соответствия молекулы некоторым свойствам, характерным для лекарственных соединений. Значение рассчитывается по совокупности молекулярных характеристик. Обычно находится в диапазоне от 0 до 1, где большее значение соответствует более высокому показателю drug-likeness.

* **`SAS`** — **Synthetic Accessibility Score**, оценка сложности синтеза молекулы. Показатель характеризует, насколько молекулу сложно получить химическим синтезом с учётом особенностей её структуры. Меньшее значение обычно соответствует более простой синтетической доступности, а большее — более сложной.

В данной работе столбец **`smiles`** будет использоваться как основное входное представление молекул для обучения VAE. Столбцы `logP`, `qed` и `SAS` будут использоваться для анализа свойств исходных и сгенерированных молекул. Столбцы `index` и, на первом этапе, `selfies` непосредственно в качестве входа стандартного VAE использоваться не будут.


Липофильность (гидрофобность) — это способность вещества лучше растворяться в жирах и органических растворителях, чем в воде.

Проще:

высокая липофильность → молекула плохо взаимодействует с водой, лучше — с жирами;
низкая липофильность → молекула лучше взаимодействует с водой.

Для logP: чем выше logP, тем более липофильной обычно является молекула.

                         ZINC250K
                            │
              ┌─────────────┼─────────────┐
              │             │             │
           SELFIES      Fingerprint    Descriptors
              │             │             │
        Transformer          MLP           MLP
              │             │             │
              └─────────────┼─────────────┘
                            │
                         Fusion
                            │
                            ▼
                       clean latent
                            │
                     Forward diffusion
                            │
                     x₀ → x₁ → ... → xT
                            │
                            ▼
                         noise xT
                            │
             ┌──────────────┴──────────────┐
             │                             │
      Target properties              timestep t
             │                             │
             └──────────────┬──────────────┘
                            ▼
                 Conditional Diffusion
                       Transformer
                            │
                  iterative denoising
                            │
                            ▼
                       generated x₀
                            │
                            ▼
                         SELFIES
                            │
                            ▼
                          SMILES

==============================================================================================================================
==============================================================================================================================


In [ ]:
import pandas as pd

df = ds["train"].to_pandas()

print("Размер:", df.shape)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Количество строк:", len(df))
print("\nПропуски:")
print(df.isnull().sum())

print("\nДубликаты SMILES:", df["smiles"].duplicated().sum())
print("Уникальных SMILES:", df["smiles"].nunique())

In [ ]:
Посмотрим табличные QED и SAS и сравним с расчетными

In [ ]:
from rdkit import Chem
from rdkit.Chem import QED
import pandas as pd

df = pd.read_csv("zinc250k.csv")

def calculate_qed(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return QED.qed(mol)

df["qed_calculated"] = df["smiles"].apply(calculate_qed)

df["qed_difference"] = (
    df["qed"] - df["qed_calculated"]
).abs()

print(df[["qed", "qed_calculated", "qed_difference"]].head())

print("Средняя ошибка:",
      df["qed_difference"].mean())

print("Максимальная ошибка:",
      df["qed_difference"].max())

print("Корреляция:",
      df[["qed", "qed_calculated"]].corr().iloc[0, 1])


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(
    df["qed"],
    df["qed_calculated"],
    s=3,
    alpha=0.3
)

plt.xlabel("QED в датасете")
plt.ylabel("QED, рассчитанный RDKit")
plt.plot([0, 1], [0, 1], "r--")

plt.show()


In [ ]:
from rdkit import Chem
import sys

sys.path.append("path/to/rdkit/Contrib/SA_Score")

import sascorer

smiles = "CCO"

mol = Chem.MolFromSmiles(smiles)

sa = sascorer.calculateScore(mol)

print(sa)


In [ ]:
def calculate_sa(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return sascorer.calculateScore(mol)

df["SAS_calculated"] = df["smiles"].apply(calculate_sa)

df["SAS_difference"] = (
    df["SAS"] - df["SAS_calculated"]
).abs()


In [ ]:
print("Средняя абсолютная ошибка:",
      df["SAS_difference"].mean())

print("Максимальная ошибка:",
      df["SAS_difference"].max())

print(
    "Корреляция:",
    df[["SAS", "SAS_calculated"]].corr().iloc[0, 1]
)


EDA

In [ ]:
import matplotlib.pyplot as plt

df["logP"].hist(bins=50)
plt.xlabel("logP")
plt.ylabel("Количество молекул")
plt.title("Распределение logP")
plt.show()

In [ ]:
df["qed"].hist(bins=50)
plt.xlabel("qed")
plt.ylabel("Количество молекул")
plt.title("Распределение qed")
plt.show()

In [ ]:
df["SAS"].hist(bins=50)
plt.xlabel("SAS")
plt.ylabel("Количество молекул")
plt.title("Распределение SAS")
plt.show()

In [ ]:
df["smiles_length"] = df["smiles"].str.len()

print(df["smiles_length"].describe())

In [ ]:
df["smiles_length"].hist(bins=50)

plt.xlabel("Длина SMILES, символов")
plt.ylabel("Количество молекул")
plt.title("Распределение длины SMILES")
plt.show()

### Анализ длины SMILES

Для определения размера последовательности, используемой при обучении VAE, была рассчитана длина строк SMILES для всех молекул датасета.

Анализ распределения длины позволяет определить характерный размер молекулярных последовательностей и выбрать подходящее максимальное значение длины последовательности при последующей токенизации данных.


### Результаты анализа длины SMILES

Средняя длина строк SMILES в датасете составляет **45,3 символа**, медианное значение — **45 символов**. Половина молекул имеет длину от **39 до 51 символа**.

Минимальная длина SMILES составляет **10 символов**, максимальная — **110 символов**. Таким образом, большинство молекул имеет относительно небольшую длину, однако в датасете присутствуют более длинные молекулярные структуры.

Полученное распределение необходимо учитывать при выборе максимальной длины последовательности для VAE. При токенизации SMILES длина последовательности может отличаться от длины исходной строки, поэтому окончательный размер последовательности будет выбран после токенизации.


### Проверка корректности SMILES

С помощью библиотеки RDKit была выполнена проверка корректности SMILES всех **249 455 молекул** датасета.

Все **249 455 SMILES** успешно преобразованы RDKit в химические структуры. Некорректных SMILES не обнаружено.

Таким образом, все молекулы исходного датасета прошли проверку корректности и могут быть использованы для дальнейшей подготовки данных и обучения VAE.


In [ ]:
from rdkit import Chem
from tqdm.auto import tqdm

valid_count = 0

for smiles in tqdm(df["smiles"]):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        valid_count += 1

print("Всего SMILES:", len(df))
print("Корректных SMILES:", valid_count)
print("Некорректных SMILES:", len(df) - valid_count)

Проверяем нет ли не допустимых элементов:
Допустимые элементты "C", "N", "O", "S", "P", "F", "Cl", "Br", "I"
### Проверка допустимых элементов

Проверяется состав молекулы и наличие элементов, не входящих в заранее заданный список допустимых элементов.

- `atom.GetSymbol()` — получает символ химического элемента.
- `elements.issubset(allowed_elements)` — проверяет, входят ли все элементы молекулы в разрешённый набор.
- `True` — все элементы допустимы.
- `False` — присутствует хотя бы один элемент вне заданного набора.

Такой фильтр позволяет ограничить химическое пространство датасета и исключить молекулы с нежелательными для конкретной задачи элементами.


In [ ]:
allowed_elements = {
    "C", "N", "O", "S", "P",
    "F", "Cl", "Br", "I"
}
def check_elements(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return False

    elements = {
        atom.GetSymbol()
        for atom in mol.GetAtoms()
    }

    return elements.issubset(allowed_elements)


df["valid_elements"] = df["smiles"].apply(check_elements)


## Стандартизация молекулярных структур

Перед обучением VAE выполняется стандартизация SMILES. Цель этого этапа — уменьшить вариативность представления молекул и привести структуры к более однородному виду.

### 1. Удаление солей

В SMILES точка `.` разделяет отдельные компоненты структуры.

Например:

```text
CC[NH3+].[Cl-]
```

Здесь:

* `CC[NH3+]` — основной молекулярный фрагмент;
* `[Cl-]` — противоион.

Для генеративной модели наличие солей увеличивает количество вариантов представления и усложняет задачу генерации.

Поэтому из структуры удаляются небольшие сопутствующие компоненты, такие как противоионы, оставляя основной молекулярный фрагмент.

### 2. Нейтрализация зарядов

Одна и та же химическая структура может иметь разные формы представления с различными формальными зарядами.

Например:

```text
CC[NH3+]
```

может быть представлена в нейтральной форме:

```text
CCN
```

Для VAE это две разные последовательности токенов, поэтому наличие большого количества различных зарядовых форм увеличивает вариативность обучающей выборки.

Нейтрализация зарядов позволяет привести подходящие структуры к более единому представлению:

```text
различные формы представления
            ↓
   стандартизация структуры
            ↓
 единое представление SMILES
            ↓
 более однородные данные
            ↓
       обучение VAE
```

> **Важно:** нейтрализация не означает, что заряженная форма химически неправильна. Формальный заряд может быть важной частью реальной структуры и её свойств. В данном проекте нейтрализация используется как этап стандартизации представления данных для генеративной модели.

### 3. Каноникализация SMILES

После удаления солей и нейтрализации структура преобразуется в канонический SMILES.

Это позволяет уменьшить количество различных строковых представлений одной и той же молекулы.

Таким образом, общий этап стандартизации имеет вид:

**Исходный SMILES → удаление солей → нейтрализация зарядов → каноникализация → удаление дубликатов**

Основная цель этапа — получить более однородную обучающую выборку перед обучением VAE.


### Канонизация SMILES

Канонизация приводит различные записи одной и той же молекулы к единому стандартному представлению SMILES. Это позволяет корректно выявлять и удалять дубликаты молекул перед дальнейшей обработкой датасета.

### Стандартизация SMILES

Функция выполняет следующие операции:

1. **`Chem.MolFromSmiles(smiles)` — преобразование SMILES в молекулу RDKit**
   - Проверяет, можно ли интерпретировать строку как химическую структуру.
   - Не удаляет атомы или фрагменты.
   - При некорректном SMILES возвращает `None`.

2. **`remover.StripMol(mol, dontRemoveEverything=True)` — удаление солей**
   - Удаляет фрагменты, распознаваемые `SaltRemover` как соли/контр-ионы.
   - Например, из записи вида `CC[NH3+].[Cl-]` удаляет `Cl-`.
   - Может удалять другие неорганические ионы и небольшие солевые компоненты, входящие в стандартный набор RDKit.
   - `dontRemoveEverything=True` не позволяет удалить всю молекулу, если все её фрагменты распознаны как соли.
   - **Не удаляет произвольные органические фрагменты только потому, что они маленькие.**

3. **`uncharger.uncharge(mol)` — нейтрализация зарядов**
   - Пытается убрать формальные положительные и отрицательные заряды там, где это химически возможно.
   - Например, некоторые протонированные/депротонированные функциональные группы переводятся в нейтральную форму.
   - **Не удаляет атомы или молекулярные фрагменты.**
   - Заряды, которые необходимы для валентности структуры (например, у некоторых четвертичных аммониевых соединений), сохраняются.

4. **`Chem.SanitizeMol(mol)` — санитизация**
   - Ничего намеренно не удаляет.
   - Проверяет химическую корректность структуры: валентности, ароматичность, типы связей, заряды и другие свойства.
   - При обнаружении несовместимой структуры генерируется исключение.

5. **`Chem.MolToSmiles(mol, canonical=True)` — каноникализация**
   - Не изменяет химический состав молекулы и не удаляет атомы.
   - Переводит объект `Mol` обратно в канонический SMILES.
   - Разные записи одной и той же структуры получают одинаковое каноническое представление.

### Итог

`SMILES → Mol → удаление солей → нейтрализация зарядов → проверка структуры → canonical SMILES`

То есть фактически код **удаляет солевые/контр-ионные фрагменты, изменяет некоторые формальные заряды и приводит запись структуры к каноническому SMILES**. Остальные функции в основном выполняют проверку, преобразование и представление структуры.


=====================================================================================================================
=====================================================================================================================

In [ ]:
from rdkit import Chem
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize


remover = SaltRemover.SaltRemover()
uncharger = rdMolStandardize.Uncharger()


def standardize_smiles(smiles):
    """
    Стандартизация SMILES:

    1. Проверка структуры
    2. Удаление солей
    3. Нейтрализация зарядов
    4. Санитизация
    5. Каноникализация
    """

    # SMILES → RDKit Mol
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    try:
        # Удаление солей
        mol = remover.StripMol(
            mol,
            dontRemoveEverything=True
        )

        # Нейтрализация зарядов
        mol = uncharger.uncharge(mol)

        # Проверка структуры
        Chem.SanitizeMol(mol)

        # Canonical SMILES
        return Chem.MolToSmiles(
            mol,
            canonical=True
        )

    except Exception:
        return None

In [ ]:
df["standardized_smiles"] = df["smiles"].apply(
    standardize_smiles
)
print("Всего:", len(df))
print("Не удалось стандартизировать:",
      df["standardized_smiles"].isna().sum())

удаляем структуры, которые после стандартизации не удалось получить:

In [ ]:
df = df[
    df["standardized_smiles"].notna()
].copy()

делаем standardized_smiles основным SMILES

In [ ]:
df["smiles"] = df["standardized_smiles"]

df = df.drop(
    columns=["standardized_smiles"]
)

удаляем дубликаты

In [ ]:
df = df.drop_duplicates(
    subset="smiles"
).copy()

### Фильтры drug-likeness

Для оценки соответствия молекул основным критериям drug-likeness применяются правила Lipinski и Veber, а также PAINS-фильтрация.

1. Lipinski

MW   ≤ 500
LogP ≤ 5
HBD  ≤ 5
HBA  ≤ 10

### Фильтрация по правилам Lipinski

Для каждой молекулы из SMILES рассчитываются:

- **MW** — молекулярная масса: `Descriptors.MolWt()`, условие ≤ 500 Da
- **LogP** — липофильность: `Crippen.MolLogP()`, условие ≤ 5
- **HBD** — число доноров H-связей: `Lipinski.NumHDonors()`, условие ≤ 5
- **HBA** — число акцепторов H-связей: `Lipinski.NumHAcceptors()`, условие ≤ 10

Затем создаётся столбец `lipinski_pass`, который принимает `True`, если **все четыре условия выполняются одновременно**, и `False`, если нарушено хотя бы одно.

В конце подсчитывается количество молекул, прошедших и не прошедших фильтр.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski

def get_molecule(smiles):
    return Chem.MolFromSmiles(smiles)

df["MW"] = df["smiles"].apply(
    lambda x: Descriptors.MolWt(get_molecule(x))
)

df["LogP"] = df["smiles"].apply(
    lambda x: Crippen.MolLogP(get_molecule(x))
)

df["HBD"] = df["smiles"].apply(
    lambda x: Lipinski.NumHDonors(get_molecule(x))
)

df["HBA"] = df["smiles"].apply(
    lambda x: Lipinski.NumHAcceptors(get_molecule(x))
)

# Проверка соответствия правилам Lipinski
df["lipinski_pass"] = (
    (df["MW"] <= 500) &
    (df["LogP"] <= 5) &
    (df["HBD"] <= 5) &
    (df["HBA"] <= 10)
)

print("Lipinski PASS:", df["lipinski_pass"].sum())
print("Lipinski FAIL:", (~df["lipinski_pass"]).sum())

2. Veber

TPSA     ≤ 140 Å²
RotBonds ≤ 10
- **TPSA** — топологическая полярная поверхность, рассчитывается через `Descriptors.TPSA()`. Условие: **≤ 140 Å²**.
- **RotBonds** — количество вращаемых связей, рассчитывается через `Descriptors.NumRotatableBonds()`. Условие: **≤ 10**.

In [ ]:
from rdkit.Chem import Descriptors

df["TPSA"] = df["smiles"].apply(
    lambda x: Descriptors.TPSA(get_molecule(x))
)

df["RotBonds"] = df["smiles"].apply(
    lambda x: Descriptors.NumRotatableBonds(get_molecule(x))
)

# Проверка соответствия правилам Veber
df["veber_pass"] = (
    (df["TPSA"] <= 140) &
    (df["RotBonds"] <= 10)
)

print("Veber PASS:", df["veber_pass"].sum())
print("Veber FAIL:", (~df["veber_pass"]).sum())

3. PAINS

Для PAINS лучше использовать встроенный FilterCatalog RDKit:
Есть ли в структуре известный PAINS-фрагмент
Это молекулы, которые могут показывать активность в разных биологических тестах не потому, что они действительно взаимодействуют с нужной мишенью, а из-за особенностей своей химической структуры.

In [ ]:
from rdkit.Chem import FilterCatalog

params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(
    FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS
)

pains_catalog = FilterCatalog.FilterCatalog(params)

def check_pains(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return False

    return not pains_catalog.HasMatch(mol)

df["pains_pass"] = df["smiles"].apply(check_pains)

print("PAINS PASS:", df["pains_pass"].sum())
print("PAINS FAIL:", (~df["pains_pass"]).sum())

In [ ]:
# Финальное условие: молекула должна пройти все фильтры
df["final_pass"] = (
    df["lipinski_pass"] &
    df["veber_pass"] &
    df["pains_pass"] &
    df["valid_elements"]
)

print("Всего молекул:", len(df))

print("\nLipinski:")
print(df["lipinski_pass"].value_counts())

print("\nVeber:")
print(df["veber_pass"].value_counts())

print("\nPAINS:")
print(df["pains_pass"].value_counts())

print("\nДопустимые элементы:")
print(df["valid_elements"].value_counts())

print("\nФинальная фильтрация:")
print(df["final_pass"].value_counts())


Финальный фильтр

In [ ]:
# ============================================================
# 0. ФИНАЛЬНАЯ ФИЛЬТРАЦИЯ
# ============================================================

n_before = len(df)

required_filter_columns = [
    "lipinski_pass",
    "veber_pass",
    "pains_pass",
    "valid_elements"
]

missing_filter_columns = [
    col
    for col in required_filter_columns
    if col not in df.columns
]

if missing_filter_columns:
    raise ValueError(
        f"Отсутствуют колонки для фильтрации: "
        f"{missing_filter_columns}"
    )


df = df[
    df["lipinski_pass"] &
    df["veber_pass"] &
    df["pains_pass"] &
    df["valid_elements"]
].copy()

df = df.reset_index(drop=True)


print("=" * 70)
print("ФИНАЛЬНАЯ ФИЛЬТРАЦИЯ")
print("=" * 70)

print("До фильтрации:", n_before)
print("После фильтрации:", len(df))
print("Удалено:", n_before - len(df))


# ------------------------------------------------------------
# Сохранение
# ------------------------------------------------------------

FILTERED_DATASET_PATH = "zinc250k_filtered.parquet"

df.to_parquet(
    FILTERED_DATASET_PATH,
    index=False
)

print()
print("✓ Обработанный датасет сохранён:")
print(FILTERED_DATASET_PATH)

===========================================================================================================================
===========================================================================================================================

Расчитаем и введем дополнительные признаки

In [1]:
# ============================================================
# 2. ИМПОРТЫ И НАСТРОЙКИ
# ============================================================

import os
import random
import time
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import StandardScaler

import selfies as sf


# ------------------------------------------------------------
# RDKit
# ------------------------------------------------------------

RDLogger.DisableLog("rdApp.*")


# ------------------------------------------------------------
# SEED
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )


# ------------------------------------------------------------
# MODEL CONFIG
# ------------------------------------------------------------

LATENT_DIM = 128

EMBED_DIM = 128

NUM_HEADS = 8

NUM_LAYERS = 4

FF_DIM = 512

DROPOUT = 0.1

BATCH_SIZE = 32

LEARNING_RATE = 3e-4

NUM_EPOCHS = 60

WARMUP_EPOCHS = 10

MAX_GRAD_NORM = 1.0

CHECKPOINT_EVERY = 5

LATENT_STATS_BATCHES = 20


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

CHECKPOINT_DIR = "checkpoints_vae"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


print()
print("=" * 70)
print("MODEL CONFIG")
print("=" * 70)

print("LATENT_DIM:", LATENT_DIM)
print("EMBED_DIM:", EMBED_DIM)
print("NUM_HEADS:", NUM_HEADS)
print("NUM_LAYERS:", NUM_LAYERS)
print("FF_DIM:", FF_DIM)
print("DROPOUT:", DROPOUT)
print("BATCH_SIZE:", BATCH_SIZE)
print("LEARNING_RATE:", LEARNING_RATE)
print("NUM_EPOCHS:", NUM_EPOCHS)

C:\Users\apex_\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce GTX 1650
CUDA: 12.8

MODEL CONFIG
LATENT_DIM: 128
EMBED_DIM: 128
NUM_HEADS: 8
NUM_LAYERS: 4
FF_DIM: 512
DROPOUT: 0.1
BATCH_SIZE: 32
LEARNING_RATE: 0.0003
NUM_EPOCHS: 60


In [2]:
# ============================================================
# 1. ЗАГРУЗКА DATASET
# ============================================================

import pandas as pd

df = pd.read_parquet(
    "zinc250k_filtered.parquet"
)

df = df.reset_index(drop=True)


print("=" * 70)
print("DATASET")
print("=" * 70)

print("Количество молекул:", len(df))
print("Количество столбцов:", len(df.columns))

print()
print("Колонки:")

for col in df.columns:
    print(" ", col)

DATASET
Количество молекул: 236659
Количество столбцов: 18

Колонки:
  index
  smiles
  selfies
  logP
  qed
  SAS
  smiles_length
  valid_elements
  MW
  LogP
  HBD
  HBA
  lipinski_pass
  TPSA
  RotBonds
  veber_pass
  pains_pass
  final_pass


Можно добавить рассчитанные из SMILES:

MW — молекулярная масса;

TPSA — топологическая полярная площадь;

NumHDonors — число доноров H-связей;

NumHAcceptors — число акцепторов;

NumRotatableBonds — число вращаемых связей;

NumRings — количество колец;

NumAromaticRings — количество ароматических колец;

NumAliphaticRings — количество алифатических колец;

FractionCSP3 — доля sp³-углеродов;

HeavyAtomCount — количество тяжёлых атомов;

NumHeteroatoms — количество гетероатомов;

FormalCharge;

NumAmideBonds.

Большинство этих признаков можно получить непосредственно из RDKit

Предполагается, что включение дополнительной физико-химической информации в Encoder позволит сформировать более информативное латентное представление молекул и, как следствие, повысить качество их восстановления и генерации. Данное предположение проверяется экспериментально

ВMLP не обучается предсказывать сами дескрипторы. вместо этого MLP учится преобразовывать дескрипторы в такое представление, которое помогает всей VAE минимизировать общую функцию потерь:

Ячейка 1 — импорты и настройки

Ячейка 2 — проверка исходного DataFrame

In [3]:
# ============================================================
# 3. ПРОВЕРКА DATAFRAME
# ============================================================

required_columns = [
    "smiles",
    "selfies",
    "logP",
    "qed",
    "SAS"
]


missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]


if missing_columns:

    raise ValueError(
        f"Отсутствуют необходимые колонки: "
        f"{missing_columns}"
    )


print("=" * 70)
print("ПРОВЕРКА DATAFRAME")
print("=" * 70)

print("Размер:", df.shape)

print()
print("Первые строки:")

display(
    df[
        required_columns
    ].head()
)

ПРОВЕРКА DATAFRAME
Размер: (236659, 18)

Первые строки:


,smiles,selfies,logP,qed,SAS
0,C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1,[C][C@@H1][C][C][Branch2][Ring1][Ring2][N][C][...,3.11370,0.928975,3.432004
1,N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...,[N][#C][C][=C][C][=C][Branch2][Ring2][Ring2][C...,4.96778,0.599682,2.470633
2,CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c...,[C][C][O][C][=Branch1][C][=O][C@@H1][C][C][C][...,4.00022,0.690944,2.822753
3,N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C(O)[C@H](C#N)C...,[N][#C][C][=C][Branch2][Ring1][Ring1][S][C][C]...,3.60956,0.789027,4.035182
4,CCN(CC)[C@](C)(CC)[C@H](O)c1cscc1Br,[C][C][NH1+1][Branch1][Ring1][C][C][C@][Branch...,2.63740,0.824369,5.091438


Ячейка 3 — SELFIES-токенизация

In [4]:
# ============================================================
# 4. SELFIES ТОКЕНИЗАЦИЯ
# ============================================================

def tokenize_selfies(selfies_string):
    """
    SELFIES -> список SELFIES-токенов
    """

    return list(
        sf.split_selfies(
            selfies_string
        )
    )


df["tokens"] = df["selfies"].apply(
    tokenize_selfies
)


df["token_length"] = (
    df["tokens"].apply(len)
)


print("=" * 70)
print("SELFIES TOKENIZATION")
print("=" * 70)

print(
    "Средняя длина:",
    df["token_length"].mean()
)

print(
    "Медиана:",
    df["token_length"].median()
)

print(
    "Минимум:",
    df["token_length"].min()
)

print(
    "Максимум:",
    df["token_length"].max()
)

print(
    "99-й процентиль:",
    df["token_length"].quantile(0.99)
)

SELFIES TOKENIZATION
Средняя длина: 37.12580125835062
Медиана: 37.0
Минимум: 8
Максимум: 72
99-й процентиль: 58.0


Ячейка 4 — выбор MAX_LEN

In [5]:
# ============================================================
# 5. MAX LENGTH
# ============================================================

RAW_MAX_LEN = 64

# + <START> + <END>
MAX_LEN = RAW_MAX_LEN + 2


n_before_length_filter = len(df)


df_vae = df[
    df["token_length"] <= RAW_MAX_LEN
].copy()


df_vae = df_vae.reset_index(
    drop=True
)


print("=" * 70)
print("ОГРАНИЧЕНИЕ ДЛИНЫ")
print("=" * 70)

print(
    "До:",
    n_before_length_filter
)

print(
    "После:",
    len(df_vae)
)

print(
    "Удалено:",
    n_before_length_filter - len(df_vae)
)

print(
    "RAW_MAX_LEN:",
    RAW_MAX_LEN
)

print(
    "MAX_LEN:",
    MAX_LEN
)

ОГРАНИЧЕНИЕ ДЛИНЫ
До: 236659
После: 236548
Удалено: 111
RAW_MAX_LEN: 64
MAX_LEN: 66


Ячейка 5 — vocabulary

In [6]:
# ============================================================
# 6. VOCABULARY
# ============================================================

special_tokens = [
    "<PAD>",
    "<START>",
    "<END>",
    "<UNK>"
]


all_tokens = set()


for tokens in df_vae["tokens"]:

    all_tokens.update(
        tokens
    )


vocab = (
    special_tokens
    + sorted(all_tokens)
)


token_to_idx = {
    token: idx
    for idx, token in enumerate(vocab)
}


idx_to_token = {
    idx: token
    for token, idx in token_to_idx.items()
}


PAD_IDX = token_to_idx["<PAD>"]

START_IDX = token_to_idx["<START>"]

END_IDX = token_to_idx["<END>"]

UNK_IDX = token_to_idx["<UNK>"]

VOCAB_SIZE = len(vocab)


print("=" * 70)
print("VOCABULARY")
print("=" * 70)

print("VOCAB_SIZE:", VOCAB_SIZE)

print("PAD:", PAD_IDX)
print("START:", START_IDX)
print("END:", END_IDX)
print("UNK:", UNK_IDX)

VOCABULARY
VOCAB_SIZE: 109
PAD: 0
START: 1
END: 2
UNK: 3


Ячейка 6 — кодирование SELFIES

In [7]:
# ============================================================
# 7. ENCODING SELFIES
# ============================================================

def encode_tokens(tokens):
    """
    SELFIES tokens
    ->
    <START> + tokens + <END> + <PAD>
    """

    indices = [
        START_IDX
    ]


    for token in tokens:

        indices.append(
            token_to_idx.get(
                token,
                UNK_IDX
            )
        )


    indices.append(
        END_IDX
    )


    if len(indices) > MAX_LEN:

        raise ValueError(
            f"Последовательность имеет "
            f"{len(indices)} токенов, "
            f"MAX_LEN={MAX_LEN}"
        )


    padding_length = (
        MAX_LEN - len(indices)
    )


    indices.extend(
        [PAD_IDX] * padding_length
    )


    return indices


encoded_sequences = (
    df_vae["tokens"]
    .apply(encode_tokens)
)


X = np.asarray(
    encoded_sequences.tolist(),
    dtype=np.int64
)


print("=" * 70)
print("ENCODING")
print("=" * 70)

print("X shape:", X.shape)

print("dtype:", X.dtype)

print("Expected sequence length:", MAX_LEN)

ENCODING
X shape: (236548, 66)
dtype: int64
Expected sequence length: 66


Ячейка 7 — проверка encode/decode

In [8]:
# ============================================================
# 8. CHECK ENCODE / DECODE
# ============================================================

def decode_indices(indices):
    """
    Индексы -> SELFIES tokens
    """

    tokens = []


    for idx in indices:

        idx = int(idx)


        if idx == PAD_IDX:
            break


        if idx == START_IDX:
            continue


        if idx == END_IDX:
            break


        tokens.append(
            idx_to_token.get(
                idx,
                "<UNK>"
            )
        )


    return tokens


errors = 0

N_CHECK = min(
    1000,
    len(df_vae)
)


for i in range(N_CHECK):

    original = (
        df_vae.iloc[i]["tokens"]
    )

    decoded = decode_indices(
        X[i]
    )


    if original != decoded:

        errors += 1


        if errors <= 5:

            print("Ошибка:", i)
            print(
                "Original:",
                original
            )
            print(
                "Decoded:",
                decoded
            )


print()
print("Проверено:", N_CHECK)
print("Ошибок:", errors)


assert errors == 0


print(
    "✓ Encode/decode работает правильно"
)


Проверено: 1000
Ошибок: 0
✓ Encode/decode работает правильно


Ячейка 8 — 11 молекулярных свойств

In [9]:
# ============================================================
# 9. MOLECULAR PROPERTIES
# ============================================================

FEATURE_NAMES = [

    "logP",

    "QED",

    "SAS",

    "MW",

    "TPSA",

    "HBD",

    "HBA",

    "RotatableBonds",

    "Rings",

    "AromaticRings",

    "FractionCSP3"
]


PROPERTY_DIM = len(
    FEATURE_NAMES
)


print("=" * 70)
print("MOLECULAR PROPERTIES")
print("=" * 70)

print(
    "PROPERTY_DIM:",
    PROPERTY_DIM
)

for i, name in enumerate(
    FEATURE_NAMES
):

    print(
        f"{i:2d}: {name}"
    )


assert PROPERTY_DIM == 11

MOLECULAR PROPERTIES
PROPERTY_DIM: 11
 0: logP
 1: QED
 2: SAS
 3: MW
 4: TPSA
 5: HBD
 6: HBA
 7: RotatableBonds
 8: Rings
 9: AromaticRings
10: FractionCSP3


Ячейка 9 — расчёт дополнительных 8 свойств

In [10]:
# ============================================================
# 10. ADDITIONAL MOLECULAR DESCRIPTORS
# ============================================================

def calculate_additional_features(
    smiles_list
):

    features = []


    for smiles in tqdm(
        smiles_list,
        desc="Calculating descriptors"
    ):

        mol = Chem.MolFromSmiles(
            smiles
        )


        if mol is None:

            raise ValueError(
                f"Некорректный SMILES:\n{smiles}"
            )


        MW = Descriptors.MolWt(
            mol
        )


        TPSA = (
            rdMolDescriptors.CalcTPSA(
                mol
            )
        )


        HBD = (
            Lipinski.NumHDonors(
                mol
            )
        )


        HBA = (
            Lipinski.NumHAcceptors(
                mol
            )
        )


        RotatableBonds = (
            Lipinski.NumRotatableBonds(
                mol
            )
        )


        Rings = (
            rdMolDescriptors.CalcNumRings(
                mol
            )
        )


        AromaticRings = (
            rdMolDescriptors.CalcNumAromaticRings(
                mol
            )
        )


        FractionCSP3 = (
            rdMolDescriptors.CalcFractionCSP3(
                mol
            )
        )


        features.append([

            MW,

            TPSA,

            HBD,

            HBA,

            RotatableBonds,

            Rings,

            AromaticRings,

            FractionCSP3

        ])


    return np.asarray(
        features,
        dtype=np.float32
    )


additional_features = (
    calculate_additional_features(
        df_vae["smiles"].tolist()
    )
)


print()
print(
    "Additional features shape:",
    additional_features.shape
)


assert additional_features.shape == (
    len(df_vae),
    8
)


print(
    "✓ 8 дополнительных признаков рассчитаны"
)

Calculating descriptors: 100%|██████████| 236548/236548 [01:40<00:00, 2342.51it/s]



Additional features shape: (236548, 8)
✓ 8 дополнительных признаков рассчитаны


Добавляем Fingerprint

In [11]:
from rdkit import Chem
from rdkit.Chem import AllChem

FP_BITS = 2048
FP_RADIUS = 2


def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=FP_RADIUS,
        nBits=FP_BITS
    )

    return np.asarray(fp, dtype=np.float32)

In [12]:
df["fingerprint"] = df["smiles"].apply(
    smiles_to_fingerprint
)

In [13]:
# ============================================================
# CHECK FINGERPRINT
# ============================================================

print(
    "Fingerprint examples:"
)

print(
    df["fingerprint"].head()
)

print(
    "\nFingerprint length:",
    len(df["fingerprint"].iloc[0])
)

print(
    "Fingerprint dtype:",
    df["fingerprint"].iloc[0].dtype
)

print(
    "Missing fingerprints:",
    df["fingerprint"].isna().sum()
)

Fingerprint examples:
0    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
1    [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
2    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
3    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
4    [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
Name: fingerprint, dtype: object

Fingerprint length: 2048
Fingerprint dtype: float32
Missing fingerprints: 0


Ячейка 10 — объединяем все 11 свойств

In [14]:
# ============================================================
# 11. COMBINE 11 PROPERTIES
# ============================================================

base_features = (
    df_vae[
        [
            "logP",
            "qed",
            "SAS"
        ]
    ]
    .to_numpy(
        dtype=np.float32
    )
)


print(
    "Base features:",
    base_features.shape
)


features = np.concatenate(
    [
        base_features,
        additional_features
    ],
    axis=1
)


print(
    "All features:",
    features.shape
)


assert features.shape == (
    len(df_vae),
    PROPERTY_DIM
)


assert np.isfinite(
    features
).all()


print(
    "✓ Все 11 свойств готовы"
)

print(
    "✓ NaN / Inf отсутствуют"
)

Base features: (236548, 3)
All features: (236548, 11)
✓ Все 11 свойств готовы
✓ NaN / Inf отсутствуют


Ячейка 11 — таблица статистик свойств

In [15]:
# ============================================================
# 12. PROPERTY STATISTICS
# ============================================================

features_df = pd.DataFrame(
    features,
    columns=FEATURE_NAMES
)


display(
    features_df.describe().T
)

,count,mean,std,min,25%,50%,75%,max
logP,236548.0,2.394091,1.398369,-6.876200,1.537780,2.559400,3.422700,5.670980
QED,236548.0,0.735184,0.134281,0.116660,0.656875,0.765038,0.837843,0.947882
SAS,236548.0,3.063494,0.840874,1.132738,2.422866,2.901909,3.564343,7.223421
MW,236548.0,329.438324,60.855633,149.123993,289.265991,331.459991,365.860992,499.998993
TPSA,236548.0,63.974285,23.011419,0.000000,48.000000,63.680000,79.190002,139.949997
HBD,236548.0,1.156962,0.835533,0.000000,1.000000,1.000000,2.000000,5.000000
HBA,236548.0,3.906780,1.382399,0.000000,3.000000,4.000000,5.000000,10.000000
RotatableBonds,236548.0,4.561527,1.550250,0.000000,3.000000,5.000000,6.000000,10.000000
Rings,236548.0,2.720623,0.997562,0.000000,2.000000,3.000000,3.000000,9.000000
AromaticRings,236548.0,1.818540,0.958901,0.000000,1.000000,2.000000,2.000000,6.000000


Ячейка 12 — Scaffold Split

In [16]:
# ============================================================
# 13. SCAFFOLD SPLIT
# ============================================================

def get_scaffold(smiles):

    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is None:

        return None


    return (
        MurckoScaffold
        .MurckoScaffoldSmiles(
            mol=mol,
            includeChirality=False
        )
    )


scaffold_to_indices = {}


for idx, smiles in enumerate(
    tqdm(
        df_vae["smiles"],
        desc="Building scaffolds"
    )
):

    scaffold = get_scaffold(
        smiles
    )


    if scaffold not in scaffold_to_indices:

        scaffold_to_indices[
            scaffold
        ] = []


    scaffold_to_indices[
        scaffold
    ].append(idx)


scaffold_groups = list(
    scaffold_to_indices.values()
)


print()
print(
    "Количество scaffold-групп:",
    len(scaffold_groups)
)

Building scaffolds: 100%|██████████| 236548/236548 [01:36<00:00, 2455.79it/s]


Количество scaffold-групп: 121169


Ячейка 13 — разделение 80/10/10

In [17]:
# ============================================================
# 14. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

random.seed(SEED)

random.shuffle(
    scaffold_groups
)


n_total = len(df_vae)


train_target = int(
    0.80 * n_total
)


val_target = int(
    0.10 * n_total
)


train_indices = []

val_indices = []

test_indices = []


for group in scaffold_groups:

    group_size = len(group)


    if (
        len(train_indices)
        + group_size
        <= train_target
    ):

        train_indices.extend(
            group
        )


    elif (
        len(val_indices)
        + group_size
        <= val_target
    ):

        val_indices.extend(
            group
        )


    else:

        test_indices.extend(
            group
        )


train_indices = np.asarray(
    train_indices,
    dtype=np.int64
)


val_indices = np.asarray(
    val_indices,
    dtype=np.int64
)


test_indices = np.asarray(
    test_indices,
    dtype=np.int64
)


print("=" * 70)
print("SCAFFOLD SPLIT")
print("=" * 70)

print(
    f"Train: {len(train_indices):,}"
)

print(
    f"Val:   {len(val_indices):,}"
)

print(
    f"Test:  {len(test_indices):,}"
)

print(
    f"Total: "
    f"{len(train_indices) + len(val_indices) + len(test_indices):,}"
)


assert (
    len(train_indices)
    + len(val_indices)
    + len(test_indices)
    == len(df_vae)
)

SCAFFOLD SPLIT
Train: 189,238
Val:   23,654
Test:  23,656
Total: 236,548


Ячейка 14 — проверка scaffold split

In [18]:
# ============================================================
# 15. CHECK SPLIT
# ============================================================

train_set = set(
    train_indices
)

val_set = set(
    val_indices
)

test_set = set(
    test_indices
)


assert len(
    train_set & val_set
) == 0


assert len(
    train_set & test_set
) == 0


assert len(
    val_set & test_set
) == 0


assert len(
    train_set
    | val_set
    | test_set
) == len(df_vae)


print(
    "✓ Train / Val / Test индексы не пересекаются"
)

✓ Train / Val / Test индексы не пересекаются


Ячейка 15 — разделение X и properties

In [19]:
# ============================================================
# 16. SPLIT SEQUENCES + PROPERTIES
# ============================================================

X_train = X[
    train_indices
]

X_val = X[
    val_indices
]

X_test = X[
    test_indices
]


features_train = features[
    train_indices
]

features_val = features[
    val_indices
]

features_test = features[
    test_indices
]


# ============================================================
# CHECK
# ============================================================

print("=" * 70)
print("SPLIT DATA")
print("=" * 70)


print("\nSELFIES:")

print(
    "Train:",
    X_train.shape
)

print(
    "Val:  ",
    X_val.shape
)

print(
    "Test: ",
    X_test.shape
)


print("\nProperties:")

print(
    "Train:",
    features_train.shape
)

print(
    "Val:  ",
    features_val.shape
)

print(
    "Test: ",
    features_test.shape
)


# ============================================================
# CHECK SIZES
# ============================================================

assert len(X_train) == len(features_train)
assert len(X_val) == len(features_val)
assert len(X_test) == len(features_test)


print(
    "\n✓ SELFIES и properties синхронизированы"
)

SPLIT DATA

SELFIES:
Train: (189238, 66)
Val:   (23654, 66)
Test:  (23656, 66)

Properties:
Train: (189238, 11)
Val:   (23654, 11)
Test:  (23656, 11)

✓ SELFIES и properties синхронизированы


Ячейка 16 — нормализация свойств

Очень важный момент: fit только на train.

In [20]:
# ============================================================
# 17. STANDARD SCALER
# ============================================================

property_scaler = StandardScaler()


# ВАЖНО:
# fit только на TRAIN

features_train_scaled = (
    property_scaler
    .fit_transform(
        features_train
    )
    .astype(np.float32)
)


features_val_scaled = (
    property_scaler
    .transform(
        features_val
    )
    .astype(np.float32)
)


features_test_scaled = (
    property_scaler
    .transform(
        features_test
    )
    .astype(np.float32)
)


print("=" * 70)
print("PROPERTY NORMALIZATION")
print("=" * 70)

print(
    "Train mean:"
)

print(
    features_train_scaled.mean(
        axis=0
    )
)


print()
print(
    "Train std:"
)

print(
    features_train_scaled.std(
        axis=0
    )


)


assert np.isfinite(
    features_train_scaled
).all()


assert np.isfinite(
    features_val_scaled
).all()


assert np.isfinite(
    features_test_scaled
).all()


print(
    "\n✓ Properties нормализованы"
)

PROPERTY NORMALIZATION
Train mean:
[ 2.4720528e-08  2.0473172e-08  4.1399311e-08 -1.6153552e-07
 -1.3401210e-07 -1.1730585e-05  7.8015164e-06 -3.2665266e-06
 -1.0948150e-05 -1.1531695e-05 -9.0850386e-07]

Train std:
[0.9999779  0.9999732  0.99997556 0.9999926  0.9999878  0.99945676
 0.99955684 0.9997771  0.99975425 1.0001967  0.99996734]

✓ Properties нормализованы


Ячейка 17 — Dataset

In [21]:
# ============================================================
# 18. MOLECULE DATASET
# ============================================================

class MoleculeDataset(
    Dataset
):

    def __init__(
        self,
        sequences,
        properties,
        fingerprint_source,
        indices
    ):

        if len(sequences) != len(properties):

            raise ValueError(
                "Количество sequences "
                "и properties не совпадает"
            )

        if len(sequences) != len(indices):

            raise ValueError(
                "Количество sequences "
                "и indices не совпадает"
            )


        self.sequences = torch.tensor(
            sequences,
            dtype=torch.long
        )


        self.properties = torch.tensor(
            properties,
            dtype=torch.float32
        )


        # Не копируем все fingerprints в RAM
        self.fingerprint_source = fingerprint_source
        self.indices = np.asarray(
            indices
        )


    def __len__(self):

        return len(
            self.sequences
        )


    def __getitem__(
        self,
        idx
    ):

        # Берём только один fingerprint
        source_idx = self.indices[idx]

        fingerprint = np.asarray(
            self.fingerprint_source.iloc[source_idx],
            dtype=np.float32
        )

        fingerprint = torch.from_numpy(
            fingerprint
        )

        return (
            self.sequences[idx],
            self.properties[idx],
            fingerprint
        )

Ячейка 18 — Dataset objects

In [22]:
# ============================================================
# 19. DATASET OBJECTS
# ============================================================

train_dataset = MoleculeDataset(
    X_train,
    features_train_scaled,
    df["fingerprint"],
    train_indices
)


val_dataset = MoleculeDataset(
    X_val,
    features_val_scaled,
    df["fingerprint"],
    val_indices
)


test_dataset = MoleculeDataset(
    X_test,
    features_test_scaled,
    df["fingerprint"],
    test_indices
)


print(
    "Train:",
    len(train_dataset)
)

print(
    "Val:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)

Train: 189238
Val: 23654
Test: 23656


In [23]:
# ============================================================
# 19.1 CHECK DATASET
# ============================================================

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)

print("len(df):", len(df))
print("len(X):", len(X))
print("len(features):", len(features))

print("\nIndices:")
print("Train:", len(train_indices))
print("Val:  ", len(val_indices))
print("Test: ", len(test_indices))

print("\nDataset:")
print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))


# Проверяем один элемент
seq, prop, fp = train_dataset[0]

print("\nFirst train sample:")
print("Sequence shape:", seq.shape)
print("Properties shape:", prop.shape)
print("Fingerprint shape:", fp.shape)
print("Fingerprint dtype:", fp.dtype)

assert fp.shape[0] == 2048
assert prop.shape[0] == 11

print("\n✓ Dataset работает")

DATASET CHECK
len(df): 236659
len(X): 236548
len(features): 236548

Indices:
Train: 189238
Val:   23654
Test:  23656

Dataset:
Train: 189238
Val:   23654
Test:  23656

First train sample:
Sequence shape: torch.Size([66])
Properties shape: torch.Size([11])
Fingerprint shape: torch.Size([2048])
Fingerprint dtype: torch.float32

✓ Dataset работает


Ячейка 19 — DataLoader

In [24]:
# ============================================================
# 20. DATALOADERS
# ============================================================

PIN_MEMORY = torch.cuda.is_available()


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=PIN_MEMORY
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=PIN_MEMORY
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Train batches: 5914
Val batches: 740
Test batches: 740


Ячейка 20 — проверка DataLoader

In [25]:
# ============================================================
# 21. CHECK DATALOADER
# ============================================================

batch, properties, fingerprints = next(
    iter(train_loader)
)


print(
    "Batch:",
    batch.shape,
    batch.dtype
)


print(
    "Properties:",
    properties.shape,
    properties.dtype
)


print(
    "Fingerprints:",
    fingerprints.shape,
    fingerprints.dtype
)


# ============================================================
# ASSERTIONS
# ============================================================

assert batch.ndim == 2

assert properties.ndim == 2

assert fingerprints.ndim == 2


assert batch.shape[1] == MAX_LEN

assert properties.shape[1] == PROPERTY_DIM

assert fingerprints.shape[1] == 2048


assert batch.dtype == torch.long

assert properties.dtype == torch.float32

assert fingerprints.dtype == torch.float32


print(
    "\n✓ DataLoader работает правильно"
)

print(
    "✓ SELFIES + properties + fingerprints"
)

Batch: torch.Size([32, 66]) torch.int64
Properties: torch.Size([32, 11]) torch.float32
Fingerprints: torch.Size([32, 2048]) torch.float32

✓ DataLoader работает правильно
✓ SELFIES + properties + fingerprints


Ячейка 21 — Positional Encoding

In [26]:
# ============================================================
# 22. POSITIONAL ENCODING
# ============================================================

class PositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len
    ):

        super().__init__()


        pe = torch.zeros(
            max_len,
            d_model
        )


        position = torch.arange(
            0,
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)


        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (
                -math.log(10000.0)
                / d_model
            )
        )


        pe[:, 0::2] = torch.sin(
            position * div_term
        )


        pe[:, 1::2] = torch.cos(
            position * div_term
        )


        pe = pe.unsqueeze(0)


        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        seq_len = x.size(1)


        return (
            x
            + self.pe[
                :, :seq_len
            ]
        )

Ячейка 22 — Encoder

In [27]:
# ============================================================
# 23. MULTIMODAL FUSION ENCODER
#     → CLEAN LATENT x₀ FOR DIFFUSION
# ============================================================


class FingerprintEncoder(
    nn.Module
):

    def __init__(
        self,
        fingerprint_dim=2048,
        hidden_dim=512,
        output_dim=128,
        dropout=0.1
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                fingerprint_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )


    def forward(
        self,
        fingerprint
    ):

        return self.network(
            fingerprint
        )


# ============================================================
# MULTIMODAL FUSION ENCODER
#
# SELFIES + FINGERPRINT + DESCRIPTORS
#                ↓
#              FUSION
#                ↓
#               x₀
#
# x₀ — clean continuous molecular representation.
#
# Далее x₀ будет использоваться в diffusion process:
#
# x₀ → xₜ
#
# ============================================================


class Encoder(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        num_layers,
        ff_dim,
        max_len,
        property_dim,
        fingerprint_dim=2048,
        dropout=0.1
    ):

        super().__init__()


        # ====================================================
        # SELFIES EMBEDDING
        # ====================================================

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )


        self.pos_embedding = nn.Embedding(
            max_len,
            embed_dim
        )


        # ====================================================
        # TRANSFORMER ENCODER
        # ====================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                batch_first=True,
                norm_first=True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )


        self.norm = nn.LayerNorm(
            embed_dim
        )


        # ====================================================
        # DESCRIPTOR ENCODER
        #
        # Descriptors:
        # logP, QED, SAS, MW, TPSA, HBD, HBA,
        # RotatableBonds, Rings, AromaticRings, FractionCSP3
        #
        # property_dim = 11
        # ====================================================

        self.property_encoder = (
            nn.Sequential(

                nn.Linear(
                    property_dim,
                    64
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                ),

                nn.Linear(
                    64,
                    embed_dim
                ),

                nn.LayerNorm(
                    embed_dim
                )
            )
        )


        # ====================================================
        # FINGERPRINT ENCODER
        # ====================================================

        self.fingerprint_encoder = (
            FingerprintEncoder(
                fingerprint_dim=fingerprint_dim,
                hidden_dim=512,
                output_dim=embed_dim,
                dropout=dropout
            )
        )


        # ====================================================
        # MULTIMODAL FUSION
        #
        # SELFIES       -> embed_dim
        # Descriptors   -> embed_dim
        # Fingerprint   -> embed_dim
        #
        # concat -> 3 * embed_dim
        # ====================================================

        fusion_dim = (
            embed_dim * 3
        )


        self.fusion = nn.Sequential(

            nn.Linear(
                fusion_dim,
                512
            ),

            nn.LayerNorm(
                512
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                512,
                embed_dim
            )
        )


    def forward(
        self,
        x,
        properties,
        fingerprint
    ):

        batch_size, seq_len = (
            x.shape
        )


        # ====================================================
        # PADDING MASK
        # ====================================================

        padding_mask = (
            x == PAD_IDX
        )


        # ====================================================
        # TOKEN EMBEDDING
        # ====================================================

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0)


        h = self.embedding(
            x
        )


        h = (
            h
            + self.pos_embedding(
                positions
            )
        )


        # ====================================================
        # TRANSFORMER
        # ====================================================

        h = self.transformer(
            h,
            src_key_padding_mask=padding_mask
        )


        h = self.norm(
            h
        )


        # ====================================================
        # MASKED MEAN POOLING
        # ====================================================

        mask = (
            ~padding_mask
        ).unsqueeze(-1)


        h_masked = (
            h * mask
        )


        lengths = (
            mask.sum(
                dim=1
            ).clamp(
                min=1
            )
        )


        h_mean = (
            h_masked.sum(
                dim=1
            )
            / lengths
        )


        # ====================================================
        # DESCRIPTOR EMBEDDING
        # ====================================================

        property_embedding = (
            self.property_encoder(
                properties
            )
        )


        # ====================================================
        # FINGERPRINT EMBEDDING
        # ====================================================

        fingerprint_embedding = (
            self.fingerprint_encoder(
                fingerprint
            )
        )


        # ====================================================
        # MULTIMODAL FUSION
        # ====================================================

        combined = torch.cat(
            [
                h_mean,
                property_embedding,
                fingerprint_embedding
            ],
            dim=1
        )


        x0 = self.fusion(
            combined
        )


        # ====================================================
        # CLEAN LATENT REPRESENTATION x₀
        #
        # IMPORTANT:
        #
        # This is NOT:
        #   mu
        #   logvar
        #   VAE latent z
        #
        # This is the clean continuous representation
        # that will be used as x₀ in the diffusion process.
        #
        # Next:
        #
        # x₀ + noise + timestep
        #              ↓
        #             xₜ
        # ====================================================

        return x0

Ячейка 23 — Decoder

In [28]:
# ============================================================
# 24. CONDITIONAL DIFFUSION TRANSFORMER
# ============================================================
#
# Задача модели:
#
#       x₀
#        ↓
#   Forward Diffusion
#        ↓
#       xₜ
#
# Далее Diffusion Transformer получает:
#
#       xₜ
#       timestep t
#       target properties
#
# и предсказывает добавленный Gaussian noise.
#
# Output:
#
#       predicted_noise
#
# Размерность:
#
#       [batch, embed_dim]
#
# ============================================================


class TimeEmbedding(
    nn.Module
):

    def __init__(
        self,
        embedding_dim
    ):

        super().__init__()


        self.embedding_dim = (
            embedding_dim
        )


        self.network = nn.Sequential(

            nn.Linear(
                embedding_dim,
                embedding_dim * 4
            ),

            nn.GELU(),

            nn.Linear(
                embedding_dim * 4,
                embedding_dim
            )
        )


    def forward(
        self,
        timestep
    ):

        half_dim = (
            self.embedding_dim // 2
        )


        exponent = (
            -math.log(10000.0)
            / (
                half_dim - 1
            )
        )


        frequencies = torch.exp(

            torch.arange(
                half_dim,
                device=timestep.device,
                dtype=torch.float32
            )
            * exponent

        )


        angles = (
            timestep.float().unsqueeze(1)
            * frequencies.unsqueeze(0)
        )


        embedding = torch.cat(
            [
                torch.sin(angles),
                torch.cos(angles)
            ],
            dim=1
        )


        return self.network(
            embedding
        )


# ============================================================
# CONDITIONAL DIFFUSION TRANSFORMER
# ============================================================


class DiffusionTransformer(
    nn.Module
):

    def __init__(
        self,
        latent_dim,
        embed_dim,
        num_heads,
        num_layers,
        ff_dim,
        property_dim,
        dropout=0.1
    ):

        super().__init__()


        # ====================================================
        # xₜ PROJECTION
        # ====================================================

        self.latent_projection = (
            nn.Linear(
                latent_dim,
                embed_dim
            )
        )


        # ====================================================
        # TIMESTEP EMBEDDING
        # ====================================================

        self.time_embedding = (
            TimeEmbedding(
                embedding_dim=embed_dim
            )
        )


        # ====================================================
        # TARGET PROPERTY EMBEDDING
        # ====================================================

        self.property_projection = (
            nn.Sequential(

                nn.Linear(
                    property_dim,
                    64
                ),

                nn.GELU(),

                nn.Linear(
                    64,
                    embed_dim
                ),

                nn.LayerNorm(
                    embed_dim
                )
            )
        )


        # ====================================================
        # CONDITION FUSION
        #
        # xₜ
        # timestep
        # target properties
        #
        # ↓
        #
        # conditional representation
        # ====================================================

        self.condition_projection = (
            nn.Sequential(

                nn.Linear(
                    embed_dim * 3,
                    embed_dim
                ),

                nn.LayerNorm(
                    embed_dim
                ),

                nn.GELU(),

                nn.Dropout(
                    dropout
                )
            )
        )


        # ====================================================
        # TRANSFORMER
        # ====================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                batch_first=True,
                norm_first=True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )


        # ====================================================
        # OUTPUT
        #
        # Predict Gaussian noise ε
        #
        # Same dimensionality as xₜ
        # ====================================================

        self.output_layer = (
            nn.Sequential(

                nn.LayerNorm(
                    embed_dim
                ),

                nn.Linear(
                    embed_dim,
                    latent_dim
                )
            )
        )


    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        x_t,
        timestep,
        target_properties
    ):

        # ====================================================
        # xₜ
        # ====================================================

        x_embedding = (
            self.latent_projection(
                x_t
            )
        )


        # ====================================================
        # TIMESTEP
        # ====================================================

        time_embedding = (
            self.time_embedding(
                timestep
            )
        )


        # ====================================================
        # TARGET PROPERTIES
        # ====================================================

        property_embedding = (
            self.property_projection(
                target_properties
            )
        )


        # ====================================================
        # COMBINE CONDITIONS
        # ====================================================

        condition = torch.cat(
            [
                x_embedding,
                time_embedding,
                property_embedding
            ],
            dim=1
        )


        condition = (
            self.condition_projection(
                condition
            )
        )


        # ====================================================
        # TRANSFORMER
        #
        # One latent conditioning token.
        # ====================================================

        sequence = (
            condition.unsqueeze(1)
        )


        output = self.transformer(
            sequence
        )


        output = (
            output[:, 0]
        )


        # ====================================================
        # PREDICTED NOISE
        # ====================================================

        predicted_noise = (
            self.output_layer(
                output
            )
        )


        return predicted_noise

In [29]:
# ============================================================
# 24A. SELFIES DECODER
#     x₀ → SELFIES
# ============================================================

class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        num_layers,
        ff_dim,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # ----------------------------------------------------
        # TOKEN EMBEDDING
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )

        # ----------------------------------------------------
        # POSITION EMBEDDING
        # ----------------------------------------------------

        self.pos_embedding = nn.Embedding(
            max_len,
            embed_dim
        )

        # ----------------------------------------------------
        # LATENT → DECODER CONDITION
        # ----------------------------------------------------

        self.latent_projection = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU()
        )

        # ----------------------------------------------------
        # TRANSFORMER DECODER
        # ----------------------------------------------------

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        # ----------------------------------------------------
        # OUTPUT PROJECTION
        # ----------------------------------------------------

        self.output_layer = nn.Linear(
            embed_dim,
            vocab_size
        )

    def forward(self, latent, decoder_input):

        batch_size, seq_len = decoder_input.shape

        # ----------------------------------------------------
        # TOKEN EMBEDDINGS
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=decoder_input.device
        ).unsqueeze(0)

        x = self.embedding(decoder_input)

        x = x + self.pos_embedding(positions)

        # ----------------------------------------------------
        # LATENT CONDITION
        #
        # TransformerDecoder ожидает memory:
        # [batch, memory_seq_len, embed_dim]
        #
        # Наш latent — один вектор на молекулу,
        # поэтому memory_seq_len = 1.
        # ----------------------------------------------------

        memory = self.latent_projection(latent)
        memory = memory.unsqueeze(1)

        # ----------------------------------------------------
        # CAUSAL MASK
        #
        # Текущий токен не должен видеть будущие токены.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=decoder_input.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # PADDING MASK
        # ----------------------------------------------------

        tgt_padding_mask = decoder_input == PAD_IDX

        # ----------------------------------------------------
        # TRANSFORMER DECODER
        # ----------------------------------------------------

        output = self.transformer(
            tgt=x,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        # ----------------------------------------------------
        # VOCABULARY LOGITS
        # ----------------------------------------------------

        logits = self.output_layer(output)

        return logits


print("=" * 70)
print("SELFIES DECODER")
print("=" * 70)

print()
print("Input:")
print("  clean latent x₀")

print()
print("Output:")
print("  SELFIES token logits")

print()
print("Decoder:")
print("  TransformerDecoder")
print("  Causal self-attention")
print("  Latent-conditioned cross-attention")

print()
print("✓ Decoder class создан")

SELFIES DECODER

Input:
  clean latent x₀

Output:
  SELFIES token logits

Decoder:
  TransformerDecoder
  Causal self-attention
  Latent-conditioned cross-attention

✓ Decoder class создан


Ячейка 24 — VAE

In [30]:
# ============================================================
# 25. CONDITIONAL LATENT DIFFUSION MODEL
# ============================================================
#
# Off-manifold training:
#
# SELFIES ────────┐
#                 │
# Fingerprint ────┼→ Fusion Encoder → raw x₀
#                 │                     │
# Descriptors ────┘                     │
#                                       │
#                         ┌─────────────┴─────────────┐
#                         │                           │
#                         ↓                           ↓
#                  SELFIES Decoder              EMA normalization
#                         │                           │
#                         ↓                           ↓
#                   reconstruction             normalized x₀
#                                                     │
#                                            Forward Diffusion
#                                                     │
#                                                     ↓
#                                                    xₜ
#                                                     │
#                                           + perturbation δ
#                                                     │
#                                                     ↓
#                                                   xₜ'
#                                                     │
#                                       ┌─────────────┴─────────────┐
#                                       │                           │
#                                target properties             timestep
#                                       │                           │
#                                       └─────────────┬─────────────┘
#                                                     ↓
#                                        Diffusion Transformer
#                                                     │
#                                                     ↓
#                                              predicted noise
#
# ВАЖНО:
# - perturbation используется только во время training
# - validation получает чистый xₜ
# - возвращаемый x_t остаётся чистым для loss и диагностики
#
# ============================================================


class DiffusionModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        num_layers,
        ff_dim,
        max_len,
        property_dim,
        fingerprint_dim=2048,
        dropout=0.1,
        initial_latent_rms=0.18,
        latent_rms_momentum=0.99
    ):

        super().__init__()


        # ====================================================
        # LATENT RMS NORMALIZATION
        # ====================================================

        self.register_buffer(
            "latent_rms",
            torch.tensor(
                float(initial_latent_rms),
                dtype=torch.float32
            )
        )

        self.latent_rms_momentum = float(
            latent_rms_momentum
        )


        # ====================================================
        # FUSION ENCODER
        #
        # SELFIES + Fingerprint + Descriptors
        #             ↓
        #       Multimodal Fusion
        #             ↓
        #          raw x₀
        # ====================================================

        self.encoder = Encoder(

            vocab_size=vocab_size,

            embed_dim=embed_dim,

            num_heads=num_heads,

            num_layers=num_layers,

            ff_dim=ff_dim,

            max_len=max_len,

            property_dim=property_dim,

            fingerprint_dim=fingerprint_dim,

            dropout=dropout
        )


        # ====================================================
        # SELFIES DECODER
        #
        # raw x₀ → Transformer Decoder
        # ====================================================

        self.decoder = Decoder(

            vocab_size=vocab_size,

            embed_dim=embed_dim,

            num_heads=num_heads,

            num_layers=num_layers,

            ff_dim=ff_dim,

            max_len=max_len,

            dropout=dropout
        )


        # ====================================================
        # DIFFUSION TRANSFORMER
        #
        # normalized xₜ'
        # +
        # timestep
        # +
        # target properties
        # ↓
        # predicted noise
        # ====================================================

        self.diffusion = DiffusionTransformer(

            latent_dim=embed_dim,

            embed_dim=embed_dim,

            num_heads=num_heads,

            num_layers=num_layers,

            ff_dim=ff_dim,

            property_dim=property_dim,

            dropout=dropout
        )


    # ========================================================
    # LATENT NORMALIZATION
    # ========================================================

    @torch.no_grad()
    def _update_latent_rms(
        self,
        x0_raw
    ):

        # ----------------------------------------------------
        # RMS по всем значениям batch
        # ----------------------------------------------------

        batch_rms = torch.sqrt(
            torch.mean(
                x0_raw.detach() ** 2
            ) + 1e-8
        )


        # ----------------------------------------------------
        # EMA update
        # ----------------------------------------------------

        self.latent_rms.mul_(
            self.latent_rms_momentum
        ).add_(
            batch_rms,
            alpha=1.0 - self.latent_rms_momentum
        )


    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        x,
        properties,
        fingerprint,
        timestep,
        noise,
        diffusion_schedule,
        perturbation_prob=0.0,
        perturbation_std=0.0
    ):

        # ====================================================
        # 1. FUSION ENCODER
        #
        # SELFIES + FP + descriptors
        #             ↓
        #          raw x₀
        # ====================================================

        x0_raw = (
            self.encoder(
                x,
                properties,
                fingerprint
            )
        )


        # ====================================================
        # 2. SELFIES RECONSTRUCTION
        #
        # Decoder получает RAW latent.
        # ====================================================

        decoder_input = (
            x[:, :-1]
        )


        reconstruction_logits = (
            self.decoder(
                x0_raw,
                decoder_input
            )
        )


        # ====================================================
        # 3. UPDATE GLOBAL LATENT RMS
        #
        # EMA обновляется только во время training.
        # ====================================================

        if self.training:

            self._update_latent_rms(
                x0_raw
            )


        # ====================================================
        # 4. NORMALIZE LATENT FOR DIFFUSION
        # ====================================================

        latent_rms = (
            self.latent_rms.detach()
            .clamp_min(1e-6)
        )


        x0_norm = (
            x0_raw / latent_rms
        )


        # ====================================================
        # 5. FORWARD DIFFUSION
        #
        # normalized x₀ + noise + timestep
        #                    ↓
        #                   xₜ
        #
        # x_t здесь является ЧИСТЫМ forward-состоянием.
        # ====================================================

        x_t = (
            diffusion_schedule.q_sample(
                x0_norm,
                timestep,
                noise
            )
        )


        # ====================================================
        # 6. OFF-MANIFOLD PERTURBATION
        #
        # Во время training:
        #
        #   xₜ → xₜ + δ
        #
        # где:
        #
        #   δ ~ N(0, perturbation_std²)
        #
        # Perturbation применяется только к входу
        # Diffusion Transformer.
        #
        # Сам x_t НЕ изменяется.
        # ====================================================

        x_t_model = x_t


        if (
            self.training
            and perturbation_prob > 0.0
            and perturbation_std > 0.0
        ):

            perturb_mask = (
                torch.rand(
                    x_t.shape[0],
                    device=x_t.device
                )
                < perturbation_prob
            )


            delta = (
                torch.randn_like(x_t)
                * perturbation_std
            )


            x_t_model = (
                x_t
                + delta
                * perturb_mask[:, None].to(
                    x_t.dtype
                )
            )


        # ====================================================
        # 7. PREDICT NOISE
        #
        # Training:
        #   x_t_model = x_t + δ
        #
        # Validation:
        #   x_t_model = x_t
        #
        # ====================================================

        predicted_noise = (
            self.diffusion(
                x_t_model,
                timestep,
                properties
            )
        )


        # ====================================================
        # RETURN
        # ====================================================
        #
        # x0_raw
        #   → Decoder
        #
        # x0_norm
        #   → diffusion / diagnostics
        #
        # x_t
        #   → CLEAN forward diffusion state
        #
        # x_t_model
        #   → используется только внутри denoiser
        #
        # ====================================================

        return (
            predicted_noise,
            reconstruction_logits,
            x0_raw,
            x0_norm,
            x_t
        )

Ячейка 25 — создание модели

In [31]:
# ============================================================
# 25A. LATENT NORMALIZATION CONFIGURATION
# ============================================================
#
# Нормализация latent выполняется внутри DiffusionModel
# через EMA latent_rms.
#
# ВАЖНО:
# - старый LATENT_SCALE больше НЕ используется
# - raw x₀ передаётся в Decoder без изменения
# - normalized x₀ используется только для Diffusion
# ============================================================


# ============================================================
# INITIAL LATENT RMS
# ============================================================
#
# Начальное приближение получено из предыдущей диагностики:
#
# raw x₀ std ≈ 0.179
#
# Это только стартовое значение.
# Во время обучения оно будет автоматически изменяться
# через EMA.
# ============================================================

INITIAL_LATENT_RMS = 0.18


# ============================================================
# EMA MOMENTUM
# ============================================================
#
# Чем ближе к 1.0:
#   → тем медленнее меняется оценка масштаба.
#
# 0.99 означает:
#   99% старого значения
#   1% нового batch RMS
# ============================================================

LATENT_RMS_MOMENTUM = 0.99


# ============================================================
# CHECK CONFIGURATION
# ============================================================

print("=" * 70)
print("LATENT NORMALIZATION CONFIGURATION")
print("=" * 70)

print()
print(f"Initial latent RMS : {INITIAL_LATENT_RMS:.6f}")
print(f"EMA momentum       : {LATENT_RMS_MOMENTUM:.4f}")

print()
print("Normalization:")
print("  x₀ raw")
print("    ↓")
print("  x₀ / EMA latent RMS")
print("    ↓")
print("  x₀ normalized")
print("    ↓")
print("  Forward Diffusion")

print()
print("Decoder:")
print("  raw x₀ → SELFIES reconstruction")

print()
print("Fixed LATENT_SCALE:")
print("  DISABLED")

print()
print("✓ Latent normalization configured")


LATENT NORMALIZATION CONFIGURATION

Initial latent RMS : 0.180000
EMA momentum       : 0.9900

Normalization:
  x₀ raw
    ↓
  x₀ / EMA latent RMS
    ↓
  x₀ normalized
    ↓
  Forward Diffusion

Decoder:
  raw x₀ → SELFIES reconstruction

Fixed LATENT_SCALE:
  DISABLED

✓ Latent normalization configured


In [32]:
# ============================================================
# 26. CREATE CONDITIONAL LATENT DIFFUSION MODEL
# ============================================================

from pathlib import Path


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================
#
# Новая версия модели использует EMA latent RMS
# вместо старого LATENT_SCALE.
#
# Используем отдельную директорию, чтобы не смешивать
# checkpoints старой и новой архитектуры.
# ============================================================

CHECKPOINT_DIR = Path(
    "checkpoints_diffusion_x0_minsnr"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CREATE MODEL
# ============================================================

model = DiffusionModel(

    vocab_size=VOCAB_SIZE,

    embed_dim=EMBED_DIM,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS,

    ff_dim=FF_DIM,

    max_len=MAX_LEN,

    property_dim=PROPERTY_DIM,

    fingerprint_dim=FP_BITS,

    dropout=DROPOUT,

    initial_latent_rms=INITIAL_LATENT_RMS,

    latent_rms_momentum=LATENT_RMS_MOMENTUM

).to(device)


# ============================================================
# MODEL COMPONENTS
# ============================================================

print("=" * 70)
print("CONDITIONAL LATENT DIFFUSION MODEL")
print("=" * 70)

print()

print("Checkpoint directory:")
print(
    f"  {CHECKPOINT_DIR.resolve()}"
)

print()

print("Latent normalization:")
print(
    "  Initial RMS:",
    f"{model.latent_rms.item():.6f}"
)

print(
    "  EMA momentum:",
    f"{model.latent_rms_momentum:.4f}"
)

print()

print("Encoder:")
print("  SELFIES + Fingerprint + Descriptors")
print("  → clean latent x₀ (raw)")

print()

print("Latent normalization:")
print("  x₀ raw")
print("  → / EMA latent RMS")
print("  → x₀ normalized")

print()

print("Decoder:")
print("  x₀ raw")
print("  → reconstructed SELFIES")

print()

print("Diffusion:")
print("  x₀ normalized")
print("  → xₜ")
print("  → predicted noise")

print()

print("Model:")
print(model)

print()
print("=" * 70)
print("MODEL COMPONENTS")
print("=" * 70)

print(
    "Encoder parameters:",
    sum(
        p.numel()
        for p in model.encoder.parameters()
    )
)

print(
    "Decoder parameters:",
    sum(
        p.numel()
        for p in model.decoder.parameters()
    )
)

print(
    "Diffusion parameters:",
    sum(
        p.numel()
        for p in model.diffusion.parameters()
    )
)

print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

print()

print(
    "Latent RMS buffer:",
    model.latent_rms.item()
)

print(
    "Latent RMS momentum:",
    model.latent_rms_momentum
)

print()

print("✓ Encoder создан")
print("✓ SELFIES Decoder создан")
print("✓ Diffusion Transformer создан")
print("✓ EMA latent normalization создана")
print("✓ Fixed LATENT_SCALE отключён")
print("✓ Checkpoint directory создан:", CHECKPOINT_DIR)
print("✓ Общая модель готова")


C:\Users\apex_\AppData\Local\Temp\ipykernel_23704\908471559.py:128: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


CONDITIONAL LATENT DIFFUSION MODEL

Checkpoint directory:
  C:\apex\APEX\МИФИ\СНО\medic\checkpoints_diffusion_x0_minsnr

Latent normalization:
  Initial RMS: 0.180000
  EMA momentum: 0.9900

Encoder:
  SELFIES + Fingerprint + Descriptors
  → clean latent x₀ (raw)

Latent normalization:
  x₀ raw
  → / EMA latent RMS
  → x₀ normalized

Decoder:
  x₀ raw
  → reconstructed SELFIES

Diffusion:
  x₀ normalized
  → xₜ
  → predicted noise

Model:
DiffusionModel(
  (encoder): Encoder(
    (embedding): Embedding(109, 128, padding_idx=0)
    (pos_embedding): Embedding(66, 128)
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_fea

C:\Users\apex_\AppData\Local\Temp\ipykernel_23704\958820297.py:237: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


Ячейка 26 — количество параметров

In [33]:
# ============================================================
# 27. MODEL SIZE
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Всего параметров: "
    f"{total_params:,}"
)


print(
    f"Обучаемых параметров: "
    f"{trainable_params:,}"
)

Всего параметров: 4,333,165
Обучаемых параметров: 4,333,165


Ячейка 27 — проверка одного forward pass

In [34]:
# ============================================================
# 28-0. DIFFUSION SCHEDULE
# ============================================================

import math
import torch
import torch.nn as nn


# ============================================================
# DIFFUSION CONFIGURATION
# ============================================================

NUM_DIFFUSION_STEPS = 1000

BETA_START = 1e-4
BETA_END = 0.02

DIFFUSION_SCHEDULE_TYPE = "linear"


# ============================================================
# DIFFUSION SCHEDULE
# ============================================================

class DiffusionSchedule(nn.Module):

    def __init__(
        self,
        num_steps=1000,
        beta_start=1e-4,
        beta_end=0.02,
        schedule_type="linear"
    ):
        super().__init__()

        self.num_steps = num_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.schedule_type = schedule_type


        # ----------------------------------------------------
        # Beta schedule
        # ----------------------------------------------------

        if schedule_type == "linear":

            betas = torch.linspace(
                beta_start,
                beta_end,
                num_steps,
                dtype=torch.float32
            )

        else:

            raise ValueError(
                f"Неизвестный diffusion schedule: "
                f"{schedule_type}"
            )


        # ----------------------------------------------------
        # Alpha
        # ----------------------------------------------------

        alphas = (
            1.0
            - betas
        )


        # ----------------------------------------------------
        # Cumulative product of alpha
        #
        # ᾱ_t = ∏ α_i
        # ----------------------------------------------------

        alpha_bars = torch.cumprod(
            alphas,
            dim=0
        )


        # ----------------------------------------------------
        # Values required for forward diffusion
        # ----------------------------------------------------

        sqrt_alpha_bars = torch.sqrt(
            alpha_bars
        )

        sqrt_one_minus_alpha_bars = torch.sqrt(
            1.0
            - alpha_bars
        )


        # ----------------------------------------------------
        # Values required later for reverse diffusion
        # ----------------------------------------------------

        sqrt_recip_alphas = torch.sqrt(
            1.0
            / alphas
        )


        # ----------------------------------------------------
        # ᾱ_{t-1}
        # ----------------------------------------------------

        alpha_bars_prev = torch.cat(
            [
                torch.ones(
                    1,
                    dtype=torch.float32
                ),
                alpha_bars[:-1]
            ]
        )


        # ----------------------------------------------------
        # Posterior variance
        #
        # β_t * (1 - ᾱ_{t-1})
        # ----------------------
        #      1 - ᾱ_t
        # ----------------------------------------------------

        posterior_variance = (
            betas
            * (
                1.0
                - alpha_bars_prev
            )
            / (
                1.0
                - alpha_bars
            )
        )


        # ----------------------------------------------------
        # Register buffers
        # ----------------------------------------------------

        self.register_buffer(
            "betas",
            betas
        )

        self.register_buffer(
            "alphas",
            alphas
        )

        self.register_buffer(
            "alpha_bars",
            alpha_bars
        )

        self.register_buffer(
            "sqrt_alpha_bars",
            sqrt_alpha_bars
        )

        self.register_buffer(
            "sqrt_one_minus_alpha_bars",
            sqrt_one_minus_alpha_bars
        )

        self.register_buffer(
            "sqrt_recip_alphas",
            sqrt_recip_alphas
        )

        self.register_buffer(
            "posterior_variance",
            posterior_variance
        )


    # ========================================================
    # EXTRACT VALUES FOR BATCH TIMESTEPS
    # ========================================================

    def extract(
        self,
        values,
        timestep,
        shape
    ):

        values_t = values.gather(
            0,
            timestep
        )

        return values_t.reshape(
            timestep.shape[0],
            *(
                [1] * (len(shape) - 1)
            )
        )


    # ========================================================
    # FORWARD DIFFUSION
    #
    # x₀ → xₜ
    #
    # xₜ =
    # √ᾱₜ x₀
    # +
    # √(1-ᾱₜ) ε
    # ========================================================

    def q_sample(
        self,
        x0,
        timestep,
        noise
    ):

        sqrt_alpha_bar = self.extract(
            self.sqrt_alpha_bars,
            timestep,
            x0.shape
        )

        sqrt_one_minus_alpha_bar = self.extract(
            self.sqrt_one_minus_alpha_bars,
            timestep,
            x0.shape
        )

        x_t = (
            sqrt_alpha_bar * x0
            +
            sqrt_one_minus_alpha_bar * noise
        )

        return x_t


# ============================================================
# CREATE DIFFUSION SCHEDULE
# ============================================================

diffusion_schedule = DiffusionSchedule(
    num_steps=NUM_DIFFUSION_STEPS,
    beta_start=BETA_START,
    beta_end=BETA_END,
    schedule_type=DIFFUSION_SCHEDULE_TYPE
).to(device)


# ============================================================
# INFORMATION
# ============================================================

print(
    "=" * 70
)

print(
    "DIFFUSION SCHEDULE"
)

print(
    "=" * 70
)

print()

print(
    f"Schedule type  : "
    f"{diffusion_schedule.schedule_type}"
)

print(
    f"Diffusion steps: "
    f"{diffusion_schedule.num_steps}"
)

print(
    f"Beta start     : "
    f"{diffusion_schedule.beta_start}"
)

print(
    f"Beta end       : "
    f"{diffusion_schedule.beta_end}"
)

print()

print(
    f"betas shape    : "
    f"{diffusion_schedule.betas.shape}"
)

print(
    f"alpha_bar[0]   : "
    f"{diffusion_schedule.alpha_bars[0].item():.6f}"
)

print(
    f"alpha_bar[-1]  : "
    f"{diffusion_schedule.alpha_bars[-1].item():.6f}"
)


# ============================================================
# TEST FORWARD DIFFUSION
# ============================================================

test_x0 = torch.randn(
    4,
    EMBED_DIM,
    device=device
)

test_noise = torch.randn_like(
    test_x0
)


# ------------------------------------------------------------
# t = 0
# ------------------------------------------------------------

t0 = torch.zeros(
    4,
    dtype=torch.long,
    device=device
)

x_t0 = diffusion_schedule.q_sample(
    test_x0,
    t0,
    test_noise
)


# ------------------------------------------------------------
# t = T-1
# ------------------------------------------------------------

t_last = torch.full(
    (
        4,
    ),
    NUM_DIFFUSION_STEPS - 1,
    dtype=torch.long,
    device=device
)

x_t_last = diffusion_schedule.q_sample(
    test_x0,
    t_last,
    test_noise
)


# ============================================================
# ASSERTIONS
# ============================================================

assert x_t0.shape == (
    4,
    EMBED_DIM
)

assert x_t_last.shape == (
    4,
    EMBED_DIM
)

assert torch.isfinite(
    x_t0
).all()

assert torch.isfinite(
    x_t_last
).all()


# ============================================================
# RESULT
# ============================================================

print()

print(
    "✓ Diffusion Schedule создан"
)

print(
    "✓ Forward diffusion x₀ → xₜ работает"
)

print(
    "✓ Размерности корректны"
)

print(
    "✓ NaN / Inf отсутствуют"
)

print()

print(
    "✓ Ячейка 28-0 готова"
)

DIFFUSION SCHEDULE

Schedule type  : linear
Diffusion steps: 1000
Beta start     : 0.0001
Beta end       : 0.02

betas shape    : torch.Size([1000])
alpha_bar[0]   : 0.999900
alpha_bar[-1]  : 0.000040

✓ Diffusion Schedule создан
✓ Forward diffusion x₀ → xₜ работает
✓ Размерности корректны
✓ NaN / Inf отсутствуют

✓ Ячейка 28-0 готова


In [35]:
# ============================================================
# 28. FORWARD TEST
# ============================================================

model.eval()


batch, properties, fingerprint = next(
    iter(train_loader)
)


batch = batch.to(
    device,
    non_blocking=True
)


properties = properties.to(
    device,
    non_blocking=True
)


fingerprint = fingerprint.to(
    device,
    non_blocking=True
)


# ============================================================
# DIFFUSION TEST PARAMETERS
# ============================================================

TEST_TIMESTEP = torch.full(
    (
        batch.size(0),
    ),
    500,
    dtype=torch.long,
    device=device
)


test_noise = torch.randn(
    batch.size(0),
    EMBED_DIM,
    device=device
)


# ============================================================
# FORWARD PASS
# ============================================================

with torch.no_grad():

    # --------------------------------------------------------
    # 1. Получаем clean latent x₀ RAW
    # --------------------------------------------------------

    x0_raw = model.encoder(
        batch,
        properties,
        fingerprint
    )


    # --------------------------------------------------------
    # 2. Получаем текущий EMA latent RMS
    #
    # В eval() значение не обновляется.
    # Используется сохранённый масштаб модели.
    # --------------------------------------------------------

    latent_rms = (
        model.latent_rms
        .detach()
        .clamp_min(1e-6)
    )


    # --------------------------------------------------------
    # 3. Нормализуем latent
    #
    # x₀ raw
    #    ↓
    # / EMA RMS
    #    ↓
    # x₀ normalized
    # --------------------------------------------------------

    x0_norm = (
        x0_raw
        / latent_rms
    )


    # --------------------------------------------------------
    # 4. Forward diffusion
    #
    # x₀ normalized → xₜ
    # --------------------------------------------------------

    x_t = diffusion_schedule.q_sample(
        x0_norm,
        TEST_TIMESTEP,
        test_noise
    )


    # --------------------------------------------------------
    # 5. Predict noise
    #
    # xₜ + timestep + properties
    # → predicted noise
    # --------------------------------------------------------

    predicted_noise = model.diffusion(
        x_t,
        TEST_TIMESTEP,
        properties
    )


# ============================================================
# OUTPUT SHAPES
# ============================================================

print(
    "Input:",
    batch.shape
)

print(
    "Properties:",
    properties.shape
)

print(
    "Fingerprint:",
    fingerprint.shape
)

print(
    "Clean latent x₀ raw:",
    x0_raw.shape
)

print(
    "Normalized latent x₀:",
    x0_norm.shape
)

print(
    "Noisy latent xₜ:",
    x_t.shape
)

print(
    "Predicted noise:",
    predicted_noise.shape
)


# ============================================================
# LATENT STATISTICS
# ============================================================

print()

print(
    "LATENT NORMALIZATION:"
)

print(
    f"  EMA latent RMS : "
    f"{latent_rms.item():.6f}"
)

print(
    f"  x₀ raw mean    : "
    f"{x0_raw.mean().item():.6f}"
)

print(
    f"  x₀ raw std     : "
    f"{x0_raw.std().item():.6f}"
)

print(
    f"  x₀ norm mean   : "
    f"{x0_norm.mean().item():.6f}"
)

print(
    f"  x₀ norm std    : "
    f"{x0_norm.std().item():.6f}"
)

print(
    f"  xₜ std         : "
    f"{x_t.std().item():.6f}"
)

print(
    f"  noise std      : "
    f"{test_noise.std().item():.6f}"
)


# ============================================================
# ASSERTIONS
# ============================================================

assert batch.shape == (
    BATCH_SIZE,
    MAX_LEN
)

assert properties.shape == (
    BATCH_SIZE,
    PROPERTY_DIM
)

assert fingerprint.shape == (
    BATCH_SIZE,
    FP_BITS
)

assert x0_raw.shape == (
    BATCH_SIZE,
    EMBED_DIM
)

assert x0_norm.shape == (
    BATCH_SIZE,
    EMBED_DIM
)

assert x_t.shape == (
    BATCH_SIZE,
    EMBED_DIM
)

assert predicted_noise.shape == (
    BATCH_SIZE,
    EMBED_DIM
)


# ============================================================
# NORMALIZATION CONSISTENCY CHECK
# ============================================================

expected_norm_std = (
    x0_raw.std()
    / latent_rms
)


normalization_error = abs(
    x0_norm.std().item()
    - expected_norm_std.item()
)


assert normalization_error < 1e-5


# ============================================================
# NUMERICAL CHECKS
# ============================================================

assert torch.isfinite(
    x0_raw
).all()

assert torch.isfinite(
    x0_norm
).all()

assert torch.isfinite(
    x_t
).all()

assert torch.isfinite(
    predicted_noise
).all()


# ============================================================
# TEST LOSS
# ============================================================

test_loss = nn.functional.mse_loss(
    predicted_noise,
    test_noise
)


print()

print(
    f"Test diffusion loss: "
    f"{test_loss.item():.6f}"
)


assert torch.isfinite(
    test_loss
)

assert test_loss.item() >= 0.0


# ============================================================
# RESULT
# ============================================================

print()

print(
    "✓ Fusion Encoder → raw x₀ работает"
)

print(
    "✓ EMA latent normalization работает"
)

print(
    "✓ Forward diffusion x₀ normalized → xₜ работает"
)

print(
    "✓ Diffusion Transformer работает"
)

print(
    "✓ Noise prediction работает"
)

print(
    "✓ Diffusion loss рассчитывается"
)

print(
    "✓ Полный forward diffusion pipeline корректен"
)



Input: torch.Size([32, 66])
Properties: torch.Size([32, 11])
Fingerprint: torch.Size([32, 2048])
Clean latent x₀ raw: torch.Size([32, 128])
Normalized latent x₀: torch.Size([32, 128])
Noisy latent xₜ: torch.Size([32, 128])
Predicted noise: torch.Size([32, 128])

LATENT NORMALIZATION:
  EMA latent RMS : 0.180000
  x₀ raw mean    : 0.001118
  x₀ raw std     : 0.388482
  x₀ norm mean   : 0.006212
  x₀ norm std    : 2.158231
  xₜ std         : 1.129801
  noise std      : 0.993709

Test diffusion loss: 1.327341

✓ Fusion Encoder → raw x₀ работает
✓ EMA latent normalization работает
✓ Forward diffusion x₀ normalized → xₜ работает
✓ Diffusion Transformer работает
✓ Noise prediction работает
✓ Diffusion loss рассчитывается
✓ Полный forward diffusion pipeline корректен


Ячейка 28 — Loss

In [36]:
# ============================================================
# 29. DIFFUSION + X0 + RECONSTRUCTION LOSS
# ============================================================

# ============================================================
# CONFIGURATION
# ============================================================

RECON_WEIGHT = 1.0

X0_LOSS_WEIGHT = 0.01

SNR_GAMMA = 5.0

MIN_DIFFUSION_WEIGHT = 0.1


# ============================================================
# SELFIES RECONSTRUCTION LOSS
# ============================================================

reconstruction_criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


def reconstruction_loss(
    reconstruction_logits,
    target
):

    # --------------------------------------------------------
    # reconstruction_logits:
    #
    # [batch, seq_len - 1, vocab_size]
    #
    # target:
    #
    # [batch, seq_len - 1]
    # --------------------------------------------------------

    loss = reconstruction_criterion(
        reconstruction_logits.reshape(
            -1,
            reconstruction_logits.size(-1)
        ),
        target.reshape(-1)
    )

    return loss


# ============================================================
# DIFFUSION LOSS
#
# Min-SNR weighting
#
# Основная цель по-прежнему:
#
#     predicted_noise → true noise
#
# Но вклад каждого timestep зависит от SNR.
# ============================================================

def diffusion_loss(
    predicted_noise,
    noise,
    timestep,
    diffusion_schedule
):

    # --------------------------------------------------------
    # SNR(t) = alpha_bar / (1 - alpha_bar)
    # --------------------------------------------------------

    alpha_bar = (
        diffusion_schedule.alpha_bars[
            timestep
        ]
    )

    snr = (
        alpha_bar
        /
        (1.0 - alpha_bar).clamp_min(1e-8)
    )


    # --------------------------------------------------------
    # Min-SNR weighting
    #
    # weight = min(SNR, gamma) / SNR
    #
    # Это уменьшает чрезмерный вклад timestep
    # с очень высоким SNR.
    # --------------------------------------------------------

    snr_weight = (
        torch.minimum(
            snr,
            torch.full_like(
                snr,
                SNR_GAMMA
            )
        )
        /
        snr.clamp_min(1e-8)
    )


    # --------------------------------------------------------
    # Минимальный вес
    #
    # Не позволяем отдельным timestep получить
    # практически нулевой вклад.
    # --------------------------------------------------------

    snr_weight = torch.clamp(
        snr_weight,
        min=MIN_DIFFUSION_WEIGHT
    )


    # --------------------------------------------------------
    # Per-sample MSE
    # --------------------------------------------------------

    mse = (
        (predicted_noise - noise) ** 2
    ).mean(
        dim=tuple(
            range(1, predicted_noise.ndim)
        )
    )


    # --------------------------------------------------------
    # Weighted diffusion loss
    # --------------------------------------------------------

    loss = (
        snr_weight * mse
    ).mean()


    return loss


# ============================================================
# X0 PREDICTION LOSS
#
# Из predicted noise восстанавливаем x₀:
#
# x₀_hat =
#
# (xₜ - sqrt(1-alpha_bar) * epsilon_hat)
# ---------------------------------------
#              sqrt(alpha_bar)
#
# Сравниваем x₀_hat с настоящим normalized x₀.
# ============================================================

def x0_prediction_loss(
    predicted_noise,
    x_t,
    x0_norm,
    timestep,
    diffusion_schedule
):

    # --------------------------------------------------------
    # alpha_bar(t)
    # --------------------------------------------------------

    alpha_bar = (
        diffusion_schedule.alpha_bars[
            timestep
        ]
    )


    # --------------------------------------------------------
    # Reshape for latent dimension
    # --------------------------------------------------------

    sqrt_alpha_bar = torch.sqrt(
        alpha_bar
    ).unsqueeze(1)


    sqrt_one_minus_alpha_bar = torch.sqrt(
        1.0 - alpha_bar
    ).unsqueeze(1)


    # --------------------------------------------------------
    # Reconstruct x₀ from predicted epsilon
    # --------------------------------------------------------

    predicted_x0 = (
        x_t
        -
        sqrt_one_minus_alpha_bar
        * predicted_noise
    ) / sqrt_alpha_bar.clamp_min(1e-5)


    # --------------------------------------------------------
    # Per-sample MSE
    # --------------------------------------------------------

    loss = (
        (predicted_x0 - x0_norm) ** 2
    ).mean(
        dim=1
    )


    return loss.mean()


# ============================================================
# TOTAL LOSS
# ============================================================

def total_loss(
    predicted_noise,
    noise,
    x_t,
    x0_norm,
    timestep,
    reconstruction_logits,
    target,
    diffusion_schedule
):

    # --------------------------------------------------------
    # 1. Min-SNR weighted diffusion loss
    # --------------------------------------------------------

    loss_diffusion = diffusion_loss(
        predicted_noise,
        noise,
        timestep,
        diffusion_schedule
    )


    # --------------------------------------------------------
    # 2. Auxiliary x₀ prediction loss
    # --------------------------------------------------------

    loss_x0 = x0_prediction_loss(
        predicted_noise,
        x_t,
        x0_norm,
        timestep,
        diffusion_schedule
    )


    # --------------------------------------------------------
    # 3. SELFIES reconstruction loss
    # --------------------------------------------------------

    loss_reconstruction = reconstruction_loss(
        reconstruction_logits,
        target
    )


    # --------------------------------------------------------
    # 4. TOTAL LOSS
    # --------------------------------------------------------

    loss = (
        loss_diffusion
        +
        X0_LOSS_WEIGHT * loss_x0
        +
        RECON_WEIGHT * loss_reconstruction
    )


    return (
        loss,
        loss_diffusion,
        loss_x0,
        loss_reconstruction
    )


# ============================================================
# INFORMATION
# ============================================================

print(
    "=" * 70
)

print(
    "DIFFUSION + X0 + RECONSTRUCTION LOSS"
)

print(
    "=" * 70
)

print()

print(
    "Diffusion loss       : Min-SNR weighted MSE"
)

print(
    "Diffusion target     : Gaussian noise"
)

print(
    "SNR gamma            :",
    SNR_GAMMA
)

print(
    "Minimum diffusion weight:",
    MIN_DIFFUSION_WEIGHT
)

print(
    "X0 prediction loss   : MSE"
)

print(
    "X0 loss weight       :",
    X0_LOSS_WEIGHT
)

print(
    "Reconstruction loss  : CrossEntropy"
)

print(
    "Reconstruction target: SELFIES tokens"
)

print(
    "Reconstruction weight:",
    RECON_WEIGHT
)

print(
    "Padding              : ignored"
)

print(
    "KL divergence        : disabled"
)

print(
    "VAE beta             : not used"
)

print()

print(
    "Total loss:"
)

print(
    "L = L_diffusion "
    "+ "
    f"{X0_LOSS_WEIGHT} × L_x0 "
    "+ "
    f"{RECON_WEIGHT} × L_reconstruction"
)

print()

print(
    "✓ Min-SNR diffusion loss создан"
)

print(
    "✓ X0 prediction loss создан"
)

print(
    "✓ SELFIES reconstruction loss создан"
)

print(
    "✓ Combined loss создан"
)

DIFFUSION + X0 + RECONSTRUCTION LOSS

Diffusion loss       : Min-SNR weighted MSE
Diffusion target     : Gaussian noise
SNR gamma            : 5.0
Minimum diffusion weight: 0.1
X0 prediction loss   : MSE
X0 loss weight       : 0.01
Reconstruction loss  : CrossEntropy
Reconstruction target: SELFIES tokens
Reconstruction weight: 1.0
Padding              : ignored
KL divergence        : disabled
VAE beta             : not used

Total loss:
L = L_diffusion + 0.01 × L_x0 + 1.0 × L_reconstruction

✓ Min-SNR diffusion loss создан
✓ X0 prediction loss создан
✓ SELFIES reconstruction loss создан
✓ Combined loss создан


Ячейка 29 — проверка loss

In [37]:
# ============================================================
# 30. DIFFUSION + X0 + RECONSTRUCTION LOSS TEST
# ============================================================

model.eval()


# ============================================================
# PREPARE TEST DATA
# ============================================================

batch, properties, fingerprint = next(
    iter(train_loader)
)

batch = batch.to(
    device,
    non_blocking=True
)

properties = properties.to(
    device,
    non_blocking=True
)

fingerprint = fingerprint.to(
    device,
    non_blocking=True
)


# ============================================================
# TIMESTEP
# ============================================================

TEST_TIMESTEP = torch.full(
    (
        batch.size(0),
    ),
    500,
    dtype=torch.long,
    device=device
)


# ============================================================
# RANDOM NOISE
# ============================================================

test_noise = torch.randn(
    batch.size(0),
    EMBED_DIM,
    device=device
)


# ============================================================
# FORWARD PASS
# ============================================================

with torch.no_grad():

    (
        predicted_noise,
        reconstruction_logits,
        x0_raw,
        x0_norm,
        x_t
    ) = model(
        batch,
        properties,
        fingerprint,
        TEST_TIMESTEP,
        test_noise,
        diffusion_schedule
    )


# ============================================================
# RECONSTRUCTION TARGET
# ============================================================

reconstruction_target = (
    batch[:, 1:]
)


# ============================================================
# CALCULATE LOSSES
# ============================================================

(
    loss,
    loss_diffusion,
    loss_x0,
    loss_reconstruction
) = total_loss(

    predicted_noise,

    test_noise,

    x_t,

    x0_norm,

    TEST_TIMESTEP,

    reconstruction_logits,

    reconstruction_target,

    diffusion_schedule
)


# ============================================================
# RESULTS
# ============================================================

print(
    "Input SELFIES:",
    batch.shape
)

print(
    "Properties:",
    properties.shape
)

print(
    "Fingerprint:",
    fingerprint.shape
)

print(
    "Clean latent x₀ raw:",
    x0_raw.shape
)

print(
    "Normalized latent x₀:",
    x0_norm.shape
)

print(
    "Noisy latent xₜ:",
    x_t.shape
)

print(
    "Predicted noise:",
    predicted_noise.shape
)

print(
    "Reconstruction logits:",
    reconstruction_logits.shape
)

print(
    "Reconstruction target:",
    reconstruction_target.shape
)

print()

print(
    "Diffusion loss (Min-SNR):",
    loss_diffusion.item()
)

print(
    "X0 prediction loss:",
    loss_x0.item()
)

print(
    "X0 loss weight:",
    X0_LOSS_WEIGHT
)

print(
    "Reconstruction loss:",
    loss_reconstruction.item()
)

print(
    "Reconstruction weight:",
    RECON_WEIGHT
)

print(
    "Total loss:",
    loss.item()
)


# ============================================================
# LOSS DECOMPOSITION
# ============================================================

weighted_x0_loss = (
    X0_LOSS_WEIGHT
    * loss_x0
).item()

weighted_reconstruction_loss = (
    RECON_WEIGHT
    * loss_reconstruction
).item()

print()

print(
    "LOSS DECOMPOSITION:"
)

print(
    f"  Diffusion       : {loss_diffusion.item():.6f}"
)

print(
    f"  X0 weighted     : {weighted_x0_loss:.6f}"
)

print(
    f"  Reconstruction  : "
    f"{weighted_reconstruction_loss:.6f}"
)

print(
    f"  Total           : {loss.item():.6f}"
)


# ============================================================
# LATENT NORMALIZATION
# ============================================================

print()

print(
    "LATENT NORMALIZATION:"
)

print(
    "  EMA latent RMS:",
    model.latent_rms.item()
)

print(
    "  x₀ raw mean:",
    x0_raw.mean().item()
)

print(
    "  x₀ raw std:",
    x0_raw.std().item()
)

print(
    "  x₀ raw range:",
    (
        x0_raw.min().item(),
        x0_raw.max().item()
    )
)

print()

print(
    "  x₀ norm mean:",
    x0_norm.mean().item()
)

print(
    "  x₀ norm std:",
    x0_norm.std().item()
)

print(
    "  x₀ norm range:",
    (
        x0_norm.min().item(),
        x0_norm.max().item()
    )
)


# ============================================================
# MIN-SNR DIAGNOSTICS
# ============================================================

alpha_bar = (
    diffusion_schedule.alpha_bars[
        TEST_TIMESTEP
    ]
)

snr = (
    alpha_bar
    /
    (1.0 - alpha_bar).clamp_min(1e-8)
)

snr_weight = (
    torch.minimum(
        snr,
        torch.full_like(
            snr,
            SNR_GAMMA
        )
    )
    /
    snr.clamp_min(1e-8)
)

snr_weight = torch.clamp(
    snr_weight,
    min=MIN_DIFFUSION_WEIGHT
)

print()

print(
    "MIN-SNR:"
)

print(
    f"  Timestep          : "
    f"{TEST_TIMESTEP[0].item()}"
)

print(
    f"  SNR               : "
    f"{snr.mean().item():.6f}"
)

print(
    f"  SNR gamma         : "
    f"{SNR_GAMMA:.6f}"
)

print(
    f"  Diffusion weight  : "
    f"{snr_weight.mean().item():.6f}"
)


# ============================================================
# ASSERTIONS
# ============================================================

assert x0_raw.shape == (
    batch.size(0),
    EMBED_DIM
)

assert x0_norm.shape == (
    batch.size(0),
    EMBED_DIM
)

assert x_t.shape == (
    batch.size(0),
    EMBED_DIM
)

assert predicted_noise.shape == (
    batch.size(0),
    EMBED_DIM
)

assert test_noise.shape == (
    batch.size(0),
    EMBED_DIM
)

assert reconstruction_logits.shape == (
    batch.size(0),
    MAX_LEN - 1,
    VOCAB_SIZE
)

assert reconstruction_target.shape == (
    batch.size(0),
    MAX_LEN - 1
)


# ============================================================
# NUMERICAL CHECKS
# ============================================================

assert torch.isfinite(
    x0_raw
).all()

assert torch.isfinite(
    x0_norm
).all()

assert torch.isfinite(
    x_t
).all()

assert torch.isfinite(
    predicted_noise
).all()

assert torch.isfinite(
    reconstruction_logits
).all()

assert torch.isfinite(
    loss_diffusion
)

assert torch.isfinite(
    loss_x0
)

assert torch.isfinite(
    loss_reconstruction
)

assert torch.isfinite(
    loss


)

assert loss_diffusion.item() >= 0
assert loss_x0.item() >= 0
assert loss_reconstruction.item() >= 0
assert loss.item() >= 0


# ============================================================
# LATENT NORMALIZATION CHECK
# ============================================================

latent_rms = (
    model.latent_rms
    .detach()
    .clamp_min(1e-6)
)


expected_norm_std = (
    x0_raw.std()
    / latent_rms
)


normalization_error = abs(
    x0_norm.std().item()
    - expected_norm_std.item()
)


print()

print(
    "Expected normalized std:",
    expected_norm_std.item()
)

print(
    "Actual normalized std:",
    x0_norm.std().item()
)

print(
    "Normalization error:",
    normalization_error
)


assert normalization_error < 1e-5


# ============================================================
# RESULT
# ============================================================

print()

print(
    "✓ Encoder → raw clean latent x₀ работает"
)

print(
    "✓ raw x₀ → SELFIES Decoder работает"
)

print(
    "✓ raw x₀ → EMA normalization работает"
)

print(
    "✓ normalized x₀ → xₜ работает"
)

print(
    "✓ Diffusion Transformer работает"
)

print(
    "✓ Noise prediction работает"
)

print(
    "✓ Min-SNR diffusion loss работает"
)

print(
    "✓ X0 prediction loss работает"
)

print(
    "✓ Reconstruction loss работает"
)

print(
    "✓ Combined loss работает"
)

print(
    "✓ Полный forward pipeline корректен"
)

Input SELFIES: torch.Size([32, 66])
Properties: torch.Size([32, 11])
Fingerprint: torch.Size([32, 2048])
Clean latent x₀ raw: torch.Size([32, 128])
Normalized latent x₀: torch.Size([32, 128])
Noisy latent xₜ: torch.Size([32, 128])
Predicted noise: torch.Size([32, 128])
Reconstruction logits: torch.Size([32, 65, 109])
Reconstruction target: torch.Size([32, 65])

Diffusion loss (Min-SNR): 1.371910572052002
X0 prediction loss: 16.262659072875977
X0 loss weight: 0.01
Reconstruction loss: 5.457822322845459
Reconstruction weight: 1.0
Total loss: 6.992359638214111

LOSS DECOMPOSITION:
  Diffusion       : 1.371911
  X0 weighted     : 0.162627
  Reconstruction  : 5.457822
  Total           : 6.992360

LATENT NORMALIZATION:
  EMA latent RMS: 0.18000000715255737
  x₀ raw mean: 0.007441595196723938
  x₀ raw std: 0.38427475094795227
  x₀ raw range: (-1.24313485622406, 1.284708023071289)

  x₀ norm mean: 0.04134218394756317
  x₀ norm std: 2.134859561920166
  x₀ norm range: (-6.906304359436035, 7.137

In [39]:
# ============================================================
# 31. CONDITIONAL LATENT DIFFUSION MODEL CONFIGURATION
# ============================================================

# ============================================================
# OFF-MANIFOLD TRAINING PARAMETERS
# ============================================================

PERTURBATION_PROB = 0.5
PERTURBATION_STD = 0.01


print("=" * 70)
print("CONDITIONAL LATENT DIFFUSION MODEL CONFIGURATION")
print("=" * 70)

print()


# ============================================================
# MODEL
# ============================================================

print("Architecture:")

print(
    "  Encoder            : SELFIES + Fingerprint + Descriptors"
)

print(
    "  Clean latent       : raw x₀"
)

print(
    "  Decoder            : raw x₀ → SELFIES"
)

print(
    "  Latent normalization:"
)

print(
    "    raw x₀ → x₀ / EMA latent RMS"
)

print(
    "  Diffusion          : normalized x₀ → xₜ → predicted noise"
)

print()


# ============================================================
# LOSS
# ============================================================

print("Loss:")

print(
    "  Diffusion loss     : Min-SNR weighted MSE"
)

print(
    "  X0 prediction loss : MSE"
)

print(
    "  X0 loss weight     :",
    X0_LOSS_WEIGHT
)

print(
    "  Reconstruction     : CrossEntropy"
)

print(
    "  Reconstruction weight:",
    RECON_WEIGHT
)

print(
    "  SNR gamma          :",
    SNR_GAMMA
)

print(
    "  Minimum diffusion weight:",
    MIN_DIFFUSION_WEIGHT
)

print(
    "  Total loss         : "
    "L_diffusion + "
    f"{X0_LOSS_WEIGHT} × L_x0 + "
    f"{RECON_WEIGHT} × L_reconstruction"
)

print()


# ============================================================
# OFF-MANIFOLD TRAINING
# ============================================================

print("Off-manifold training:")

print(
    "  Enabled            : YES"
)

print(
    "  Perturbation prob. :",
    PERTURBATION_PROB
)

print(
    "  Perturbation std   :",
    PERTURBATION_STD
)

print(
    "  Training input     : xₜ + δ"
)

print(
    "  Validation input   : clean xₜ"
)

print(
    "  Perturbation       : Gaussian"
)

print()


# ============================================================
# VAE COMPONENTS
# ============================================================

print("VAE components:")

print(
    "  KL divergence      : disabled"
)

print(
    "  VAE beta           : not used"
)

print(
    "  Latent sampling    : disabled"
)

print(
    "  μ / logvar         : not used"
)

print()


# ============================================================
# LATENT
# ============================================================

print("Latent:")

print(
    "  Latent dimension   :",
    EMBED_DIM
)

print(
    "  Representation     : deterministic raw x₀"
)

print(
    "  EMA latent RMS     :",
    f"{model.latent_rms.item():.6f}"
)

print(
    "  EMA momentum       :",
    f"{model.latent_rms_momentum:.4f}"
)

print()


# ============================================================
# DIFFUSION
# ============================================================

print("Diffusion:")

print(
    "  Diffusion steps    :",
    NUM_DIFFUSION_STEPS
)

print(
    "  Beta start         :",
    BETA_START
)

print(
    "  Beta end           :",
    BETA_END
)

print(
    "  Schedule           :",
    DIFFUSION_SCHEDULE_TYPE
)

print()


# ============================================================
# CONDITIONS
# ============================================================

print("Conditions:")

print(
    "  Target descriptors :",
    PROPERTY_DIM
)

print(
    "  Fingerprint bits   :",
    FP_BITS
)

print()


# ============================================================
# NORMALIZATION
# ============================================================

print("Latent normalization:")

print(
    "  Method             : EMA global RMS"
)

print(
    "  Initial RMS        :",
    f"{INITIAL_LATENT_RMS:.6f}"
)

print(
    "  EMA momentum       :",
    f"{LATENT_RMS_MOMENTUM:.4f}"
)

print(
    "  Fixed LATENT_SCALE : disabled"
)

print()


# ============================================================
# RESULT
# ============================================================

print(
    "✓ Encoder configuration проверена"
)

print(
    "✓ SELFIES Decoder configuration проверена"
)

print(
    "✓ EMA latent normalization configuration проверена"
)

print(
    "✓ Diffusion configuration проверена"
)

print(
    "✓ Min-SNR loss configuration проверена"
)

print(
    "✓ X0 prediction loss configuration проверена"
)

print(
    "✓ Reconstruction loss configuration проверена"
)

print(
    "✓ Off-manifold training configuration проверена"
)

print(
    "✓ VAE components disabled"
)

print(
    "✓ Fixed latent scaling disabled"
)

CONDITIONAL LATENT DIFFUSION MODEL CONFIGURATION

Architecture:
  Encoder            : SELFIES + Fingerprint + Descriptors
  Clean latent       : raw x₀
  Decoder            : raw x₀ → SELFIES
  Latent normalization:
    raw x₀ → x₀ / EMA latent RMS
  Diffusion          : normalized x₀ → xₜ → predicted noise

Loss:
  Diffusion loss     : Min-SNR weighted MSE
  X0 prediction loss : MSE
  X0 loss weight     : 0.01
  Reconstruction     : CrossEntropy
  Reconstruction weight: 1.0
  SNR gamma          : 5.0
  Minimum diffusion weight: 0.1
  Total loss         : L_diffusion + 0.01 × L_x0 + 1.0 × L_reconstruction

Off-manifold training:
  Enabled            : YES
  Perturbation prob. : 0.5
  Perturbation std   : 0.01
  Training input     : xₜ + δ
  Validation input   : clean xₜ
  Perturbation       : Gaussian

VAE components:
  KL divergence      : disabled
  VAE beta           : not used
  Latent sampling    : disabled
  μ / logvar         : not used

Latent:
  Latent dimension   : 128
  Rep

Ячейка 31 — optimizer

In [40]:
# ============================================================
# 32. DIFFUSION MODEL OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)


print(
    "Optimizer: AdamW"
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    1e-5
)

print(
    "\n✓ Diffusion model optimizer создан"
)

Optimizer: AdamW
Learning rate: 0.0003
Weight decay: 1e-05

✓ Diffusion model optimizer создан


Ячейка 32 — train_epoch

In [41]:
# ============================================================
# 33. TRAIN EPOCH — LATENT DIFFUSION + X0 + RECONSTRUCTION
#     + OFF-MANIFOLD TRAINING
# ============================================================

def train_epoch(
    model,
    dataloader,
    optimizer,
    device,
    diffusion_schedule
):

    model.train()

    total_loss_sum = 0.0
    diffusion_loss_sum = 0.0
    x0_loss_sum = 0.0
    reconstruction_loss_sum = 0.0

    last_grad_norm = 0.0

    progress_bar = tqdm(
        dataloader,
        desc="Train",
        leave=False
    )

    for batch, properties, fingerprint in progress_bar:

        # ----------------------------------------------------
        # Device
        # ----------------------------------------------------

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # RANDOM DIFFUSION TIMESTEP
        # ----------------------------------------------------

        timestep = torch.randint(
            0,
            diffusion_schedule.num_steps,
            (
                batch.size(0),
            ),
            device=device,
            dtype=torch.long
        )

        # ----------------------------------------------------
        # RANDOM GAUSSIAN NOISE
        # ----------------------------------------------------

        noise = torch.randn(
            batch.size(0),
            EMBED_DIM,
            device=device
        )

        # ----------------------------------------------------
        # FULL MODEL FORWARD
        #
        # Encoder:
        #   SELFIES + properties + fingerprint → raw x₀
        #
        # Decoder:
        #   raw x₀ → reconstructed SELFIES
        #
        # Latent normalization:
        #   raw x₀ → x₀ / EMA latent RMS
        #
        # Diffusion:
        #   normalized x₀ → xₜ
        #
        # Off-manifold training:
        #   xₜ → xₜ + δ
        #
        # IMPORTANT:
        #   x_t returned by model remains CLEAN.
        #   Perturbation is applied only to the
        #   diffusion model input.
        # ----------------------------------------------------

        (
            predicted_noise,
            reconstruction_logits,
            x0_raw,
            x0_norm,
            x_t
        ) = model(
            batch,
            properties,
            fingerprint,
            timestep,
            noise,
            diffusion_schedule,
            perturbation_prob=PERTURBATION_PROB,
            perturbation_std=PERTURBATION_STD
        )

        # ----------------------------------------------------
        # RECONSTRUCTION TARGET
        #
        # Decoder input:
        #   <START> ... tokens before the last token
        #
        # Target:
        #   next token at every position
        # ----------------------------------------------------

        reconstruction_target = batch[:, 1:]

        # ----------------------------------------------------
        # TOTAL LOSS
        #
        # L =
        #   L_diffusion
        #   + X0_LOSS_WEIGHT × L_x0
        #   + RECON_WEIGHT × L_reconstruction
        # ----------------------------------------------------

        (
            loss,
            loss_diffusion,
            loss_x0,
            loss_reconstruction
        ) = total_loss(

            predicted_noise,

            noise,

            x_t,

            x0_norm,

            timestep,

            reconstruction_logits,

            reconstruction_target,

            diffusion_schedule
        )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        last_grad_norm = float(
            grad_norm
        )

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss_sum += loss.item()

        diffusion_loss_sum += (
            loss_diffusion.item()
        )

        x0_loss_sum += (
            loss_x0.item()
        )

        reconstruction_loss_sum += (
            loss_reconstruction.item()
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            diff=f"{loss_diffusion.item():.4f}",
            x0=f"{loss_x0.item():.4f}",
            recon=f"{loss_reconstruction.item():.4f}",
            grad=f"{last_grad_norm:.3f}"
        )

    n_batches = len(
        dataloader
    )

    return (
        total_loss_sum / n_batches,
        diffusion_loss_sum / n_batches,
        x0_loss_sum / n_batches,
        reconstruction_loss_sum / n_batches,
        last_grad_norm
    )

Ячейка 33 — validation

In [42]:
# ============================================================
# 34. VALIDATION — LATENT DIFFUSION + X0 + RECONSTRUCTION
# ============================================================

@torch.no_grad()
def validate_epoch(
    model,
    dataloader,
    device,
    diffusion_schedule
):

    model.eval()

    total_loss_sum = 0.0
    diffusion_loss_sum = 0.0
    x0_loss_sum = 0.0
    reconstruction_loss_sum = 0.0

    progress_bar = tqdm(
        dataloader,
        desc="Validation",
        leave=False
    )

    for batch, properties, fingerprint in progress_bar:

        # ----------------------------------------------------
        # Device
        # ----------------------------------------------------

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # RANDOM DIFFUSION TIMESTEP
        # ----------------------------------------------------

        timestep = torch.randint(
            0,
            diffusion_schedule.num_steps,
            (
                batch.size(0),
            ),
            device=device,
            dtype=torch.long
        )

        # ----------------------------------------------------
        # RANDOM GAUSSIAN NOISE
        # ----------------------------------------------------

        noise = torch.randn(
            batch.size(0),
            EMBED_DIM,
            device=device
        )

        # ----------------------------------------------------
        # FULL MODEL FORWARD
        #
        # Encoder:
        #   SELFIES + properties + fingerprint → raw x₀
        #
        # Decoder:
        #   raw x₀ → reconstructed SELFIES
        #
        # Latent normalization:
        #   raw x₀ → x₀ / EMA latent RMS
        #
        # Diffusion:
        #   normalized x₀ → xₜ → predicted noise
        # ----------------------------------------------------

        (
            predicted_noise,
            reconstruction_logits,
            x0_raw,
            x0_norm,
            x_t
        ) = model(
            batch,
            properties,
            fingerprint,
            timestep,
            noise,
            diffusion_schedule
        )

        # ----------------------------------------------------
        # RECONSTRUCTION TARGET
        # ----------------------------------------------------

        reconstruction_target = batch[:, 1:]

        # ----------------------------------------------------
        # TOTAL LOSS
        #
        # L =
        #   L_diffusion
        #   + X0_LOSS_WEIGHT × L_x0
        #   + RECON_WEIGHT × L_reconstruction
        # ----------------------------------------------------

        (
            loss,
            loss_diffusion,
            loss_x0,
            loss_reconstruction
        ) = total_loss(

            predicted_noise,

            noise,

            x_t,

            x0_norm,

            timestep,

            reconstruction_logits,

            reconstruction_target,

            diffusion_schedule
        )

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss_sum += (
            loss.item()
        )

        diffusion_loss_sum += (
            loss_diffusion.item()
        )

        x0_loss_sum += (
            loss_x0.item()
        )

        reconstruction_loss_sum += (
            loss_reconstruction.item()
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            diff=f"{loss_diffusion.item():.4f}",
            x0=f"{loss_x0.item():.4f}",
            recon=f"{loss_reconstruction.item():.4f}"
        )

    n_batches = len(
        dataloader
    )

    return (
        total_loss_sum / n_batches,
        diffusion_loss_sum / n_batches,
        x0_loss_sum / n_batches,
        reconstruction_loss_sum / n_batches
    )

Ячейка 34 — latent statistics

In [43]:
# ============================================================
# 35. CLEAN LATENT x₀ STATISTICS
# ============================================================

@torch.no_grad()
def get_latent_statistics(
    model,
    dataloader,
    device,
    max_batches=20
):

    model.eval()

    raw_latents = []
    normalized_latents = []

    # --------------------------------------------------------
    # Current EMA latent RMS
    # --------------------------------------------------------

    latent_rms = (
        model.latent_rms
        .detach()
        .clamp_min(1e-6)
    )

    for batch_idx, (
        batch,
        properties,
        fingerprint
    ) in enumerate(
        dataloader
    ):

        if batch_idx >= max_batches:
            break

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # RAW CLEAN LATENT x₀
        #
        # This is the latent used by the SELFIES decoder.
        # ----------------------------------------------------

        x0_raw = model.encoder(
            batch,
            properties,
            fingerprint
        )

        # ----------------------------------------------------
        # NORMALIZED CLEAN LATENT
        #
        # x₀_norm = x₀_raw / EMA latent RMS
        #
        # This is the latent used by diffusion.
        # ----------------------------------------------------

        x0_norm = (
            x0_raw / latent_rms
        )

        raw_latents.append(
            x0_raw.cpu()
        )

        normalized_latents.append(
            x0_norm.cpu()
        )

    if not raw_latents:

        raise RuntimeError(
            "Не удалось получить "
            "clean latent x₀ statistics"
        )

    raw_latents = torch.cat(
        raw_latents,
        dim=0
    )

    normalized_latents = torch.cat(
        normalized_latents,
        dim=0
    )

    # ========================================================
    # RESULTS
    # ========================================================

    print("=" * 70)
    print("CLEAN LATENT x₀ STATISTICS")
    print("=" * 70)

    print(
        f"Samples       : {raw_latents.shape[0]}"
    )

    print(
        f"Latent dim    : {raw_latents.shape[1]}"
    )

    print()

    print("RAW x₀")
    print(
        f"  mean        : {raw_latents.mean().item():.6f}"
    )
    print(
        f"  std         : {raw_latents.std().item():.6f}"
    )
    print(
        f"  min         : {raw_latents.min().item():.6f}"
    )
    print(
        f"  max         : {raw_latents.max().item():.6f}"
    )

    print()

    print("NORMALIZED x₀")
    print(
        f"  mean        : {normalized_latents.mean().item():.6f}"
    )
    print(
        f"  std         : {normalized_latents.std().item():.6f}"
    )
    print(
        f"  min         : {normalized_latents.min().item():.6f}"
    )
    print(
        f"  max         : {normalized_latents.max().item():.6f}"
    )

    print()

    print(
        f"EMA latent RMS : {latent_rms.item():.6f}"
    )

    # --------------------------------------------------------
    # Consistency check
    # --------------------------------------------------------

    expected_normalized_std = (
        raw_latents.std() / latent_rms.cpu()
    )

    normalization_error = abs(
        normalized_latents.std().item()
        - expected_normalized_std.item()
    )

    print()

    print(
        f"Expected norm std : "
        f"{expected_normalized_std.item():.6f}"
    )

    print(
        f"Normalization error: "
        f"{normalization_error:.8f}"
    )

    print("=" * 70)

    return {

        "x0_mean":
            raw_latents.mean().item(),

        "x0_std":
            raw_latents.std().item(),

        "x0_min":
            raw_latents.min().item(),

        "x0_max":
            raw_latents.max().item(),

        "x0_norm_mean":
            normalized_latents.mean().item(),

        "x0_norm_std":
            normalized_latents.std().item(),

        "x0_norm_min":
            normalized_latents.min().item(),

        "x0_norm_max":
            normalized_latents.max().item(),

        "latent_rms":
            latent_rms.item()
    }

Ячейка 35 — checkpoint

In [44]:
# ============================================================
# 36. CONDITIONAL LATENT DIFFUSION CHECKPOINT
#     + OFF-MANIFOLD TRAINING
# ============================================================

def save_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    best_val_loss,
    history,
    scaler,
    diffusion_schedule
):

    checkpoint = {

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        "epoch":
            epoch,

        "best_val_loss":
            best_val_loss,

        "history":
            history,

        # ----------------------------------------------------
        # Loss configuration
        # ----------------------------------------------------

        "recon_weight":
            RECON_WEIGHT,

        "x0_loss_weight":
            X0_LOSS_WEIGHT,

        "snr_gamma":
            SNR_GAMMA,

        "min_diffusion_weight":
            MIN_DIFFUSION_WEIGHT,

        "loss_type":
            "MinSNRDiffusionMSE + X0PredictionMSE + SELFIESReconstructionCrossEntropy",


        # ----------------------------------------------------
        # Off-manifold training
        # ----------------------------------------------------
        #
        # During training:
        #
        #   x_t → x_t + δ
        #
        # Validation remains clean:
        #
        #   x_t → diffusion
        #
        # ----------------------------------------------------

        "off_manifold_training":
            True,

        "perturbation_prob":
            PERTURBATION_PROB,

        "perturbation_std":
            PERTURBATION_STD,

        "perturbation_type":
            "Gaussian",


        # ----------------------------------------------------
        # Model
        #
        # Includes:
        #   Encoder
        #   SELFIES Decoder
        #   Diffusion Transformer
        # ----------------------------------------------------

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),


        # ----------------------------------------------------
        # Model type
        # ----------------------------------------------------

        "model_type":
            "ConditionalLatentDiffusionWithX0PredictionAndReconstructionEMAOffManifold",


        # ----------------------------------------------------
        # Architecture
        # ----------------------------------------------------

        "embed_dim":
            EMBED_DIM,

        "latent_dim":
            EMBED_DIM,

        "num_heads":
            NUM_HEADS,

        "num_layers":
            NUM_LAYERS,

        "ff_dim":
            FF_DIM,

        "dropout":
            DROPOUT,

        "max_len":
            MAX_LEN,

        "vocab_size":
            VOCAB_SIZE,

        "fingerprint_dim":
            FP_BITS,


        # ----------------------------------------------------
        # Latent normalization
        #
        # Raw x₀ is passed to the decoder.
        #
        # Diffusion uses:
        #
        #   x₀_norm = x₀_raw / latent_rms
        #
        # latent_rms is an EMA buffer stored
        # inside model_state_dict.
        # ----------------------------------------------------

        "latent_normalization":
            "EMA_RMS",

        "initial_latent_rms":
            INITIAL_LATENT_RMS,

        "latent_rms_momentum":
            LATENT_RMS_MOMENTUM,

        "latent_rms":
            model.latent_rms.detach().cpu().item(),


        # ----------------------------------------------------
        # Vocabulary
        # ----------------------------------------------------

        "vocab":
            vocab,

        "token_to_idx":
            token_to_idx,

        "idx_to_token":
            idx_to_token,


        # ----------------------------------------------------
        # Properties
        # ----------------------------------------------------

        "feature_names":
            FEATURE_NAMES,

        "property_dim":
            PROPERTY_DIM,

        "property_scaler":
            scaler,


        # ----------------------------------------------------
        # Fingerprint
        # ----------------------------------------------------

        "fingerprint_bits":
            FP_BITS,

        "fingerprint_radius":
            FP_RADIUS,


        # ----------------------------------------------------
        # Diffusion
        # ----------------------------------------------------

        "num_diffusion_steps":
            diffusion_schedule.num_steps,

        "beta_start":
            diffusion_schedule.beta_start,

        "beta_end":
            diffusion_schedule.beta_end,

        "diffusion_schedule":
            diffusion_schedule.schedule_type,


        # ----------------------------------------------------
        # Diffusion schedule state
        # ----------------------------------------------------

        "diffusion_schedule_state_dict":
            diffusion_schedule.state_dict()
    }


    torch.save(
        checkpoint,
        path
    )


    print(
        f"Checkpoint saved: {path}"
    )

    print(
        f"EMA latent RMS saved: "
        f"{model.latent_rms.detach().item():.6f}"
    )

    print(
        f"EMA momentum: "
        f"{LATENT_RMS_MOMENTUM:.6f}"
    )

    print(
        f"X0 loss weight: "
        f"{X0_LOSS_WEIGHT:.6f}"
    )

    print(
        f"SNR gamma: "
        f"{SNR_GAMMA:.6f}"
    )

    print(
        f"Minimum diffusion weight: "
        f"{MIN_DIFFUSION_WEIGHT:.6f}"
    )

    print(
        f"Off-manifold training: "
        f"enabled"
    )

    print(
        f"Perturbation probability: "
        f"{PERTURBATION_PROB:.4f}"
    )

    print(
        f"Perturbation std: "
        f"{PERTURBATION_STD:.6f}"
    )

Ячейка 36 — история обучения

In [45]:
# ============================================================
# 37. DIFFUSION TRAINING HISTORY
# ============================================================

history = {

    # --------------------------------------------------------
    # Losses
    # --------------------------------------------------------

    "epoch": [],

    "train_loss": [],
    "train_diffusion_loss": [],
    "train_x0_loss": [],
    "train_reconstruction_loss": [],

    "val_loss": [],
    "val_diffusion_loss": [],
    "val_x0_loss": [],
    "val_reconstruction_loss": [],

    # --------------------------------------------------------
    # Gradients
    # --------------------------------------------------------

    "grad_norm": [],

    # --------------------------------------------------------
    # RAW x₀ statistics
    # --------------------------------------------------------

    "x0_mean": [],
    "x0_std": [],
    "x0_min": [],
    "x0_max": [],

    # --------------------------------------------------------
    # NORMALIZED x₀ statistics
    # --------------------------------------------------------

    "x0_norm_mean": [],
    "x0_norm_std": [],
    "x0_norm_min": [],
    "x0_norm_max": [],

    # --------------------------------------------------------
    # EMA latent normalization
    # --------------------------------------------------------

    "latent_rms": [],

    # --------------------------------------------------------
    # Timing
    # --------------------------------------------------------

    "epoch_time": []
}


best_val_loss = float(
    "inf"
)


print(
    "✓ Diffusion Model history initialized"
)

print(
    "✓ Tracking diffusion loss"
)

print(
    "✓ Tracking X0 prediction loss"
)

print(
    "✓ Tracking reconstruction loss"
)

print(
    "✓ Tracking raw x₀ statistics"
)

print(
    "✓ Tracking normalized x₀ statistics"
)

print(
    "✓ Tracking EMA latent RMS"
)

✓ Diffusion Model history initialized
✓ Tracking diffusion loss
✓ Tracking X0 prediction loss
✓ Tracking reconstruction loss
✓ Tracking raw x₀ statistics
✓ Tracking normalized x₀ statistics
✓ Tracking EMA latent RMS


============================================================================================================================
============================================================================================================================

Ячейка 37 — обучение

In [46]:
# ============================================================
# 38. CONDITIONAL LATENT DIFFUSION TRAINING
#     OFF-MANIFOLD TRAINING
# ============================================================

print("=" * 70)
print("START CONDITIONAL LATENT DIFFUSION TRAINING")
print("=" * 70)

print(
    "Device:",
    device
)

print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Parameters:",
    f"{trainable_params:,}"
)

print(
    "Diffusion steps:",
    diffusion_schedule.num_steps
)

print(
    "Diffusion loss:",
    "Min-SNR weighted MSE"
)

print(
    "X0 prediction loss:",
    "MSE"
)

print(
    "X0 loss weight:",
    X0_LOSS_WEIGHT
)

print(
    "SNR gamma:",
    SNR_GAMMA
)

print(
    "Minimum diffusion weight:",
    MIN_DIFFUSION_WEIGHT
)

print(
    "Reconstruction weight:",
    RECON_WEIGHT
)

print(
    "Latent normalization:",
    "EMA_RMS"
)

print(
    "Initial latent RMS:",
    f"{INITIAL_LATENT_RMS:.6f}"
)

print(
    "EMA momentum:",
    f"{LATENT_RMS_MOMENTUM:.6f}"
)

print(
    "Current latent RMS:",
    f"{model.latent_rms.detach().item():.6f}"
)

print()

# ============================================================
# OFF-MANIFOLD TRAINING
# ============================================================

print(
    "Off-manifold training:",
    "ENABLED"
)

print(
    "Perturbation probability:",
    f"{PERTURBATION_PROB:.4f}"
)

print(
    "Perturbation std:",
    f"{PERTURBATION_STD:.6f}"
)

print(
    "Training diffusion input:",
    "xₜ + Gaussian perturbation"
)

print(
    "Validation diffusion input:",
    "clean xₜ"
)

print()

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR
)

print()


# ============================================================
# RESUME SETTINGS
# ============================================================

# IMPORTANT:
#
# Это НОВЫЙ эксперимент:
#
#   Min-SNR
#   + X0 prediction
#   + SELFIES reconstruction
#   + EMA latent normalization
#   + OFF-MANIFOLD training
#
# Поэтому первый запуск должен начинаться
# с новой модели и нового checkpoint directory.
#
# После создания checkpoint:
#
#   RESUME_TRAINING = True
#
# ============================================================

RESUME_TRAINING = False

RESUME_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "diffusion_ema_best.pt"
)


# ============================================================
# INITIALIZATION
# ============================================================

start_epoch = 1

best_val_loss = float(
    "inf"
)


# ============================================================
# HISTORY INITIALIZATION
# ============================================================

if not RESUME_TRAINING:

    history = {

        # Losses
        "epoch": [],

        "train_loss": [],
        "train_diffusion_loss": [],
        "train_x0_loss": [],
        "train_reconstruction_loss": [],

        "val_loss": [],
        "val_diffusion_loss": [],
        "val_x0_loss": [],
        "val_reconstruction_loss": [],

        # Gradients
        "grad_norm": [],

        # Raw x0 statistics
        "x0_mean": [],
        "x0_std": [],
        "x0_min": [],
        "x0_max": [],

        # Normalized x0 statistics
        "x0_norm_mean": [],
        "x0_norm_std": [],
        "x0_norm_min": [],
        "x0_norm_max": [],

        # EMA latent RMS
        "latent_rms": [],

        # Timing
        "epoch_time": []
    }


# ============================================================
# RESUME FROM CHECKPOINT
# ============================================================

if RESUME_TRAINING:

    if not os.path.exists(
        RESUME_CHECKPOINT
    ):

        raise FileNotFoundError(
            f"Checkpoint not found:\n"
            f"{RESUME_CHECKPOINT}"
        )


    print("=" * 70)
    print("RESUMING CONDITIONAL LATENT DIFFUSION TRAINING")
    print("=" * 70)

    print(
        "Checkpoint:",
        RESUME_CHECKPOINT
    )


    checkpoint = torch.load(
        RESUME_CHECKPOINT,
        map_location=device,
        weights_only=False
    )


    # --------------------------------------------------------
    # CHECK MODEL TYPE
    # --------------------------------------------------------

    checkpoint_model_type = checkpoint.get(
        "model_type",
        None
    )

    expected_model_type = (
        "ConditionalLatentDiffusionWithX0PredictionAndReconstructionEMAOffManifold"
    )


    if checkpoint_model_type != expected_model_type:

        raise RuntimeError(
            "Checkpoint is incompatible with "
            "the current off-manifold model.\n"
            f"Detected model_type: "
            f"{checkpoint_model_type}\n"
            f"Expected: "
            f"{expected_model_type}\n"
            "Do not load a baseline or previous "
            "EMA checkpoint."
        )


    print(
        "✓ Model type verified:",
        checkpoint_model_type
    )


    # --------------------------------------------------------
    # CHECK OFF-MANIFOLD TRAINING
    # --------------------------------------------------------

    checkpoint_off_manifold = checkpoint.get(
        "off_manifold_training",
        None
    )


    if checkpoint_off_manifold is not True:

        raise RuntimeError(
            "Checkpoint was not created with "
            "off-manifold training."
        )


    checkpoint_perturbation_prob = checkpoint.get(
        "perturbation_prob",
        None
    )


    checkpoint_perturbation_std = checkpoint.get(
        "perturbation_std",
        None
    )


    if checkpoint_perturbation_prob is None:

        raise RuntimeError(
            "Checkpoint does not contain "
            "perturbation probability."
        )


    if checkpoint_perturbation_std is None:

        raise RuntimeError(
            "Checkpoint does not contain "
            "perturbation std."
        )


    if abs(
        checkpoint_perturbation_prob
        - PERTURBATION_PROB
    ) > 1e-12:

        raise RuntimeError(
            "PERTURBATION_PROB mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_perturbation_prob}\n"
            f"Current:    "
            f"{PERTURBATION_PROB}"
        )


    if abs(
        checkpoint_perturbation_std
        - PERTURBATION_STD
    ) > 1e-12:

        raise RuntimeError(
            "PERTURBATION_STD mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_perturbation_std}\n"
            f"Current:    "
            f"{PERTURBATION_STD}"
        )


    print(
        "✓ Off-manifold training verified"
    )

    print(
        "  Perturbation probability:",
        PERTURBATION_PROB
    )

    print(
        "  Perturbation std:",
        PERTURBATION_STD
    )


    # --------------------------------------------------------
    # CHECK LATENT NORMALIZATION
    # --------------------------------------------------------

    checkpoint_normalization = checkpoint.get(
        "latent_normalization",
        None
    )


    if checkpoint_normalization != "EMA_RMS":

        raise RuntimeError(
            "Checkpoint uses incompatible latent "
            "normalization.\n"
            f"Detected: {checkpoint_normalization}\n"
            "Expected: EMA_RMS"
        )


    print(
        "✓ Latent normalization verified: EMA_RMS"
    )


    # --------------------------------------------------------
    # CHECK EMA MOMENTUM
    # --------------------------------------------------------

    checkpoint_momentum = checkpoint.get(
        "latent_rms_momentum",
        None
    )


    if checkpoint_momentum is not None:

        if abs(
            checkpoint_momentum
            - LATENT_RMS_MOMENTUM
        ) > 1e-12:

            raise RuntimeError(
                "LATENT_RMS_MOMENTUM mismatch.\n"
                f"Checkpoint: "
                f"{checkpoint_momentum}\n"
                f"Current:    "
                f"{LATENT_RMS_MOMENTUM}"
            )


    # --------------------------------------------------------
    # CHECK RECONSTRUCTION WEIGHT
    # --------------------------------------------------------

    checkpoint_recon_weight = checkpoint.get(
        "recon_weight",
        None
    )


    if checkpoint_recon_weight is not None:

        if abs(
            checkpoint_recon_weight
            - RECON_WEIGHT
        ) > 1e-12:

            raise RuntimeError(
                "RECON_WEIGHT mismatch.\n"
                f"Checkpoint: "
                f"{checkpoint_recon_weight}\n"
                f"Current:    "
                f"{RECON_WEIGHT}"
            )


    # --------------------------------------------------------
    # CHECK X0 LOSS WEIGHT
    # --------------------------------------------------------

    checkpoint_x0_weight = checkpoint.get(
        "x0_loss_weight",
        None
    )


    if checkpoint_x0_weight is None:

        raise RuntimeError(
            "Checkpoint does not contain "
            "X0_LOSS_WEIGHT."
        )


    if abs(
        checkpoint_x0_weight
        - X0_LOSS_WEIGHT
    ) > 1e-12:

        raise RuntimeError(
            "X0_LOSS_WEIGHT mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_x0_weight}\n"
            f"Current:    "
            f"{X0_LOSS_WEIGHT}"
        )


    print(
        "✓ X0 loss weight verified:",
        X0_LOSS_WEIGHT
    )


    # --------------------------------------------------------
    # CHECK SNR GAMMA
    # --------------------------------------------------------

    checkpoint_snr_gamma = checkpoint.get(
        "snr_gamma",
        None
    )


    if checkpoint_snr_gamma is None:

        raise RuntimeError(
            "Checkpoint does not contain "
            "SNR_GAMMA."
        )


    if abs(
        checkpoint_snr_gamma
        - SNR_GAMMA
    ) > 1e-12:

        raise RuntimeError(
            "SNR_GAMMA mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_snr_gamma}\n"
            f"Current:    "
            f"{SNR_GAMMA}"
        )


    print(
        "✓ SNR gamma verified:",
        SNR_GAMMA
    )


    # --------------------------------------------------------
    # CHECK MINIMUM DIFFUSION WEIGHT
    # --------------------------------------------------------

    checkpoint_min_diffusion_weight = checkpoint.get(
        "min_diffusion_weight",
        None
    )


    if checkpoint_min_diffusion_weight is None:

        raise RuntimeError(
            "Checkpoint does not contain "
            "MIN_DIFFUSION_WEIGHT."
        )


    if abs(
        checkpoint_min_diffusion_weight
        - MIN_DIFFUSION_WEIGHT
    ) > 1e-12:

        raise RuntimeError(
            "MIN_DIFFUSION_WEIGHT mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_min_diffusion_weight}\n"
            f"Current:    "
            f"{MIN_DIFFUSION_WEIGHT}"
        )


    print(
        "✓ Minimum diffusion weight verified:",
        MIN_DIFFUSION_WEIGHT
    )


    # --------------------------------------------------------
    # CHECK DIFFUSION STEPS
    # --------------------------------------------------------

    checkpoint_steps = checkpoint.get(
        "num_diffusion_steps",
        None
    )


    if checkpoint_steps != (
        diffusion_schedule.num_steps
    ):

        raise RuntimeError(
            "Diffusion schedule mismatch.\n"
            f"Checkpoint steps: "
            f"{checkpoint_steps}\n"
            f"Current steps: "
            f"{diffusion_schedule.num_steps}"
        )


    # --------------------------------------------------------
    # CHECK SCHEDULE TYPE
    # --------------------------------------------------------

    checkpoint_schedule = checkpoint.get(
        "diffusion_schedule",
        None
    )


    if checkpoint_schedule != (
        diffusion_schedule.schedule_type
    ):

        raise RuntimeError(
            "Diffusion schedule type mismatch.\n"
            f"Checkpoint schedule: "
            f"{checkpoint_schedule}\n"
            f"Current schedule: "
            f"{diffusion_schedule.schedule_type}"
        )


    # --------------------------------------------------------
    # CHECK BETA START
    # --------------------------------------------------------

    checkpoint_beta_start = checkpoint.get(
        "beta_start",
        None
    )


    if (
        checkpoint_beta_start is not None
        and abs(
            checkpoint_beta_start
            - diffusion_schedule.beta_start
        ) > 1e-12
    ):

        raise RuntimeError(
            "BETA_START mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_beta_start}\n"
            f"Current:    "
            f"{diffusion_schedule.beta_start}"
        )


    # --------------------------------------------------------
    # CHECK BETA END
    # --------------------------------------------------------

    checkpoint_beta_end = checkpoint.get(
        "beta_end",
        None
    )


    if (
        checkpoint_beta_end is not None
        and abs(
            checkpoint_beta_end
            - diffusion_schedule.beta_end
        ) > 1e-12
    ):

        raise RuntimeError(
            "BETA_END mismatch.\n"
            f"Checkpoint: "
            f"{checkpoint_beta_end}\n"
            f"Current:    "
            f"{diffusion_schedule.beta_end}"
        )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(
        "✓ Model state restored"
    )


    # --------------------------------------------------------
    # CHECK RESTORED LATENT RMS
    # --------------------------------------------------------

    restored_latent_rms = (
        model.latent_rms.detach().item()
    )

    checkpoint_latent_rms = checkpoint.get(
        "latent_rms",
        None
    )


    if checkpoint_latent_rms is not None:

        rms_error = abs(
            restored_latent_rms
            - checkpoint_latent_rms
        )

        if rms_error > 1e-6:

            raise RuntimeError(
                "Restored latent RMS does not match "
                "checkpoint metadata.\n"
                f"Model:      {restored_latent_rms:.8f}\n"
                f"Checkpoint: {checkpoint_latent_rms:.8f}"
            )


    print(
        "✓ EMA latent RMS restored:",
        f"{restored_latent_rms:.6f}"
    )


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    print(
        "✓ Optimizer state restored"
    )


    # --------------------------------------------------------
    # DIFFUSION SCHEDULE STATE
    # --------------------------------------------------------

    if "diffusion_schedule_state_dict" in checkpoint:

        diffusion_schedule.load_state_dict(
            checkpoint[
                "diffusion_schedule_state_dict"
            ]
        )

        print(
            "✓ Diffusion schedule state restored"
        )

    else:

        print(
            "WARNING: checkpoint does not contain "
            "diffusion_schedule_state_dict"
        )


    # --------------------------------------------------------
    # EPOCH
    # --------------------------------------------------------

    saved_epoch = checkpoint["epoch"]

    start_epoch = (
        saved_epoch + 1
    )


    # --------------------------------------------------------
    # BEST LOSS
    # --------------------------------------------------------

    best_val_loss = (
        checkpoint["best_val_loss"]
    )


    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    history = checkpoint["history"]


    # --------------------------------------------------------
    # CHECK HISTORY FORMAT
    # --------------------------------------------------------

    required_history_keys = [

        "epoch",

        "train_loss",
        "train_diffusion_loss",
        "train_x0_loss",
        "train_reconstruction_loss",

        "val_loss",
        "val_diffusion_loss",
        "val_x0_loss",
        "val_reconstruction_loss",

        "grad_norm",

        # Raw x0
        "x0_mean",
        "x0_std",
        "x0_min",
        "x0_max",

        # Normalized x0
        "x0_norm_mean",
        "x0_norm_std",
        "x0_norm_min",
        "x0_norm_max",

        # EMA RMS
        "latent_rms",

        "epoch_time"
    ]


    missing_history_keys = [

        key
        for key in required_history_keys
        if key not in history

    ]


    if missing_history_keys:

        raise RuntimeError(
            "Checkpoint history is incompatible "
            "with the current EMA training format.\n"
            f"Missing keys: {missing_history_keys}\n"
            "Start a new training run with "
            "RESUME_TRAINING = False."
        )


    print()

    print(
        f"Saved epoch:    {saved_epoch}"
    )

    print(
        f"Starting epoch: {start_epoch}"
    )

    print(
        f"Best Val Loss:  "
        f"{best_val_loss:.5f}"
    )

    print(
        f"History epochs: "
        f"{len(history['epoch'])}"
    )

    print()

    print(
        f"Current EMA RMS: "
        f"{model.latent_rms.detach().item():.6f}"
    )

    print("=" * 70)

    print()

else:

    print(
        "Starting Conditional Latent Diffusion "
        "training from epoch 1"
    )

    print(
        f"EMA latent RMS = "
        f"{model.latent_rms.detach().item():.6f}"
    )

    print(
        f"EMA momentum = "
        f"{LATENT_RMS_MOMENTUM:.6f}"
    )

    print(
        f"X0 loss weight = "
        f"{X0_LOSS_WEIGHT:.6f}"
    )

    print(
        f"SNR gamma = "
        f"{SNR_GAMMA:.6f}"
    )

    print(
        f"Minimum diffusion weight = "
        f"{MIN_DIFFUSION_WEIGHT:.6f}"
    )

    print(
        f"Reconstruction weight = "
        f"{RECON_WEIGHT:.6f}"
    )

    print()

    print(
        "OFF-MANIFOLD TRAINING = ENABLED"
    )

    print(
        f"Perturbation probability = "
        f"{PERTURBATION_PROB:.6f}"
    )

    print(
        f"Perturbation std = "
        f"{PERTURBATION_STD:.6f}"
    )

    print()

    print(
        "RESUME_TRAINING = False"
    )

    print(
        "Old checkpoints will NOT be loaded."
    )

    print()


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(
    start_epoch,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()


    # ========================================================
    # TRAIN
    # ========================================================

    (
        train_loss,
        train_diffusion_loss,
        train_x0_loss,
        train_reconstruction_loss,
        grad_norm
    ) = train_epoch(

        model=model,

        dataloader=train_loader,

        optimizer=optimizer,

        device=device,

        diffusion_schedule=diffusion_schedule
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    (
        val_loss,
        val_diffusion_loss,
        val_x0_loss,
        val_reconstruction_loss
    ) = validate_epoch(

        model=model,

        dataloader=val_loader,

        device=device,

        diffusion_schedule=diffusion_schedule
    )


    # ========================================================
    # CLEAN LATENT STATISTICS
    # ========================================================

    x0_stats = (
        get_latent_statistics(

            model=model,

            dataloader=val_loader,

            device=device,

            max_batches=LATENT_STATS_BATCHES
        )
    )


    # ========================================================
    # TIME
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    # ========================================================
    # HISTORY
    # ========================================================

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        train_loss
    )

    history["train_diffusion_loss"].append(
        train_diffusion_loss
    )

    history["train_x0_loss"].append(
        train_x0_loss
    )

    history["train_reconstruction_loss"].append(
        train_reconstruction_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_diffusion_loss"].append(
        val_diffusion_loss
    )

    history["val_x0_loss"].append(
        val_x0_loss
    )

    history["val_reconstruction_loss"].append(
        val_reconstruction_loss
    )

    history["grad_norm"].append(
        grad_norm
    )


    # --------------------------------------------------------
    # RAW x0 STATISTICS
    # --------------------------------------------------------

    history["x0_mean"].append(
        x0_stats["x0_mean"]
    )

    history["x0_std"].append(
        x0_stats["x0_std"]
    )

    history["x0_min"].append(
        x0_stats["x0_min"]
    )

    history["x0_max"].append(
        x0_stats["x0_max"]
    )


    # --------------------------------------------------------
    # NORMALIZED x0 STATISTICS
    # --------------------------------------------------------

    history["x0_norm_mean"].append(
        x0_stats["x0_norm_mean"]
    )

    history["x0_norm_std"].append(
        x0_stats["x0_norm_std"]
    )

    history["x0_norm_min"].append(
        x0_stats["x0_norm_min"]
    )

    history["x0_norm_max"].append(
        x0_stats["x0_norm_max"]
    )


    # --------------------------------------------------------
    # EMA LATENT RMS
    # --------------------------------------------------------

    history["latent_rms"].append(
        x0_stats["latent_rms"]
    )

    history["epoch_time"].append(
        epoch_time
    )


    # ========================================================
    # CONSOLE
    # ========================================================

    print()

    print("=" * 70)

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS}"
    )

    print()

    print(
        f"Train Loss:       {train_loss:.5f}"
    )

    print(
        f"  Diffusion:       "
        f"{train_diffusion_loss:.5f}"
    )

    print(
        f"  X0 prediction:   "
        f"{train_x0_loss:.5f}"
    )

    print(
        f"  Reconstruction:  "
        f"{train_reconstruction_loss:.5f}"
    )

    print()

    print(
        f"Val Loss:         {val_loss:.5f}"
    )

    print(
        f"  Diffusion:       "
        f"{val_diffusion_loss:.5f}"
    )

    print(
        f"  X0 prediction:   "
        f"{val_x0_loss:.5f}"
    )

    print(
        f"  Reconstruction:  "
        f"{val_reconstruction_loss:.5f}"
    )

    print()

    print(
        f"Grad Norm:        {grad_norm:.5f}"
    )

    print()

    print(
        f"x₀ raw mean:      "
        f"{x0_stats['x0_mean']:.5f}"
    )

    print(
        f"x₀ raw std:       "
        f"{x0_stats['x0_std']:.5f}"
    )

    print(
        f"x₀ raw range:     "
        f"[{x0_stats['x0_min']:.5f}, "
        f"{x0_stats['x0_max']:.5f}]"
    )

    print()

    print(
        f"x₀ norm mean:     "
        f"{x0_stats['x0_norm_mean']:.5f}"
    )

    print(
        f"x₀ norm std:      "
        f"{x0_stats['x0_norm_std']:.5f}"
    )

    print(
        f"x₀ norm range:    "
        f"[{x0_stats['x0_norm_min']:.5f}, "
        f"{x0_stats['x0_norm_max']:.5f}]"
    )

    print()

    print(
        f"EMA latent RMS:    "
        f"{x0_stats['latent_rms']:.6f}"
    )

    print(
        f"Time:              "
        f"{epoch_time / 60:.2f} min"
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_path = os.path.join(
            CHECKPOINT_DIR,
            "diffusion_ema_best.pt"
        )

        save_checkpoint(

            path=best_path,

            model=model,

            optimizer=optimizer,

            epoch=epoch,

            best_val_loss=best_val_loss,

            history=history,

            scaler=property_scaler,

            diffusion_schedule=diffusion_schedule
        )

        print(
            f"✓ NEW BEST: "
            f"{best_val_loss:.5f}"
        )


    # ========================================================
    # PERIODIC CHECKPOINT
    # ========================================================

    if (
        epoch % CHECKPOINT_EVERY
        == 0
    ):

        checkpoint_path = os.path.join(

            CHECKPOINT_DIR,

            f"diffusion_ema_epoch_"
            f"{epoch:03d}.pt"
        )

        save_checkpoint(

            path=checkpoint_path,

            model=model,

            optimizer=optimizer,

            epoch=epoch,

            best_val_loss=best_val_loss,

            history=history,

            scaler=property_scaler,

            diffusion_schedule=diffusion_schedule
        )

        print(
            f"✓ CHECKPOINT SAVED: "
            f"{checkpoint_path}"
        )

    print("=" * 70)

START CONDITIONAL LATENT DIFFUSION TRAINING
Device: cuda
Train samples: 189238
Validation samples: 23654
Test samples: 23656
Train batches: 5914
Validation batches: 740
Parameters: 4,333,165
Diffusion steps: 1000
Diffusion loss: Min-SNR weighted MSE
X0 prediction loss: MSE
X0 loss weight: 0.01
SNR gamma: 5.0
Minimum diffusion weight: 0.1
Reconstruction weight: 1.0
Latent normalization: EMA_RMS
Initial latent RMS: 0.180000
EMA momentum: 0.990000
Current latent RMS: 0.180000

Off-manifold training: ENABLED
Perturbation probability: 0.5000
Perturbation std: 0.010000
Training diffusion input: xₜ + Gaussian perturbation
Validation diffusion input: clean xₜ

Checkpoint directory: checkpoints_diffusion_x0_minsnr

Starting Conditional Latent Diffusion training from epoch 1
EMA latent RMS = 0.180000
EMA momentum = 0.990000
X0 loss weight = 0.010000
SNR gamma = 5.000000
Minimum diffusion weight = 0.100000
Reconstruction weight = 1.000000

OFF-MANIFOLD TRAINING = ENABLED
Perturbation probability 

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.001992
  std         : 0.055341
  min         : -0.257291
  max         : 0.277788

NORMALIZED x₀
  mean        : -0.034552
  std         : 0.959795
  min         : -4.462301
  max         : 4.817803

EMA latent RMS : 0.057659

Expected norm std : 0.959795
Normalization error: 0.00000000

Epoch 001/60

Train Loss:       9.94270
  Diffusion:       0.59338
  X0 prediction:   836.15036
  Reconstruction:  0.98781

Val Loss:         6.33408
  Diffusion:       0.37324
  X0 prediction:   539.33766
  Reconstruction:  0.56747

Grad Norm:        17.78512

x₀ raw mean:      -0.00199
x₀ raw std:       0.05534
x₀ raw range:     [-0.25729, 0.27779]

x₀ norm mean:     -0.03455
x₀ norm std:      0.95980
x₀ norm range:    [-4.46230, 4.81780]

EMA latent RMS:    0.057659
Time:              10.02 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.057659
EMA moment

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.001272
  std         : 0.050362
  min         : -0.279779
  max         : 0.265685

NORMALIZED x₀
  mean        : -0.024594
  std         : 0.973586
  min         : -5.408602
  max         : 5.136128

EMA latent RMS : 0.051729

Expected norm std : 0.973586
Normalization error: 0.00000000

Epoch 002/60

Train Loss:       6.85969
  Diffusion:       0.41920
  X0 prediction:   585.12616
  Reconstruction:  0.58923

Val Loss:         4.33618
  Diffusion:       0.26994
  X0 prediction:   362.08233
  Reconstruction:  0.44541

Grad Norm:        10.32689

x₀ raw mean:      -0.00127
x₀ raw std:       0.05036
x₀ raw range:     [-0.27978, 0.26568]

x₀ norm mean:     -0.02459
x₀ norm std:      0.97359
x₀ norm range:    [-5.40860, 5.13613]

EMA latent RMS:    0.051729
Time:              9.44 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.051729
EMA momentu

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000735
  std         : 0.046436
  min         : -0.273071
  max         : 0.239722

NORMALIZED x₀
  mean        : -0.015483
  std         : 0.978385
  min         : -5.753527
  max         : 5.050878

EMA latent RMS : 0.047462

Expected norm std : 0.978385
Normalization error: 0.00000000

Epoch 003/60

Train Loss:       5.89415
  Diffusion:       0.36461
  X0 prediction:   505.83095
  Reconstruction:  0.47123

Val Loss:         3.81755
  Diffusion:       0.23281
  X0 prediction:   321.03676
  Reconstruction:  0.37437

Grad Norm:        17.90176

x₀ raw mean:      -0.00073
x₀ raw std:       0.04644
x₀ raw range:     [-0.27307, 0.23972]

x₀ norm mean:     -0.01548
x₀ norm std:      0.97838
x₀ norm range:    [-5.75353, 5.05088]

EMA latent RMS:    0.047462
Time:              9.30 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.047462
EMA momentu

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000112
  std         : 0.032394
  min         : -0.186139
  max         : 0.177504

NORMALIZED x₀
  mean        : -0.003395
  std         : 0.978191
  min         : -5.620733
  max         : 5.360006

EMA latent RMS : 0.033116

Expected norm std : 0.978191
Normalization error: 0.00000000

Epoch 004/60

Train Loss:       5.40837
  Diffusion:       0.33932
  X0 prediction:   466.34998
  Reconstruction:  0.40555

Val Loss:         3.18768
  Diffusion:       0.21257
  X0 prediction:   269.48776
  Reconstruction:  0.28024

Grad Norm:        4.30417

x₀ raw mean:      -0.00011
x₀ raw std:       0.03239
x₀ raw range:     [-0.18614, 0.17750]

x₀ norm mean:     -0.00339
x₀ norm std:      0.97819
x₀ norm range:    [-5.62073, 5.36001]

EMA latent RMS:    0.033116
Time:              9.55 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.033116
EMA momentum

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000269
  std         : 0.024185
  min         : -0.119052
  max         : 0.122486

NORMALIZED x₀
  mean        : -0.010957
  std         : 0.984201
  min         : -4.844789
  max         : 4.984519

EMA latent RMS : 0.024573

Expected norm std : 0.984201
Normalization error: 0.00000000

Epoch 005/60

Train Loss:       4.95507
  Diffusion:       0.31424
  X0 prediction:   427.93150
  Reconstruction:  0.36152

Val Loss:         3.08009
  Diffusion:       0.19646
  X0 prediction:   264.15198
  Reconstruction:  0.24211

Grad Norm:        10.44119

x₀ raw mean:      -0.00027
x₀ raw std:       0.02419
x₀ raw range:     [-0.11905, 0.12249]

x₀ norm mean:     -0.01096
x₀ norm std:      0.98420
x₀ norm range:    [-4.84479, 4.98452]

EMA latent RMS:    0.024573
Time:              9.94 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.024573
EMA momentu

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000367
  std         : 0.021639
  min         : -0.099088
  max         : 0.122558

NORMALIZED x₀
  mean        : -0.016877
  std         : 0.993846
  min         : -4.550865
  max         : 5.628777

EMA latent RMS : 0.021773

Expected norm std : 0.993846
Normalization error: 0.00000000

Epoch 006/60

Train Loss:       4.14514
  Diffusion:       0.26229
  X0 prediction:   354.70065
  Reconstruction:  0.33584

Val Loss:         2.48159
  Diffusion:       0.15942
  X0 prediction:   206.08872
  Reconstruction:  0.26129

Grad Norm:        34.15007

x₀ raw mean:      -0.00037
x₀ raw std:       0.02164
x₀ raw range:     [-0.09909, 0.12256]

x₀ norm mean:     -0.01688
x₀ norm std:      0.99385
x₀ norm range:    [-4.55086, 5.62878]

EMA latent RMS:    0.021773
Time:              9.69 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.021773
EMA momentu

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000165
  std         : 0.020639
  min         : -0.117433
  max         : 0.132229

NORMALIZED x₀
  mean        : -0.007874
  std         : 0.987495
  min         : -5.618648
  max         : 6.326575

EMA latent RMS : 0.020900

Expected norm std : 0.987495
Normalization error: 0.00000000

Epoch 007/60

Train Loss:       3.64401
  Diffusion:       0.23213
  X0 prediction:   308.62091
  Reconstruction:  0.32566

Val Loss:         2.31685
  Diffusion:       0.15224
  X0 prediction:   192.75531
  Reconstruction:  0.23706

Grad Norm:        25.76057

x₀ raw mean:      -0.00016
x₀ raw std:       0.02064
x₀ raw range:     [-0.11743, 0.13223]

x₀ norm mean:     -0.00787
x₀ norm std:      0.98750
x₀ norm range:    [-5.61865, 6.32657]

EMA latent RMS:    0.020900
Time:              9.58 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.020900
EMA momentu

CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000134
  std         : 0.019610
  min         : -0.111806
  max         : 0.117760

NORMALIZED x₀
  mean        : -0.006756
  std         : 0.987708
  min         : -5.631274
  max         : 5.931182

EMA latent RMS : 0.019854

Expected norm std : 0.987708
Normalization error: 0.00000000

Epoch 008/60

Train Loss:       3.61084
  Diffusion:       0.22811
  X0 prediction:   307.06165
  Reconstruction:  0.31212

Val Loss:         2.36104
  Diffusion:       0.15377
  X0 prediction:   201.07612
  Reconstruction:  0.19650

Grad Norm:        18.88460

x₀ raw mean:      -0.00013
x₀ raw std:       0.01961
x₀ raw range:     [-0.11181, 0.11776]

x₀ norm mean:     -0.00676
x₀ norm std:      0.98771
x₀ norm range:    [-5.63127, 5.93118]

EMA latent RMS:    0.019854
Time:              9.23 min


CLEAN LATENT x₀ STATISTICS
Samples       : 640
Latent dim    : 128

RAW x₀
  mean        : -0.000078
  std         : 0.019356
  min         : -0.105867
  max         : 0.101746

NORMALIZED x₀
  mean        : -0.003924
  std         : 0.977703
  min         : -5.347517
  max         : 5.139383

EMA latent RMS : 0.019797

Expected norm std : 0.977703
Normalization error: 0.00000000

Epoch 009/60

Train Loss:       3.53802
  Diffusion:       0.22534
  X0 prediction:   301.22647
  Reconstruction:  0.30041

Val Loss:         2.23419
  Diffusion:       0.14936
  X0 prediction:   186.75337
  Reconstruction:  0.21729

Grad Norm:        16.53226

x₀ raw mean:      -0.00008
x₀ raw std:       0.01936
x₀ raw range:     [-0.10587, 0.10175]

x₀ norm mean:     -0.00392
x₀ norm std:      0.97770
x₀ norm range:    [-5.34752, 5.13938]

EMA latent RMS:    0.019797
Time:              10.04 min
Checkpoint saved: checkpoints_diffusion_x0_minsnr\diffusion_ema_best.pt
EMA latent RMS saved: 0.019797
EMA moment

KeyboardInterrupt: 

=======================================================================================================================
=======================================================================================================================

Ячейка 38 — сохранить историю

In [ ]:
# ============================================================
# 39. SAVE CONDITIONAL DIFFUSION TRAINING HISTORY
#     OFF-MANIFOLD EXPERIMENT A
# ============================================================

history_df = pd.DataFrame(
    history
)


# ============================================================
# PATH
# ============================================================

history_path = os.path.join(
    CHECKPOINT_DIR,
    "diffusion_offmanifold_training_history.csv"
)


history_df.to_csv(
    history_path,
    index=False
)


print(
    "История обучения Conditional Latent Diffusion "
    "Model сохранена:"
)

print(
    history_path
)

print()


# ============================================================
# EXPERIMENT TYPE
# ============================================================

print(
    "=" * 70
)

print(
    "EXPERIMENT:"
)

print(
    "  Off-Manifold Training"
)

print(
    "  Experiment A"
)

print(
    "=" * 70
)

print()


# ============================================================
# EXPERIMENT PARAMETERS
# ============================================================

print(
    "Параметры эксперимента:"
)

print(
    "  Experiment type:",
    "Off-Manifold Training"
)

print(
    "  Perturbation type:",
    "Gaussian"
)

print(
    "  Perturbation probability:",
    f"{PERTURBATION_PROB:.4f}"
)

print(
    "  Perturbation std:",
    f"{PERTURBATION_STD:.6f}"
)

print(
    "  Training input:",
    "xₜ + δ"
)

print(
    "  Validation input:",
    "clean xₜ"
)

print()

print(
    "  Latent normalization:",
    "EMA_RMS"
)

print(
    "  Initial latent RMS:",
    f"{INITIAL_LATENT_RMS:.6f}"
)

print(
    "  EMA momentum:",
    f"{LATENT_RMS_MOMENTUM:.6f}"
)

print(
    "  Current latent RMS:",
    f"{model.latent_rms.detach().item():.6f}"
)

print(
    "  Diffusion loss:",
    "Min-SNR weighted MSE"
)

print(
    "  X0 prediction loss:",
    "MSE"
)

print(
    "  X0 loss weight:",
    f"{X0_LOSS_WEIGHT:.4f}"
)

print(
    "  SNR gamma:",
    f"{SNR_GAMMA:.4f}"
)

print(
    "  Minimum diffusion weight:",
    f"{MIN_DIFFUSION_WEIGHT:.4f}"
)

print(
    "  Recon weight:",
    f"{RECON_WEIGHT:.4f}"
)

print(
    "  Diffusion steps:",
    NUM_DIFFUSION_STEPS
)

print()


# ============================================================
# METRICS
# ============================================================

print(
    "Метрики истории:"
)

print(
    "  Epochs:",
    len(history_df)
)

print()


# ============================================================
# TRAIN
# ============================================================

print(
    "  Train loss:"
)

print(
    "    Total          :",
    f"{history_df['train_loss'].iloc[-1]:.5f}"
)

print(
    "    Diffusion      :",
    f"{history_df['train_diffusion_loss'].iloc[-1]:.5f}"
)

print(
    "    X0 prediction  :",
    f"{history_df['train_x0_loss'].iloc[-1]:.5f}"
)

print(
    "    Reconstruction :",
    f"{history_df['train_reconstruction_loss'].iloc[-1]:.5f"
)

print()


# ============================================================
# VALIDATION
# ============================================================

print(
    "  Validation loss:"
)

print(
    "    Total          :",
    f"{history_df['val_loss'].iloc[-1]:.5f}"
)

print(
    "    Diffusion      :",
    f"{history_df['val_diffusion_loss'].iloc[-1]:.5f}"
)

print(
    "    X0 prediction  :",
    f"{history_df['val_x0_loss'].iloc[-1]:.5f}"
)

print(
    "    Reconstruction :",
    f"{history_df['val_reconstruction_loss'].iloc[-1]:.5f"
)

print()


# ============================================================
# WEIGHTED LOSS COMPONENTS
# ============================================================

print(
    "  Weighted validation components:"
)

final_val_diffusion = (
    history_df[
        "val_diffusion_loss"
    ].iloc[-1]
)

final_val_x0 = (
    history_df[
        "val_x0_loss"
    ].iloc[-1]
)

final_val_reconstruction = (
    history_df[
        "val_reconstruction_loss"
    ].iloc[-1]
)

print(
    "    Diffusion       :",
    f"{final_val_diffusion:.5f}"
)

print(
    "    X0 weighted     :",
    f"{X0_LOSS_WEIGHT * final_val_x0:.5f}"
)

print(
    "    Reconstruction  :",
    f"{RECON_WEIGHT * final_val_reconstruction:.5f}"
)

print()


# ============================================================
# RAW LATENT STATISTICS
# ============================================================

if "x0_std" in history_df.columns:

    print(
        "  Final raw x₀ statistics:"
    )

    print(
        "    Mean           :",
        f"{history_df['x0_mean'].iloc[-1]:.5f}"
    )

    print(
        "    Std            :",
        f"{history_df['x0_std'].iloc[-1]:.5f}"
    )

    print(
        "    Min            :",
        f"{history_df['x0_min'].iloc[-1]:.5f}"
    )

    print(
        "    Max            :",
        f"{history_df['x0_max'].iloc[-1]:.5f}"
    )

    print()


# ============================================================
# NORMALIZED LATENT STATISTICS
# ============================================================

if "x0_norm_std" in history_df.columns:

    print(
        "  Final normalized x₀ statistics:"
    )

    print(
        "    Mean           :",
        f"{history_df['x0_norm_mean'].iloc[-1]:.5f}"
    )

    print(
        "    Std            :",
        f"{history_df['x0_norm_std'].iloc[-1]:.5f}"
    )

    print(
        "    Min            :",
        f"{history_df['x0_norm_min'].iloc[-1]:.5f}"
    )

    print(
        "    Max            :",
        f"{history_df['x0_norm_max'].iloc[-1]:.5f}"
    )

    print()


# ============================================================
# EMA LATENT RMS
# ============================================================

if "latent_rms" in history_df.columns:

    print(
        "  Final EMA latent RMS:"
    )

    print(
        "    RMS            :",
        f"{history_df['latent_rms'].iloc[-1]:.6f}"
    )

    print()


# ============================================================
# TABLE
# ============================================================

display(
    history_df.tail()
)

Ячейка 39 — график обучения

In [ ]:
# ============================================================
# 40 PREDICTION LOSS
# ============================================================

import matplotlib.pyplot as plt
if "train_x0_loss" in history_df.columns:

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["train_x0_loss"],
        label="Train X0 Prediction Loss"
    )

    plt.plot(
        history_df["epoch"],
        history_df["val_x0_loss"],
        label="Validation X0 Prediction Loss"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "MSE"
    )

    plt.title(
        "Clean Latent x₀ Prediction Loss"
    )

    plt.legend()

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()

Ячейка 40 — Recon и KL

In [ ]:
# ============================================================
# 41. GRADIENT NORM
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["grad_norm"],
    label="Gradient Norm"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Gradient Norm"
)

plt.title(
    "Conditional Latent Diffusion — Gradient Norm"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()

Ячейка 41 — статистика latent space

In [ ]:
# ============================================================
# 42. CLEAN LATENT x₀ STATISTICS
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["x0_mean"],
    label="x₀ Mean"
)

plt.plot(
    history_df["epoch"],
    history_df["x0_min"],
    label="x₀ Min"
)

plt.plot(
    history_df["epoch"],
    history_df["x0_max"],
    label="x₀ Max"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Latent value"
)

plt.title(
    "Clean Latent x₀ Statistics"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 43. EPOCH TRAINING TIME
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["epoch_time"] / 60.0,
    label="Epoch Time"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Time (minutes)"
)

plt.title(
    "Conditional Latent Diffusion — Training Time per Epoch"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 44. TRAIN / VALIDATION LOSS GAP
# ============================================================

history_df["loss_gap"] = (
    history_df["val_loss"]
    - history_df["train_loss"]
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["loss_gap"],
    label="Validation - Train Loss"
)

plt.axhline(
    0.0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss Difference"
)

plt.title(
    "Train / Validation Total Loss Gap"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 45. CLEAN LATENT x₀ DIMENSIONS DISTRIBUTION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


@torch.no_grad()
def collect_x0(
    model,
    dataloader,
    device,
    max_batches=100
):

    model.eval()

    all_x0 = []

    for batch_idx, (
        batch,
        properties,
        fingerprint
    ) in enumerate(dataloader):

        if batch_idx >= max_batches:
            break

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        x0 = model.encoder(
            batch,
            properties,
            fingerprint
        )

        all_x0.append(
            x0.cpu()
        )

    if not all_x0:
        raise RuntimeError(
            "Не удалось получить clean latent x₀"
        )

    return torch.cat(
        all_x0,
        dim=0
    )


x0 = collect_x0(
    model,
    val_loader,
    device,
    max_batches=100
)


x0_np = (
    x0.numpy()
)


print(
    "Clean latent x₀ shape:",
    x0_np.shape
)


# ------------------------------------------------------------
# Показываем первые 12 latent dimensions
# ------------------------------------------------------------

n_dims = min(
    12,
    x0_np.shape[1]
)


plt.figure(
    figsize=(12, 7)
)


plt.boxplot(
    [
        x0_np[:, i]
        for i in range(n_dims)
    ],
    labels=[
        f"x₀_{i}"
        for i in range(n_dims)
    ]
)


plt.xlabel(
    "Clean latent dimension"
)

plt.ylabel(
    "Value"
)

plt.title(
    "Distribution of Clean Latent x₀ Dimensions"
)

plt.grid(
    True,
    axis="y",
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 45B. REVERSE DIFFUSION STABILITY DIAGNOSTIC
# ============================================================

print()
print("=" * 70)
print("REVERSE DIFFUSION STABILITY DIAGNOSTIC")
print("=" * 70)


@torch.no_grad()
def reverse_diffusion_diagnostic(
    model,
    target_properties,
    diffusion_schedule,
    latent_dim,
    device,
    print_every=50
):

    model.eval()

    batch_size = target_properties.size(0)

    # --------------------------------------------------------
    # EMA latent RMS
    # --------------------------------------------------------

    latent_rms = (
        model.latent_rms
        .detach()
        .clamp_min(1e-6)
    )

    print()
    print("LATENT NORMALIZATION")
    print(
        "method       : EMA_RMS"
    )
    print(
        "latent RMS   :",
        f"{latent_rms.item():.6f}"
    )

    # --------------------------------------------------------
    # Начинаем с чистого Gaussian noise
    #
    # x_t находится в normalized latent space
    # --------------------------------------------------------

    x_t = torch.randn(
        batch_size,
        latent_dim,
        device=device
    )

    print()
    print("INITIAL xT")
    print(
        "mean :",
        x_t.mean().item()
    )
    print(
        "std  :",
        x_t.std().item()
    )
    print(
        "min  :",
        x_t.min().item()
    )
    print(
        "max  :",
        x_t.max().item()
    )

    print()
    print("-" * 70)

    # --------------------------------------------------------
    # Reverse diffusion
    # --------------------------------------------------------

    for t in range(
        diffusion_schedule.num_steps - 1,
        -1,
        -1
    ):

        timestep = torch.full(
            (batch_size,),
            t,
            dtype=torch.long,
            device=device
        )

        # ----------------------------------------------------
        # Predict noise
        #
        # x_t находится в normalized latent space
        # ----------------------------------------------------

        predicted_noise = model.diffusion(
            x_t,
            timestep,
            target_properties
        )

        # ----------------------------------------------------
        # Schedule values
        # ----------------------------------------------------

        beta_t = (
            diffusion_schedule.betas[t]
        )

        alpha_t = (
            diffusion_schedule.alphas[t]
        )

        alpha_bar_t = (
            diffusion_schedule.alpha_bars[t]
        )

        # ----------------------------------------------------
        # DDPM posterior mean
        # ----------------------------------------------------

        mean = (
            diffusion_schedule.sqrt_recip_alphas[t]
            * (
                x_t
                -
                (
                    beta_t
                    /
                    torch.sqrt(
                        1.0 - alpha_bar_t
                    )
                )
                * predicted_noise
            )
        )

        # ----------------------------------------------------
        # Posterior variance
        # ----------------------------------------------------

        posterior_variance = (
            diffusion_schedule.posterior_variance[t]
        )

        # ----------------------------------------------------
        # Noise
        # ----------------------------------------------------

        noise = torch.randn_like(
            x_t
        )

        # ----------------------------------------------------
        # t = 0 → no additional noise
        # ----------------------------------------------------

        if t > 0:

            x_t = (
                mean
                +
                torch.sqrt(
                    posterior_variance
                )
                * noise
            )

        else:

            x_t = mean

        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------

        if (
            t % print_every == 0
            or t == diffusion_schedule.num_steps - 1
            or t == 0
        ):

            print(
                f"t={t:4d} | "
                f"x_t std={x_t.std().item():12.6f} | "
                f"x_t mean={x_t.mean().item():10.6f} | "
                f"pred_noise std={predicted_noise.std().item():10.6f}"
            )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if not torch.isfinite(
            x_t
        ).all():

            print()
            print(
                f"!!! NaN / Inf detected at t={t}"
            )

            break

    print()
    print("-" * 70)

    # ========================================================
    # FINAL NORMALIZED x₀
    # ========================================================

    x0_norm = x_t

    print()
    print("FINAL NORMALIZED x₀")

    print(
        "mean :",
        x0_norm.mean().item()
    )

    print(
        "std  :",
        x0_norm.std().item()
    )

    print(
        "min  :",
        x0_norm.min().item()
    )

    print(
        "max  :",
        x0_norm.max().item()
    )

    # ========================================================
    # CONVERT TO RAW x₀
    #
    # x₀_raw = x₀_norm × EMA latent RMS
    # ========================================================

    x0_raw = (
        x0_norm
        * latent_rms
    )

    print()
    print("FINAL RAW x₀")

    print(
        "mean :",
        x0_raw.mean().item()
    )

    print(
        "std  :",
        x0_raw.std().item()
    )

    print(
        "min  :",
        x0_raw.min().item()
    )

    print(
        "max  :",
        x0_raw.max().item()
    )

    print()
    print(
        "✓ Reverse diffusion returned normalized x₀"
    )

    print(
        "✓ EMA RMS was applied to recover raw x₀"
    )

    print(
        "✓ Raw x₀ is ready for SELFIES Decoder"
    )

    return x0_raw, x0_norm


# ============================================================
# TEST
# ============================================================

with torch.no_grad():

    _, target_properties, _ = next(
        iter(test_loader)
    )

    target_properties = (
        target_properties.to(device)
    )

    diagnostic_x0_raw, diagnostic_x0_norm = (
        reverse_diffusion_diagnostic(
            model=model,
            target_properties=target_properties,
            diffusion_schedule=diffusion_schedule,
            latent_dim=EMBED_DIM,
            device=device,
            print_every=50
        )
    )

In [ ]:
# ============================================================
# 46. PCA CLEAN LATENT x₀
# ============================================================

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

pca = PCA(
    n_components=2
)


x0_pca = pca.fit_transform(
    x0_np
)


print(
    "Explained variance:"
)

print(
    f"PC1: {pca.explained_variance_ratio_[0] * 100:.2f}%"
)

print(
    f"PC2: {pca.explained_variance_ratio_[1] * 100:.2f}%"
)

print(
    f"Total: "
    f"{pca.explained_variance_ratio_.sum() * 100:.2f}%"
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 8)
)

plt.scatter(
    x0_pca[:, 0],
    x0_pca[:, 1],
    s=8,
    alpha=0.4
)

plt.xlabel(
    "PC1"
)

plt.ylabel(
    "PC2"
)

plt.title(
    "PCA of Clean Latent x₀"
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 47. PCA CLEAN LATENT x₀ COLORED BY logP
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


@torch.no_grad()
def collect_x0_and_logp(
    model,
    dataloader,
    device,
    scaler,
    max_batches=100
):

    model.eval()

    all_x0 = []
    all_properties = []

    for batch_idx, (
        batch,
        properties,
        fingerprint
    ) in enumerate(dataloader):

        if batch_idx >= max_batches:
            break

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        x0 = model.encoder(
            batch,
            properties,
            fingerprint
        )

        all_x0.append(
            x0.cpu()
        )

        all_properties.append(
            properties.cpu()
        )

    if not all_x0:
        raise RuntimeError(
            "Не удалось получить clean latent x₀"
        )

    x0 = torch.cat(
        all_x0,
        dim=0
    )

    properties_scaled = torch.cat(
        all_properties,
        dim=0
    ).numpy()

    properties_original = scaler.inverse_transform(
        properties_scaled
    )

    logp_values = properties_original[:, 0]

    return (
        x0.numpy(),
        logp_values
    )


# ------------------------------------------------------------
# Collect x₀ and logP
# ------------------------------------------------------------

x0_np, logp_values = collect_x0_and_logp(
    model,
    val_loader,
    device,
    property_scaler,
    max_batches=100
)


# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

pca = PCA(
    n_components=2
)

x0_pca = pca.fit_transform(
    x0_np
)


print(
    "Clean latent x₀ shape:",
    x0_np.shape
)

print(
    "Explained variance:"
)

print(
    f"PC1: {pca.explained_variance_ratio_[0] * 100:.2f}%"
)

print(
    f"PC2: {pca.explained_variance_ratio_[1] * 100:.2f}%"
)

print(
    f"Total: "
    f"{pca.explained_variance_ratio_.sum() * 100:.2f}%"
)

print()

print(
    "logP range:",
    f"{logp_values.min():.3f} ... {logp_values.max():.3f}"
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(10, 8)
)

scatter = plt.scatter(
    x0_pca[:, 0],
    x0_pca[:, 1],
    c=logp_values,
    s=8,
    alpha=0.5
)

plt.xlabel(
    "PC1"
)

plt.ylabel(
    "PC2"
)

plt.title(
    "Clean Latent x₀ Colored by logP"
)

plt.colorbar(
    scatter,
    label="logP"
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# 48. PCA CLEAN LATENT x₀ COLORED BY PROPERTIES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


# ============================================================
# НАСТРОЙКИ
# ============================================================

PROPERTY_NAMES_PCA = [
    "logP",
    "QED",
    "SAS"
]

LOGP_IDX = 0
QED_IDX = 1
SAS_IDX = 2

MAX_PCA_BATCHES = 100


# ============================================================
# COLLECT CLEAN LATENT x₀ + PROPERTIES
# ============================================================

@torch.no_grad()
def collect_validation_x0_and_properties(
    model,
    dataloader,
    device,
    scaler,
    max_batches=100
):

    model.eval()

    all_x0 = []
    all_properties = []

    for batch_idx, (
        batch,
        properties,
        fingerprint
    ) in enumerate(dataloader):

        if batch_idx >= max_batches:
            break

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Clean latent x₀
        # ----------------------------------------------------

        x0 = model.encoder(
            batch,
            properties,
            fingerprint
        )

        all_x0.append(
            x0.cpu()
        )

        # ----------------------------------------------------
        # Properties
        # ----------------------------------------------------

        all_properties.append(
            properties.cpu()
        )

    if not all_x0:

        raise RuntimeError(
            "Не удалось получить clean latent x₀."
        )

    if not all_properties:

        raise RuntimeError(
            "Не удалось получить validation properties."
        )

    x0 = torch.cat(
        all_x0,
        dim=0
    )

    properties_scaled = torch.cat(
        all_properties,
        dim=0
    ).numpy()

    properties_original = (
        scaler.inverse_transform(
            properties_scaled
        )
    )

    return (
        x0.numpy(),
        properties_original
    )


# ============================================================
# COLLECT DATA
# ============================================================

x0_np, properties_original = (
    collect_validation_x0_and_properties(
        model=model,
        dataloader=val_loader,
        device=device,
        scaler=property_scaler,
        max_batches=MAX_PCA_BATCHES
    )
)


# ============================================================
# ПРОВЕРКА РАЗМЕРОВ
# ============================================================

n_samples = len(
    x0_np
)

if len(properties_original) != n_samples:

    raise RuntimeError(
        "Количество properties "
        "не совпадает с количеством x₀."
    )


if x0_np.shape[1] != EMBED_DIM:

    raise RuntimeError(
        f"Unexpected latent dimension: "
        f"{x0_np.shape[1]}. "
        f"Expected: {EMBED_DIM}."
    )


# ============================================================
# PCA
# ============================================================

pca = PCA(
    n_components=2
)

x0_pca = pca.fit_transform(
    x0_np
)


# ============================================================
# PCA INFORMATION
# ============================================================

explained_pc1 = (
    pca.explained_variance_ratio_[0]
    * 100
)

explained_pc2 = (
    pca.explained_variance_ratio_[1]
    * 100
)

explained_total = (
    explained_pc1
    + explained_pc2
)


print(
    "=" * 70
)

print(
    "PCA CLEAN LATENT x₀ COLORED BY PROPERTIES"
)

print(
    "=" * 70
)

print()

print(
    "Samples:",
    n_samples
)

print(
    "Latent dimensions:",
    x0_np.shape[1]
)

print()

print(
    "Explained variance:"
)

print(
    f"PC1:   {explained_pc1:.2f}%"
)

print(
    f"PC2:   {explained_pc2:.2f}%"
)

print(
    f"Total: {explained_total:.2f}%"
)

print()


# ============================================================
# ИЗВЛЕКАЕМ PROPERTIES
# ============================================================

logp_values = (
    properties_original[:, LOGP_IDX]
)

qed_values = (
    properties_original[:, QED_IDX]
)

sas_values = (
    properties_original[:, SAS_IDX
    ]
)


# ============================================================
# PROPERTY STATISTICS
# ============================================================

print(
    "Property statistics:"
)

print()

print(
    f"logP: "
    f"min={logp_values.min():.3f}, "
    f"max={logp_values.max():.3f}, "
    f"mean={logp_values.mean():.3f}, "
    f"std={logp_values.std():.3f}"
)

print(
    f"QED:  "
    f"min={qed_values.min():.3f}, "
    f"max={qed_values.max():.3f}, "
    f"mean={qed_values.mean():.3f}, "
    f"std={qed_values.std():.3f}"
)

print(
    f"SAS:  "
    f"min={sas_values.min():.3f}, "
    f"max={sas_values.max():.3f}, "
    f"mean={sas_values.mean():.3f}, "
    f"std={sas_values.std():.3f}"
)

print()


# ============================================================
# 1. PCA + logP
# ============================================================

plt.figure(
    figsize=(10, 8)
)

scatter = plt.scatter(
    x0_pca[:, 0],
    x0_pca[:, 1],
    c=logp_values,
    s=8,
    alpha=0.5
)

plt.xlabel(
    f"PC1 ({explained_pc1:.2f}%)"
)

plt.ylabel(
    f"PC2 ({explained_pc2:.2f}%)"
)

plt.title(
    "Clean Latent x₀ Colored by logP"
)

plt.colorbar(
    scatter,
    label="logP"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 2. PCA + QED
# ============================================================

plt.figure(
    figsize=(10, 8)
)

scatter = plt.scatter(
    x0_pca[:, 0],
    x0_pca[:, 1],
    c=qed_values,
    s=8,
    alpha=0.5
)

plt.xlabel(
    f"PC1 ({explained_pc1:.2f}%)"
)

plt.ylabel(
    f"PC2 ({explained_pc2:.2f}%)"
)

plt.title(
    "Clean Latent x₀ Colored by QED"
)

plt.colorbar(
    scatter,
    label="QED"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 3. PCA + SAS
# ============================================================

plt.figure(
    figsize=(10, 8)
)

scatter = plt.scatter(
    x0_pca[:, 0],
    x0_pca[:, 1],
    c=sas_values,
    s=8,
    alpha=0.5
)

plt.xlabel(
    f"PC1 ({explained_pc1:.2f}%)"
)

plt.ylabel(
    f"PC2 ({explained_pc2:.2f}%)"
)

plt.title(
    "Clean Latent x₀ Colored by SAS"
)

plt.colorbar(
    scatter,
    label="SAS"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# FINAL CHECK
# ============================================================

assert len(x0_np) == len(properties_original)

assert len(x0_pca) == len(properties_original)

assert np.isfinite(x0_np).all()

assert np.isfinite(properties_original).all()

print()

print(
    "✓ Clean latent x₀ собран"
)

print(
    "✓ Properties собраны из того же validation subset"
)

print(
    "✓ PCA выполнен"
)

print(
    "✓ PCA-графики для logP, QED и SAS построены"
)

=============================================================================================================================
=============================================================================================================================
=============================================================================================================================

target descriptors
        ↓
   condition
        +
x_T ~ N(0,I)
        ↓
reverse diffusion
T → ... → 0
        ↓
generated x₀
        ↓
SELFIES Decoder
        ↓
SELFIES
        ↓
SMILES

In [ ]:
# ============================================================
# 43-A. SELFIES DECODER — MODEL COMPONENT TEST
# ============================================================

model.eval()

batch, properties, fingerprint = next(iter(train_loader))

batch = batch.to(
    device,
    non_blocking=True
)

properties = properties.to(
    device,
    non_blocking=True
)

fingerprint = fingerprint.to(
    device,
    non_blocking=True
)

with torch.no_grad():

    x0 = model.encoder(
        batch,
        properties,
        fingerprint
    )

    decoder_input = batch[:, :-1]

    logits = model.decoder(
        x0,
        decoder_input
    )


# ============================================================
# ПРОВЕРКА
# ============================================================

print("=" * 70)
print("SELFIES DECODER — MODEL COMPONENT TEST")
print("=" * 70)

print()

print(
    "Input SELFIES       :",
    batch.shape
)

print(
    "Decoder input       :",
    decoder_input.shape
)

print(
    "Clean latent x₀     :",
    x0.shape
)

print(
    "Decoder output      :",
    logits.shape
)

print()

print(
    "Expected x₀         :",
    (batch.size(0), EMBED_DIM)
)

print(
    "Expected logits     :",
    (batch.size(0), MAX_LEN - 1, VOCAB_SIZE)
)


# ============================================================
# ASSERTIONS
# ============================================================

assert x0.shape == (
    batch.size(0),
    EMBED_DIM
)

assert logits.shape == (
    batch.size(0),
    MAX_LEN - 1,
    VOCAB_SIZE
)

assert torch.isfinite(x0).all()
assert torch.isfinite(logits).all()


print()

print("✓ Используется model.decoder")
print("✓ Encoder → x₀ работает")
print("✓ x₀ → SELFIES Decoder работает")
print("✓ Causal mask работает")
print("✓ Размерность logits корректна")
print("✓ Отдельный Decoder не создаётся")

In [ ]:
# ============================================================
# 45. SELFIES DECODER RECONSTRUCTION TEST
# ============================================================

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------
from pathlib import Path

reconstruction_checkpoint_path = (
    Path(CHECKPOINT_DIR) /
    "diffusion_ema_best.pt"
)


print("=" * 70)
print("SELFIES DECODER RECONSTRUCTION TEST")
print("=" * 70)
print()

if not reconstruction_checkpoint_path.exists():

    raise FileNotFoundError(
        f"Checkpoint не найден:\n"
        f"{reconstruction_checkpoint_path}\n\n"
        f"Сначала обучите Conditional Latent Diffusion Model."
    )


# ------------------------------------------------------------
# Загружаем общий checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    reconstruction_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    f"Checkpoint epoch     : "
    f"{checkpoint['epoch']}"
)

print(
    f"Best validation loss : "
    f"{checkpoint['best_val_loss']:.6f}"
)

print(
    f"Model type           : "
    f"{checkpoint.get('model_type', 'unknown')}"
)

print()


# ============================================================
# ПРОВЕРКА ТИПА МОДЕЛИ
# ============================================================

if checkpoint.get(
    "model_type"
) != "ConditionalLatentDiffusionWithReconstruction":

    raise RuntimeError(
        "Checkpoint не относится к текущей "
        "Conditional Latent Diffusion Model."
    )


# ============================================================
# КОЛИЧЕСТВО МОЛЕКУЛ
# ============================================================

N_TEST = 1000

all_original = []
all_reconstructed = []

total_checked = 0


# ============================================================
# RECONSTRUCTION
# ============================================================

with torch.no_grad():

    for batch, properties, fingerprint in test_loader:

        if total_checked >= N_TEST:
            break

        batch = batch.to(
            device,
            non_blocking=True
        )

        properties = properties.to(
            device,
            non_blocking=True
        )

        fingerprint = fingerprint.to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # Clean latent x₀
        # ----------------------------------------------------

        x0 = model.encoder(
            batch,
            properties,
            fingerprint
        )


        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        batch_size = batch.size(0)

        generated = torch.full(
            (batch_size, 1),
            START_IDX,
            dtype=torch.long,
            device=device
        )

        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=device
        )


        for step in range(
            MAX_LEN - 1
        ):

            logits = model.decoder(
                x0,
                generated
            )

            next_token = logits[
                :, -1, :
            ].argmax(
                dim=-1
            )


            # ------------------------------------------------
            # Завершившимся последовательностям
            # оставляем END token
            # ------------------------------------------------

            next_token = torch.where(
                finished,
                torch.full_like(
                    next_token,
                    END_IDX
                ),
                next_token
            )


            generated = torch.cat(
                [
                    generated,
                    next_token.unsqueeze(1)
                ],
                dim=1
            )


            finished = (
                finished |
                (next_token == END_IDX)
            )


            if finished.all():
                break


        # ----------------------------------------------------
        # Сохраняем только нужное количество
        # ----------------------------------------------------

        current_n = min(
            batch_size,
            N_TEST - total_checked
        )


        all_original.extend(
            batch[
                :current_n
            ].cpu().tolist()
        )

        all_reconstructed.extend(
            generated[
                :current_n
            ].cpu().tolist()
        )

        total_checked += current_n


print(
    f"Проверено молекул: {total_checked}"
)

print()


# ============================================================
# TOKEN IDs → SELFIES TOKENS
# ============================================================

def decode_tokens(
    token_ids
):

    """
    Преобразует последовательность token IDs
    обратно в строку SELFIES.
    """

    tokens = []

    for idx in token_ids:

        token = idx_to_token.get(
            int(idx),
            "<UNK>"
        )

        if token == "<START>":
            continue

        if token == "<END>":
            break

        if token == "<PAD>":
            continue

        tokens.append(token)

    return "".join(tokens)


original_selfies = [
    decode_tokens(sequence)
    for sequence in all_original
]

reconstructed_selfies = [
    decode_tokens(sequence)
    for sequence in all_reconstructed
]


# ============================================================
# SELFIES → SMILES
# ============================================================

original_smiles = []
reconstructed_smiles = []


for original, reconstructed in zip(
    original_selfies,
    reconstructed_selfies
):

    # --------------------------------------------------------
    # Original SELFIES
    # --------------------------------------------------------

    try:

        original_smiles.append(
            sf.decoder(original)
        )

    except Exception:

        original_smiles.append(
            None
        )


    # --------------------------------------------------------
    # Reconstructed SELFIES
    # --------------------------------------------------------

    try:

        reconstructed_smiles.append(
            sf.decoder(reconstructed)
        )

    except Exception:

        reconstructed_smiles.append(
            None
        )


# ============================================================
# RDKit VALIDITY
# ============================================================

valid_original = 0
valid_reconstructed = 0

exact_matches = 0
canonical_matches = 0


for (
    original_smiles_i,
    reconstructed_smiles_i
) in zip(
    original_smiles,
    reconstructed_smiles
):

    original_mol = None
    reconstructed_mol = None


    # --------------------------------------------------------
    # Original
    # --------------------------------------------------------

    if original_smiles_i is not None:

        original_mol = Chem.MolFromSmiles(
            original_smiles_i
        )


    # --------------------------------------------------------
    # Reconstruction
    # --------------------------------------------------------

    if reconstructed_smiles_i is not None:

        reconstructed_mol = Chem.MolFromSmiles(
            reconstructed_smiles_i
        )


    # --------------------------------------------------------
    # Validity
    # --------------------------------------------------------

    if original_mol is not None:

        valid_original += 1


    if reconstructed_mol is not None:

        valid_reconstructed += 1


    # --------------------------------------------------------
    # Exact SMILES
    # --------------------------------------------------------

    if (
        original_smiles_i is not None
        and reconstructed_smiles_i is not None
        and original_smiles_i == reconstructed_smiles_i
    ):

        exact_matches += 1


    # --------------------------------------------------------
    # Canonical SMILES
    # --------------------------------------------------------

    if (
        original_mol is not None
        and reconstructed_mol is not None
    ):

        original_canonical = Chem.MolToSmiles(
            original_mol,
            canonical=True
        )

        reconstructed_canonical = Chem.MolToSmiles(
            reconstructed_mol,
            canonical=True
        )

        if (
            original_canonical ==
            reconstructed_canonical
        ):

            canonical_matches += 1


# ============================================================
# RESULTS
# ============================================================

reconstruction_validity = (
    100.0 *
    valid_reconstructed /
    total_checked
)

exact_accuracy = (
    100.0 *
    exact_matches /
    total_checked
)

canonical_accuracy = (
    100.0 *
    canonical_matches /
    total_checked
)


print("=" * 70)
print("RECONSTRUCTION RESULTS")
print("=" * 70)

print()

print(
    f"Original RDKit valid       : "
    f"{valid_original} / {total_checked} "
    f"({100.0 * valid_original / total_checked:.2f}%)"
)

print(
    f"Reconstructed RDKit valid  : "
    f"{valid_reconstructed} / {total_checked} "
    f"({reconstruction_validity:.2f}%)"
)

print(
    f"Exact SMILES matches       : "
    f"{exact_matches} / {total_checked} "
    f"({exact_accuracy:.2f}%)"
)

print(
    f"Canonical SMILES matches   : "
    f"{canonical_matches} / {total_checked} "
    f"({canonical_accuracy:.2f}%)"
)

print()


# ============================================================
# EXAMPLES
# ============================================================

N_EXAMPLES = min(
    10,
    total_checked
)


print("=" * 70)
print("RECONSTRUCTION EXAMPLES")
print("=" * 70)

print()


for i in range(
    N_EXAMPLES
):

    print(
        f"[{i + 1}]"
    )

    print(
        "Original SELFIES      :",
        original_selfies[i]
    )

    print(
        "Reconstructed SELFIES :",
        reconstructed_selfies[i]
    )

    print(
        "Original SMILES       :",
        original_smiles[i]
    )

    print(
        "Reconstructed SMILES  :",
        reconstructed_smiles[i]
    )


    # --------------------------------------------------------
    # Canonical match
    # --------------------------------------------------------

    canonical_match = False

    if (
        original_smiles[i] is not None
        and reconstructed_smiles[i] is not None
    ):

        original_mol = Chem.MolFromSmiles(
            original_smiles[i]
        )

        reconstructed_mol = Chem.MolFromSmiles(
            reconstructed_smiles[i]
        )

        if (
            original_mol is not None
            and reconstructed_mol is not None
        ):

            canonical_match = (
                Chem.MolToSmiles(
                    original_mol,
                    canonical=True
                )
                ==
                Chem.MolToSmiles(
                    reconstructed_mol,
                    canonical=True
                )
            )


    print(
        "Canonical match       :",
        canonical_match
    )

    print(
        "-" * 70
    )


# ============================================================
# FINAL CHECK
# ============================================================

assert total_checked > 0

print()

print(
    "✓ Общий checkpoint загружен"
)

print(
    "✓ Используется model.encoder"
)

print(
    "✓ Используется model.decoder"
)

print(
    "✓ x₀ → SELFIES reconstruction выполнена"
)

print(
    "✓ SELFIES → SMILES выполнена"
)

print(
    "✓ RDKit validity проверена"
)

print(
    "✓ Reconstruction test завершён"
)

In [ ]:
print()
print("=" * 70)
print("DIFFUSION SCALE DIAGNOSTICS")
print("=" * 70)

with torch.no_grad():

    # Берём один batch
    batch, properties, fingerprint = next(iter(test_loader))

    batch = batch.to(device)
    properties = properties.to(device)
    fingerprint = fingerprint.to(device)

    # Реальный latent из Encoder
    x0_real = model.encoder(
        batch,
        properties,
        fingerprint
    )

    print("\nREAL x₀ FROM ENCODER")
    print("mean :", x0_real.mean().item())
    print("std  :", x0_real.std().item())
    print("min  :", x0_real.min().item())
    print("max  :", x0_real.max().item())

    # Проверяем несколько timestep
    for t in [0, 1, 10, 100, 500, 900, 999]:

        timestep = torch.full(
            (x0_real.size(0),),
            t,
            dtype=torch.long,
            device=device
        )

        noise = torch.randn_like(x0_real)

        x_t = diffusion_schedule.q_sample(
            x0_real,
            timestep,
            noise
        )

        predicted_noise = model.diffusion(
            x_t,
            timestep,
            properties
        )

        print(
            f"\nt={t:4d}"
        )

        print(
            "  x_t std          :",
            x_t.std().item()
        )

        print(
            "  true noise std   :",
            noise.std().item()
        )

        print(
            "  predicted std    :",
            predicted_noise.std().item()
        )

        print(
            "  predicted mean   :",
            predicted_noise.mean().item()
        )

In [ ]:
# ============================================================
# 46A. REVERSE DIFFUSION SAMPLER
# ============================================================

@torch.no_grad()
def reverse_diffusion_step(
    model,
    x_t,
    timestep,
    target_properties,
    diffusion_schedule
):
    """
    Один шаг обратной диффузии:

        x_t → x_{t-1}

    Модель предсказывает шум ε_theta.

    Важно:
    x_t находится в NORMALIZED latent space.
    """

    # --------------------------------------------------------
    # Предсказание шума
    # --------------------------------------------------------

    predicted_noise = model.diffusion(
        x_t,
        timestep,
        target_properties
    )

    # --------------------------------------------------------
    # Параметры текущего timestep
    # --------------------------------------------------------

    beta_t = (
        diffusion_schedule.betas[timestep]
    )

    alpha_t = (
        diffusion_schedule.alphas[timestep]
    )

    alpha_bar_t = (
        diffusion_schedule.alpha_bars[timestep]
    )

    # --------------------------------------------------------
    # DDPM posterior mean
    #
    # μ_theta =
    # 1 / sqrt(alpha_t) *
    # (
    #     x_t -
    #     beta_t / sqrt(1-alpha_bar_t) *
    #     predicted_noise
    # )
    # --------------------------------------------------------

    mean = (
        diffusion_schedule.sqrt_recip_alphas[timestep].unsqueeze(1)
        * (
            x_t
            - (
                beta_t /
                torch.sqrt(
                    1.0 - alpha_bar_t
                )
            ).unsqueeze(1)
            * predicted_noise
        )
    )

    # --------------------------------------------------------
    # Если t > 0, добавляем случайный шум.
    #
    # При t = 0 получаем окончательный x₀_norm.
    # --------------------------------------------------------

    posterior_variance = (
        diffusion_schedule.posterior_variance[timestep]
    )

    noise = torch.randn_like(
        x_t
    )

    nonzero_mask = (
        (timestep > 0)
        .float()
        .unsqueeze(1)
    )

    x_prev = (
        mean
        + nonzero_mask
        * torch.sqrt(
            posterior_variance
        ).unsqueeze(1)
        * noise
    )

    return x_prev


# ============================================================
# ПОЛНЫЙ REVERSE DIFFUSION
# ============================================================

@torch.no_grad()
def reverse_diffusion(
    model,
    target_properties,
    diffusion_schedule,
    latent_dim,
    device
):
    """
    Полный reverse diffusion:

        x_T ~ N(0, I)

        x_T → x_{T-1} → ... → x_1 → x₀_norm

    После reverse diffusion:

        x₀_norm × EMA latent RMS → x₀_raw

    Parameters
    ----------
    target_properties:
        [batch, PROPERTY_DIM]

    Returns
    -------
    x0_raw:
        generated clean RAW latent
        [batch, latent_dim]

    x0_norm:
        generated normalized latent
        [batch, latent_dim]
    """

    model.eval()

    batch_size = (
        target_properties.size(0)
    )

    # --------------------------------------------------------
    # EMA latent RMS
    # --------------------------------------------------------

    latent_rms = (
        model.latent_rms
        .detach()
        .clamp_min(1e-6)
    )

    # --------------------------------------------------------
    # Начинаем с Gaussian noise
    #
    # ВАЖНО:
    # x_t находится в normalized latent space.
    # --------------------------------------------------------

    x_t = torch.randn(
        batch_size,
        latent_dim,
        device=device
    )

    # --------------------------------------------------------
    # T → 0
    # --------------------------------------------------------

    progress_bar = tqdm(
        range(
            diffusion_schedule.num_steps - 1,
            -1,
            -1
        ),
        desc="Reverse diffusion",
        leave=False
    )

    for t in progress_bar:

        timestep = torch.full(
            (batch_size,),
            t,
            dtype=torch.long,
            device=device
        )

        x_t = reverse_diffusion_step(
            model=model,
            x_t=x_t,
            timestep=timestep,
            target_properties=target_properties,
            diffusion_schedule=diffusion_schedule
        )

    # ========================================================
    # FINAL NORMALIZED x₀
    # ========================================================

    x0_norm = x_t

    # ========================================================
    # RESTORE RAW x₀
    #
    # x₀_norm = x₀_raw / latent_rms
    #
    # therefore:
    #
    # x₀_raw = x₀_norm × latent_rms
    # ========================================================

    x0_raw = (
        x0_norm
        * latent_rms
    )

    return x0_raw, x0_norm


# ============================================================
# REVERSE DIFFUSION TEST
# ============================================================

print("=" * 70)
print("REVERSE DIFFUSION TEST")
print("=" * 70)
print()

# ------------------------------------------------------------
# Берём свойства из тестового batch
# ------------------------------------------------------------

batch, properties, fingerprint = next(
    iter(test_loader)
)

properties = properties.to(
    device,
    non_blocking=True
)

print(
    "Target properties:",
    properties.shape
)

print(
    "Diffusion steps:",
    diffusion_schedule.num_steps
)

print(
    "Latent dimension:",
    EMBED_DIM
)

print(
    "Latent RMS:",
    f"{model.latent_rms.detach().item():.6f}"
)

print()

# ------------------------------------------------------------
# Один полный reverse diffusion
# ------------------------------------------------------------

generated_x0, generated_x0_norm = (
    reverse_diffusion(
        model=model,
        target_properties=properties,
        diffusion_schedule=diffusion_schedule,
        latent_dim=EMBED_DIM,
        device=device
    )
)


# ============================================================
# ПРОВЕРКА
# ============================================================

print()
print(
    "Generated normalized x₀:",
    generated_x0_norm.shape
)

print(
    "Generated raw x₀:",
    generated_x0.shape
)

assert generated_x0_norm.shape == (
    properties.size(0),
    EMBED_DIM
)

assert generated_x0.shape == (
    properties.size(0),
    EMBED_DIM
)

assert torch.isfinite(
    generated_x0_norm
).all()

assert torch.isfinite(
    generated_x0
).all()


# ============================================================
# NORMALIZED x₀ STATISTICS
# ============================================================

print()
print("NORMALIZED x₀")

print(
    "mean:",
    generated_x0_norm.mean().item()
)

print(
    "std :",
    generated_x0_norm.std().item()
)

print(
    "min :",
    generated_x0_norm.min().item()
)

print(
    "max :",
    generated_x0_norm.max().item()
)


# ============================================================
# RAW x₀ STATISTICS
# ============================================================

print()
print("RAW x₀")

print(
    "mean:",
    generated_x0.mean().item()
)

print(
    "std :",
    generated_x0.std().item()
)

print(
    "min :",
    generated_x0.min().item()
)

print(
    "max :",
    generated_x0.max().item()
)


# ============================================================
# CONSISTENCY CHECK
# ============================================================

expected_raw = (
    generated_x0_norm
    * model.latent_rms.detach().clamp_min(1e-6)
)

normalization_error = (
    generated_x0 - expected_raw
).abs().max().item()

print()
print(
    "Normalization consistency error:",
    f"{normalization_error:.10f}"
)

assert normalization_error < 1e-6


# ============================================================
# FINAL STATUS
# ============================================================

print()

print("✓ xT ~ N(0,I) создан")
print("✓ Reverse diffusion T → 0 выполнен")
print("✓ Normalized x₀ получен")
print("✓ EMA RMS восстановил raw x₀")
print("✓ Raw x₀ содержит только конечные значения")
print("✓ Raw x₀ готов для SELFIES Decoder")
print("✓ Reverse Diffusion Sampler работает")

In [ ]:
# ============================================================
# 47A. END-TO-END MOLECULE GENERATION TEST
# ============================================================

print("=" * 70)
print("END-TO-END MOLECULE GENERATION TEST")
print("=" * 70)
print()


# ============================================================
# НАСТРОЙКИ
# ============================================================

N_GENERATION_TEST = 100


# ============================================================
# ЗАГРУЖАЕМ НОВЫЙ EMA CHECKPOINT
# ============================================================

reconstruction_checkpoint_path = (
    CHECKPOINT_DIR /
    "diffusion_ema_best.pt"
)

if not reconstruction_checkpoint_path.exists():

    raise FileNotFoundError(
        f"Checkpoint не найден:\n"
        f"{reconstruction_checkpoint_path}\n\n"
        f"Сначала обучите Conditional Latent Diffusion Model "
        f"с EMA latent RMS."
    )


checkpoint = torch.load(
    reconstruction_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print(
    f"Checkpoint epoch     : "
    f"{checkpoint['epoch']}"
)

print(
    f"Best validation loss : "
    f"{checkpoint['best_val_loss']:.6f}"
)

print(
    f"Model type           : "
    f"{checkpoint.get('model_type', 'unknown')}"
)

print(
    f"Latent normalization : "
    f"{checkpoint.get('latent_normalization', 'unknown')}"
)

print(
    f"Latent RMS           : "
    f"{model.latent_rms.detach().item():.6f}"
)

print()


# ============================================================
# ПРОВЕРКА CHECKPOINT
# ============================================================

checkpoint_model_type = checkpoint.get(
    "model_type",
    "unknown"
)

if checkpoint_model_type not in (
    "ConditionalLatentDiffusionWithReconstruction",
    "ConditionalLatentDiffusionWithReconstructionEMA"
):

    raise RuntimeError(
        "Загружен checkpoint другой архитектуры.\n"
        f"Получено: {checkpoint_model_type}\n"
        "Ожидался текущий Conditional Latent Diffusion checkpoint."
    )


if checkpoint.get(
    "latent_normalization"
) != "EMA_RMS":

    raise RuntimeError(
        "Checkpoint не использует EMA RMS normalization."
    )


# ============================================================
# БЕРЁМ TARGET PROPERTIES
# ============================================================

# ВАЖНО:
# N_GENERATION_TEST может быть больше BATCH_SIZE.
# Поэтому собираем properties из нескольких batch.

target_properties_list = []

collected = 0

with torch.no_grad():

    for batch, properties, fingerprint in test_loader:

        remaining = (
            N_GENERATION_TEST -
            collected
        )

        if remaining <= 0:
            break

        current_n = min(
            properties.size(0),
            remaining
        )

        target_properties_list.append(
            properties[
                :current_n
            ]
        )

        collected += current_n


if collected < N_GENERATION_TEST:

    raise RuntimeError(
        f"Удалось получить только "
        f"{collected} target properties "
        f"из требуемых {N_GENERATION_TEST}."
    )


properties = torch.cat(
    target_properties_list,
    dim=0
).to(
    device,
    non_blocking=True
)


print(
    "Target properties:",
    properties.shape
)

print()


# ============================================================
# 1. REVERSE DIFFUSION
# ============================================================

print("=" * 70)
print("STEP 1: REVERSE DIFFUSION")
print("=" * 70)
print()


generated_x0, generated_x0_norm = (
    reverse_diffusion(
        model=model,
        target_properties=properties,
        diffusion_schedule=diffusion_schedule,
        latent_dim=EMBED_DIM,
        device=device
    )
)


print()

print(
    "Generated normalized x₀:",
    generated_x0_norm.shape
)

print(
    "Generated raw x₀:",
    generated_x0.shape
)


assert generated_x0.shape == (
    N_GENERATION_TEST,
    EMBED_DIM
)

assert generated_x0_norm.shape == (
    N_GENERATION_TEST,
    EMBED_DIM
)


assert torch.isfinite(
    generated_x0
).all()

assert torch.isfinite(
    generated_x0_norm
).all()


# ------------------------------------------------------------
# NORMALIZED LATENT
# ------------------------------------------------------------

print()
print("Normalized x₀ statistics:")

print(
    "mean:",
    generated_x0_norm.mean().item()
)

print(
    "std :",
    generated_x0_norm.std().item()
)

print(
    "min :",
    generated_x0_norm.min().item()
)

print(
    "max :",
    generated_x0_norm.max().item()
)


# ------------------------------------------------------------
# RAW LATENT
# ------------------------------------------------------------

print()
print("Raw x₀ statistics:")

print(
    "mean:",
    generated_x0.mean().item()
)

print(
    "std :",
    generated_x0.std().item()
)

print(
    "min :",
    generated_x0.min().item()
)

print(
    "max :",
    generated_x0.max().item()
)


print()

print(
    "EMA latent RMS:",
    model.latent_rms.detach().item()
)

print()

print(
    "✓ Generated raw latent x₀ получен"
)

print()


# ============================================================
# 2. AUTOREGRESSIVE SELFIES GENERATION
# ============================================================

print("=" * 70)
print("STEP 2: SELFIES DECODING")
print("=" * 70)
print()


generated_token_ids = torch.full(
    (
        generated_x0.size(0),
        1
    ),
    START_IDX,
    dtype=torch.long,
    device=device
)


finished = torch.zeros(
    generated_x0.size(0),
    dtype=torch.bool,
    device=device
)


with torch.no_grad():

    for step in range(
        MAX_LEN - 1
    ):

        # ----------------------------------------------------
        # Decoder получает RAW x₀
        # ----------------------------------------------------

        logits = model.decoder(
            generated_x0,
            generated_token_ids
        )


        # ----------------------------------------------------
        # Наиболее вероятный следующий token
        # ----------------------------------------------------

        next_token = logits[
            :, -1, :
        ].argmax(
            dim=-1
        )


        # ----------------------------------------------------
        # Завершённым последовательностям
        # оставляем END
        # ----------------------------------------------------

        next_token = torch.where(
            finished,
            torch.full_like(
                next_token,
                END_IDX
            ),
            next_token
        )


        generated_token_ids = torch.cat(
            [
                generated_token_ids,
                next_token.unsqueeze(1)
            ],
            dim=1
        )


        finished = (
            finished |
            (next_token == END_IDX)
        )


        if finished.all():
            break


print(
    "Generated token sequences:",
    generated_token_ids.shape
)

print(
    "✓ Autoregressive SELFIES decoding выполнен"
)

print()


# ============================================================
# 3. TOKEN IDS → SELFIES
# ============================================================

generated_selfies = [
    decode_tokens(sequence)
    for sequence in generated_token_ids.cpu().tolist()
]


print(
    "Generated SELFIES:",
    len(generated_selfies)
)

print()


# ============================================================
# 4. SELFIES → SMILES
# ============================================================

generated_smiles = []

selfies_valid = 0


for selfies_string in generated_selfies:

    try:

        smiles = sf.decoder(
            selfies_string
        )

        generated_smiles.append(
            smiles
        )

        selfies_valid += 1

    except Exception:

        generated_smiles.append(
            None
        )


selfies_validity = (
    100.0 *
    selfies_valid /
    len(generated_selfies)
)


print(
    f"SELFIES valid: "
    f"{selfies_valid} / "
    f"{len(generated_selfies)} "
    f"({selfies_validity:.2f}%)"
)

print()


# ============================================================
# 5. RDKit VALIDATION
# ============================================================

valid_smiles = []
valid_molecules = []


for smiles in generated_smiles:

    if smiles is None:
        continue


    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is not None:

        valid_smiles.append(
            smiles
        )

        valid_molecules.append(
            mol
        )


rdkit_validity = (
    100.0 *
    len(valid_smiles) /
    len(generated_smiles)
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("GENERATION RESULTS")
print("=" * 70)
print()


print(
    f"Requested molecules : "
    f"{N_GENERATION_TEST}"
)


print(
    f"SELFIES valid       : "
    f"{selfies_valid} / "
    f"{N_GENERATION_TEST} "
    f"({selfies_validity:.2f}%)"
)


print(
    f"RDKit valid SMILES  : "
    f"{len(valid_smiles)} / "
    f"{N_GENERATION_TEST} "
    f"({rdkit_validity:.2f}%)"
)

print()


# ============================================================
# 6. EXAMPLES
# ============================================================

N_EXAMPLES = min(
    10,
    len(generated_selfies)
)


print("=" * 70)
print("GENERATED MOLECULE EXAMPLES")
print("=" * 70)
print()


for i in range(
    N_EXAMPLES
):

    print(
        f"[{i + 1}]"
    )


    print(
        "SELFIES:",
        generated_selfies[i]
    )


    print(
        "SMILES :",
        generated_smiles[i]
    )


    if generated_smiles[i] is not None:

        mol = Chem.MolFromSmiles(
            generated_smiles[i]
        )

        print(
            "RDKit  :",
            "VALID" if mol is not None else "INVALID"
        )

    else:

        print(
            "RDKit  : INVALID"
        )


    print(
        "-" * 70
    )


# ============================================================
# FINAL CHECK
# ============================================================

assert len(
    generated_selfies
) == N_GENERATION_TEST


assert len(
    generated_smiles
) == N_GENERATION_TEST


assert torch.isfinite(
    generated_x0
).all()


assert torch.isfinite(
    generated_x0_norm
).all()


print()

print(
    "✓ EMA checkpoint загружен"
)

print(
    "✓ Target properties → reverse diffusion"
)

print(
    "✓ Reverse diffusion → normalized x₀"
)

print(
    "✓ normalized x₀ → raw x₀ через EMA RMS"
)

print(
    "✓ Raw x₀ → model.decoder"
)

print(
    "✓ model.decoder → SELFIES"
)

print(
    "✓ SELFIES → SMILES"
)

print(
    "✓ SMILES → RDKit"
)

print()

print(
    "✓ END-TO-END GENERATION TEST COMPLETED"
)

In [ ]:
# ============================================================
# 49. GENERATE 5000 MOLECULES
#     TARGET-DESCRIPTOR-CONDITIONED LATENT DIFFUSION
# ============================================================

print("=" * 70)
print("GENERATING MOLECULES WITH CONDITIONAL LATENT DIFFUSION")
print("=" * 70)
print()


# ============================================================
# SETTINGS
# ============================================================

N_GENERATED = 5000

GENERATION_BATCH_SIZE = 32

SEED = 42

# ------------------------------------------------------------
# Temperature для SELFIES decoding
#
# 0.0 → greedy decoding
# > 0  → sampling
#
# Первый эксперимент:
# greedy decoding для оценки качества latent diffusion.
# ------------------------------------------------------------

DECODER_TEMPERATURE = 0.0


# ============================================================
# TARGET DESCRIPTORS
# ============================================================

TARGET_VALUES = {

    "logP": 2.5,

    "QED": 0.7351,

    "SAS": 3.0621,

    "MW": 329.5331,

    "TPSA": 63.9480,

    "HBD": 1.0,

    "HBA": 4.0,

    "RotatableBonds": 4.0,

    "Rings": 2.0,

    "AromaticRings": 1.0,

    "FractionCSP3": 0.45
}


# ============================================================
# CHECK TARGET DESCRIPTORS
# ============================================================

print(
    "TARGET DESCRIPTORS"
)

print(
    "=" * 70
)


for feature in FEATURE_NAMES:

    if feature not in TARGET_VALUES:

        raise ValueError(
            f"Target value отсутствует "
            f"для descriptor: {feature}"
        )

    print(
        f"{feature:<20}: "
        f"{TARGET_VALUES[feature]:.4f}"
    )


print()


# ============================================================
# TARGET ARRAY
# ============================================================

target_array = np.array(
    [
        TARGET_VALUES[feature]
        for feature in FEATURE_NAMES
    ],
    dtype=np.float32
).reshape(
    1,
    -1
)


# ============================================================
# STANDARDIZATION
# ============================================================

# Diffusion model обучалась на scaled properties.
# Используем тот же scaler, который был fitted только
# на training set.

target_scaled = property_scaler.transform(
    target_array
).astype(
    np.float32
)


target_properties = torch.tensor(
    target_scaled,
    dtype=torch.float32,
    device=device
)


print(
    "Raw target shape    :",
    target_array.shape
)

print(
    "Scaled target shape :",
    target_scaled.shape
)

print()


print(
    "Scaled target:"
)


for feature, value in zip(
    FEATURE_NAMES,
    target_scaled[0]
):

    print(
        f"{feature:<20}: {value: .4f}"
    )


print()


# ============================================================
# LOAD BEST EMA JOINT MODEL CHECKPOINT
# ============================================================

reconstruction_checkpoint_path = (
    CHECKPOINT_DIR /
    "diffusion_ema_best.pt"
)


if not reconstruction_checkpoint_path.exists():

    raise FileNotFoundError(
        f"Checkpoint не найден:\n"
        f"{reconstruction_checkpoint_path}\n\n"
        f"Сначала обучите Conditional Latent "
        f"Diffusion Model с EMA latent RMS."
    )


checkpoint = torch.load(
    reconstruction_checkpoint_path,
    map_location=device,
    weights_only=False
)


# ------------------------------------------------------------
# Проверяем тип модели
# ------------------------------------------------------------

checkpoint_model_type = checkpoint.get(
    "model_type",
    "unknown"
)

if checkpoint_model_type not in (
    "ConditionalLatentDiffusionWithReconstruction",
    "ConditionalLatentDiffusionWithReconstructionEMA"
):

    raise RuntimeError(
        "Checkpoint не соответствует "
        "текущей архитектуре.\n"
        f"Получено: {checkpoint_model_type}\n"
        "Ожидался Conditional Latent Diffusion checkpoint."
    )


if checkpoint.get(
    "latent_normalization"
) != "EMA_RMS":

    raise RuntimeError(
        "Checkpoint не использует EMA RMS normalization."
    )


# ------------------------------------------------------------
# Загружаем всю модель
#
# Encoder + Decoder + Diffusion
#
# latent_rms загружается автоматически
# как registered buffer через state_dict.
# ------------------------------------------------------------

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print(
    "Joint model checkpoint:",
    reconstruction_checkpoint_path
)

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Best validation loss:",
    f"{checkpoint['best_val_loss']:.6f}"
)

print(
    "Model type:",
    checkpoint_model_type
)

print(
    "Latent normalization:",
    checkpoint.get(
        "latent_normalization",
        "unknown"
    )
)

print(
    "Latent RMS:",
    f"{model.latent_rms.detach().item():.6f}"
)

print()

print(
    "✓ Encoder loaded"
)

print(
    "✓ Decoder loaded"
)

print(
    "✓ Diffusion loaded"
)

print(
    "✓ EMA latent RMS loaded"
)

print()


# ============================================================
# RANDOM SEED
# ============================================================

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

np.random.seed(
    SEED
)


# ============================================================
# GENERATION FUNCTION
# ============================================================

@torch.no_grad()
def generate_selfies_from_latent(
    decoder,
    x0,
    max_len,
    temperature=0.0
):

    """
    Autoregressive generation:

        x₀_raw → SELFIES tokens
    """

    batch_size = x0.size(0)


    generated = torch.full(
        (
            batch_size,
            1
        ),
        START_IDX,
        dtype=torch.long,
        device=x0.device
    )


    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=x0.device
    )


    for _ in range(
        max_len - 1
    ):

        logits = decoder(
            x0,
            generated
        )


        next_logits = logits[
            :,
            -1,
            :
        ]


        # ----------------------------------------------------
        # Greedy decoding
        # ----------------------------------------------------

        if temperature <= 0.0:

            next_token = next_logits.argmax(
                dim=-1
            )


        # ----------------------------------------------------
        # Temperature sampling
        # ----------------------------------------------------

        else:

            probabilities = torch.softmax(
                next_logits / temperature,
                dim=-1
            )


            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            ).squeeze(
                1
            )


        # ----------------------------------------------------
        # Finished sequences
        # ----------------------------------------------------

        next_token = torch.where(
            finished,
            torch.full_like(
                next_token,
                END_IDX
            ),
            next_token
        )


        generated = torch.cat(
            [
                generated,
                next_token.unsqueeze(1)
            ],
            dim=1
        )


        finished = (
            finished |
            (next_token == END_IDX)
        )


        if finished.all():

            break


    return generated


# ============================================================
# GENERATION
# ============================================================

all_generated_selfies = []

all_generated_smiles = []


n_batches = math.ceil(
    N_GENERATED /
    GENERATION_BATCH_SIZE
)


print("=" * 70)
print("DIFFUSION GENERATION")
print("=" * 70)

print()

print(
    "Requested molecules :",
    N_GENERATED
)

print(
    "Generation batch     :",
    GENERATION_BATCH_SIZE
)

print(
    "Diffusion steps      :",
    diffusion_schedule.num_steps
)

print(
    "Decoder temperature  :",
    DECODER_TEMPERATURE
)

print(
    "Latent RMS            :",
    f"{model.latent_rms.detach().item():.6f}"
)

print()


progress_bar = tqdm(
    range(n_batches),
    desc="Generating"
)


for batch_idx in progress_bar:

    current_batch_size = min(
        GENERATION_BATCH_SIZE,
        N_GENERATED -
        batch_idx * GENERATION_BATCH_SIZE
    )


    # --------------------------------------------------------
    # Same target properties for all molecules
    # --------------------------------------------------------

    batch_properties = (
        target_properties.repeat(
            current_batch_size,
            1
        )
    )


    # --------------------------------------------------------
    # Reverse diffusion
    #
    # xT → x₀_norm → x₀_raw
    # --------------------------------------------------------

    generated_x0, generated_x0_norm = (
        reverse_diffusion(
            model=model,
            target_properties=batch_properties,
            diffusion_schedule=diffusion_schedule,
            latent_dim=EMBED_DIM,
            device=device
        )
    )


    # --------------------------------------------------------
    # Проверка latent
    # --------------------------------------------------------

    if not torch.isfinite(
        generated_x0
    ).all():

        raise RuntimeError(
            f"NaN/Inf обнаружены в generated x₀ "
            f"на batch {batch_idx + 1}."
        )


    # --------------------------------------------------------
    # x₀_raw → SELFIES
    #
    # Decoder получает RAW latent.
    # --------------------------------------------------------

    generated_token_ids = (
        generate_selfies_from_latent(
            decoder=model.decoder,
            x0=generated_x0,
            max_len=MAX_LEN,
            temperature=DECODER_TEMPERATURE
        )
    )


    # --------------------------------------------------------
    # Token IDs → SELFIES
    # --------------------------------------------------------

    batch_selfies = [

        decode_tokens(sequence)

        for sequence
        in generated_token_ids.cpu().tolist()
    ]


    all_generated_selfies.extend(
        batch_selfies
    )


    # --------------------------------------------------------
    # SELFIES → SMILES
    # --------------------------------------------------------

    for selfies_string in batch_selfies:

        try:

            smiles = sf.decoder(
                selfies_string
            )

        except Exception:

            smiles = None


        all_generated_smiles.append(
            smiles
        )


    progress_bar.set_postfix(
        generated=len(
            all_generated_smiles
        )
    )


# ============================================================
# CHECK NUMBER OF GENERATED MOLECULES
# ============================================================

assert len(
    all_generated_selfies
) == N_GENERATED

assert len(
    all_generated_smiles
) == N_GENERATED


# ============================================================
# SELFIES VALIDITY
# ============================================================

n_selfies_valid = sum(
    smiles is not None
    for smiles
    in all_generated_smiles
)


selfies_validity = (
    100.0 *
    n_selfies_valid /
    N_GENERATED
)


# ============================================================
# RDKit VALIDATION
# ============================================================

valid_smiles = []

valid_molecules = []


for smiles in all_generated_smiles:

    if smiles is None:

        continue


    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is not None:

        valid_smiles.append(
            smiles
        )

        valid_molecules.append(
            mol
        )


n_valid = len(
    valid_smiles
)


validity = (
    100.0 *
    n_valid /
    N_GENERATED
)


# ============================================================
# CANONICAL SMILES
# ============================================================

canonical_smiles = []


for mol in valid_molecules:

    canonical_smiles.append(
        Chem.MolToSmiles(
            mol,
            canonical=True
        )
    )


# ------------------------------------------------------------
# Уникальные canonical SMILES
# ------------------------------------------------------------

unique_canonical_smiles = list(
    dict.fromkeys(
        canonical_smiles
    )
)


n_unique = len(
    unique_canonical_smiles
)


uniqueness = (
    100.0 *
    n_unique /
    n_valid
    if n_valid > 0
    else 0.0
)


# ============================================================
# RESULTS
# ============================================================

print()

print(
    "=" * 70
)

print(
    "GENERATION RESULTS"
)

print(
    "=" * 70
)

print()


print(
    f"Requested molecules : "
    f"{N_GENERATED}"
)


print(
    f"SELFIES valid       : "
    f"{n_selfies_valid} / "
    f"{N_GENERATED} "
    f"({selfies_validity:.2f}%)"
)


print(
    f"RDKit valid SMILES  : "
    f"{n_valid} / "
    f"{N_GENERATED} "
    f"({validity:.2f}%)"
)


print(
    f"Unique canonical    : "
    f"{n_unique} / "
    f"{n_valid} "
    f"({uniqueness:.2f}%)"
)


print()


# ============================================================
# EXAMPLES
# ============================================================

print(
    "=" * 70
)

print(
    "GENERATED MOLECULES"
)

print(
    "=" * 70
)

print()


for i, smiles in enumerate(
    unique_canonical_smiles[:20],
    start=1
):

    print(
        f"{i:02d}: {smiles}"
    )


# ============================================================
# SAVE ALL GENERATED RESULTS
# ============================================================

generation_df = pd.DataFrame({

    "selfies": all_generated_selfies,

    "smiles": all_generated_smiles

})


generation_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_all.csv"
)


generation_df.to_csv(
    generation_path,
    index=False
)


# ============================================================
# SAVE UNIQUE VALID MOLECULES
# ============================================================

unique_generation_df = pd.DataFrame({

    "smiles": unique_canonical_smiles

})


unique_generation_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_unique.csv"
)


unique_generation_df.to_csv(
    unique_generation_path,
    index=False
)


# ============================================================
# SAVE GENERATION METADATA
# ============================================================

generation_metadata = {

    "checkpoint": str(
        reconstruction_checkpoint_path
    ),

    "checkpoint_epoch": checkpoint["epoch"],

    "best_val_loss": checkpoint[
        "best_val_loss"
    ],

    "model_type": checkpoint.get(
        "model_type",
        "unknown"
    ),

    "latent_normalization": checkpoint.get(
        "latent_normalization",
        "unknown"
    ),

    # --------------------------------------------------------
    # TRAINING LOSS CONFIGURATION
    # --------------------------------------------------------

    "loss_type": checkpoint.get(
        "loss_type",
        "unknown"
    ),

    "diffusion_loss":
        "Min-SNR weighted epsilon MSE",

    "x0_loss_weight": checkpoint.get(
        "x0_loss_weight",
        X0_LOSS_WEIGHT
    ),

    "snr_gamma": checkpoint.get(
        "snr_gamma",
        SNR_GAMMA
    ),

    "min_diffusion_weight": checkpoint.get(
        "min_diffusion_weight",
        MIN_DIFFUSION_WEIGHT
    ),

    "recon_weight": checkpoint.get(
        "recon_weight",
        RECON_WEIGHT
    ),

    # --------------------------------------------------------
    # LATENT CONFIGURATION
    # --------------------------------------------------------

    "latent_rms": float(
        model.latent_rms.detach().item()
    ),

    # --------------------------------------------------------
    # GENERATION CONFIGURATION
    # --------------------------------------------------------

    "n_generated": N_GENERATED,

    "generation_batch_size":
        GENERATION_BATCH_SIZE,

    "diffusion_steps":
        diffusion_schedule.num_steps,

    "decoder_temperature":
        DECODER_TEMPERATURE,

    "seed": SEED,

    # --------------------------------------------------------
    # GENERATION RESULTS
    # --------------------------------------------------------

    "selfies_valid":
        n_selfies_valid,

    "selfies_validity":
        selfies_validity,

    "rdkit_valid":
        n_valid,

    "rdkit_validity":
        validity,

    "unique_canonical":
        n_unique,

    "uniqueness":
        uniqueness
}


generation_metadata_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_metadata.json"
)


with open(
    generation_metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        generation_metadata,
        f,
        ensure_ascii=False,
        indent=2
    )


print()

print(
    f"✓ All generated molecules saved:"
)

print(
    generation_path
)

print()

print(
    f"✓ Unique canonical molecules saved:"
)

print(
    unique_generation_path
)

print()

print(
    f"✓ Generation metadata saved:"
)

print(
    generation_metadata_path
)

print()

print(
    "✓ EMA joint checkpoint loaded"
)

print(
    "✓ Target descriptors standardized"
)

print(
    "✓ Reverse diffusion completed"
)

print(
    "✓ Normalized x₀ → raw x₀ through EMA RMS"
)

print(
    "✓ Raw x₀ → model.decoder"
)

print(
    "✓ x₀ → SELFIES completed"
)

print(
    "✓ SELFIES → SMILES completed"
)

print(
    "✓ RDKit validation completed"
)

print(
    "✓ Canonicalization completed"
)

print(
    "✓ Duplicate removal completed"
)

print()

print(
    "✓ 5000-molecule diffusion generation completed"
)

In [ ]:
# ============================================================
# 50. GENERATED MOLECULES EVALUATION
# ============================================================

print("=" * 70)
print("GENERATED MOLECULES EVALUATION")
print("=" * 70)
print()


# ============================================================
# LOAD GENERATED MOLECULES
# ============================================================

generation_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_all.csv"
)


if not generation_path.exists():

    raise FileNotFoundError(
        f"Файл генерации не найден:\n"
        f"{generation_path}\n\n"
        f"Сначала запустите Cell 49."
    )


generation_df = pd.read_csv(
    generation_path
)


requested_molecules = len(
    generation_df
)


print(
    f"Generated molecules loaded: "
    f"{requested_molecules}"
)

print()


# ============================================================
# RDKit MOLECULES
# ============================================================

generation_df["mol"] = generation_df[
    "smiles"
].apply(
    lambda smiles:
        Chem.MolFromSmiles(smiles)
        if pd.notna(smiles)
        else None
)


generation_df["RDKit_valid"] = (
    generation_df["mol"].notna()
)


n_valid = int(
    generation_df["RDKit_valid"].sum()
)


validity = (
    100.0 *
    n_valid /
    requested_molecules
)


print(
    f"RDKit valid molecules: "
    f"{n_valid} / {requested_molecules} "
    f"({validity:.2f}%)"
)

print()


# ============================================================
# ОСТАВЛЯЕМ ВАЛИДНЫЕ МОЛЕКУЛЫ
# ============================================================

valid_df = generation_df[
    generation_df["RDKit_valid"]
].copy()


valid_df.reset_index(
    drop=True,
    inplace=True
)


if len(valid_df) == 0:

    raise RuntimeError(
        "Нет ни одной валидной молекулы для оценки."
    )


# ============================================================
# CANONICAL SMILES
# ============================================================

valid_df["canonical_smiles"] = (
    valid_df["mol"].apply(
        lambda mol:
            Chem.MolToSmiles(
                mol,
                canonical=True
            )
    )
)


# ============================================================
# UNIQUE MOLECULES
# ============================================================

unique_df = (
    valid_df
    .drop_duplicates(
        subset=["canonical_smiles"]
    )
    .copy()
)


unique_df.reset_index(
    drop=True,
    inplace=True
)


n_unique = len(
    unique_df
)


uniqueness = (
    100.0 *
    n_unique /
    n_valid
)


print(
    f"Unique canonical molecules: "
    f"{n_unique} / {n_valid} "
    f"({uniqueness:.2f}%)"
)

print()


# ============================================================
# DESCRIPTORS
# ============================================================

def calculate_descriptors(
    mol
):

    return {

        "logP":
            Crippen.MolLogP(mol),

        "QED":
            QED.qed(mol),

        "SAS":
            sascorer.calculateScore(mol),

        "MW":
            Descriptors.MolWt(mol),

        "TPSA":
            Descriptors.TPSA(mol),

        "HBD":
            Lipinski.NumHDonors(mol),

        "HBA":
            Lipinski.NumHAcceptors(mol),

        "RotatableBonds":
            Lipinski.NumRotatableBonds(mol),

        "Rings":
            Lipinski.RingCount(mol),

        "AromaticRings":
            Lipinski.NumAromaticRings(mol),

        "FractionCSP3":
            rdMolDescriptors.CalcFractionCSP3(mol)
    }


descriptor_rows = []


for mol in tqdm(
    valid_df["mol"],
    desc="Calculating descriptors"
):

    descriptor_rows.append(
        calculate_descriptors(mol)
    )


descriptor_df = pd.DataFrame(
    descriptor_rows
)


valid_df = pd.concat(
    [
        valid_df,
        descriptor_df
    ],
    axis=1
)


# ============================================================
# LIPINSKI
# ============================================================

def passes_lipinski(
    row
):

    return (

        row["MW"] <= 500

        and row["logP"] <= 5

        and row["HBD"] <= 5

        and row["HBA"] <= 10
    )


valid_df["Lipinski"] = valid_df.apply(
    passes_lipinski,
    axis=1
)


# ============================================================
# VEBER
# ============================================================

def passes_veber(
    row
):

    return (

        row["TPSA"] <= 140

        and row["RotatableBonds"] <= 10
    )


valid_df["Veber"] = valid_df.apply(
    passes_veber,
    axis=1
)


# ============================================================
# PAINS
# ============================================================

pains_params = FilterCatalogParams()

pains_params.AddCatalog(
    FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS
)

pains_catalog = FilterCatalog(
    pains_params
)


def is_pains_free(
    mol
):

    return not pains_catalog.HasMatch(
        mol
    )


valid_df["PAINS_free"] = [
    is_pains_free(mol)
    for mol in tqdm(
        valid_df["mol"],
        desc="PAINS filtering"
    )
]


# ============================================================
# ALL FILTERS
# ============================================================

valid_df["All_filters"] = (
    valid_df["Lipinski"]
    & valid_df["Veber"]
    & valid_df["PAINS_free"]
)


# ============================================================
# TARGET PROPERTY ERRORS
# ============================================================

for feature in FEATURE_NAMES:

    target = TARGET_VALUES[
        feature
    ]

    valid_df[
        f"{feature}_error"
    ] = (
        valid_df[feature]
        - target
    ).abs()


# ============================================================
# FILTER SUMMARY
# ============================================================

n_total_valid = len(
    valid_df
)


n_lipinski = int(
    valid_df["Lipinski"].sum()
)


n_veber = int(
    valid_df["Veber"].sum()
)


n_pains_free = int(
    valid_df["PAINS_free"].sum()
)


n_all_filters = int(
    valid_df["All_filters"].sum()
)


print(
    "=" * 70
)

print(
    "FILTER RESULTS"
)

print(
    "=" * 70
)

print()


print(
    f"Lipinski   : "
    f"{n_lipinski} / {n_total_valid} "
    f"({100*n_lipinski/n_total_valid:.2f}%)"
)


print(
    f"Veber      : "
    f"{n_veber} / {n_total_valid} "
    f"({100*n_veber/n_total_valid:.2f}%)"
)


print(
    f"PAINS-free : "
    f"{n_pains_free} / {n_total_valid} "
    f"({100*n_pains_free/n_total_valid:.2f}%)"
)


print(
    f"All filters: "
    f"{n_all_filters} / {n_total_valid} "
    f"({100*n_all_filters/n_total_valid:.2f}%)"
)


print()


# ============================================================
# DESCRIPTOR STATISTICS
# ============================================================

print(
    "=" * 70
)

print(
    "GENERATED DESCRIPTOR STATISTICS"
)

print(
    "=" * 70
)

print()


for feature in FEATURE_NAMES:

    values = valid_df[
        feature
    ]

    target = TARGET_VALUES[
        feature
    ]

    mean_value = values.mean()

    median_value = values.median()

    std_value = values.std()

    mean_error = valid_df[
        f"{feature}_error"
    ].mean()


    print(
        f"{feature:<20} "
        f"target={target:8.3f} | "
        f"mean={mean_value:8.3f} | "
        f"median={median_value:8.3f} | "
        f"std={std_value:8.3f} | "
        f"MAE={mean_error:8.3f}"
    )


# ============================================================
# TARGET DISTANCE
#
# Расстояние считается в standardized descriptor space.
# Это позволяет сравнивать признаки с разными единицами
# измерения.
# ============================================================

target_array_for_distance = np.array(
    [
        [
            TARGET_VALUES[feature]
            for feature in FEATURE_NAMES
        ]
    ],
    dtype=np.float32
)


target_scaled_for_distance = (
    property_scaler.transform(
        target_array_for_distance
    )[0]
)


generated_values = valid_df[
    FEATURE_NAMES
].values


generated_scaled = (
    property_scaler.transform(
        generated_values
    )
)


distances = np.sqrt(
    np.mean(
        (
            generated_scaled
            - target_scaled_for_distance
        ) ** 2,
        axis=1
    )
)


valid_df[
    "target_distance"
] = distances


# ============================================================
# TARGET DISTANCE STATISTICS
# ============================================================

mean_target_distance = (
    valid_df["target_distance"].mean()
)

median_target_distance = (
    valid_df["target_distance"].median()
)

min_target_distance = (
    valid_df["target_distance"].min()
)

max_target_distance = (
    valid_df["target_distance"].max()
)


print()

print(
    "=" * 70
)

print(
    "TARGET DISTANCE"
)

print(
    "=" * 70
)

print()

print(
    f"Mean distance   : "
    f"{mean_target_distance:.4f}"
)

print(
    f"Median distance : "
    f"{median_target_distance:.4f}"
)

print(
    f"Minimum distance: "
    f"{min_target_distance:.4f}"
)

print(
    f"Maximum distance: "
    f"{max_target_distance:.4f}"
)

print()


# ============================================================
# CLOSEST MOLECULES
# ============================================================

closest_df = (
    valid_df
    .sort_values(
        "target_distance"
    )
    .head(20)
    .copy()
)


print()

print(
    "=" * 70
)

print(
    "CLOSEST MOLECULES TO TARGET"
)

print(
    "=" * 70
)

print()


for i, (_, row) in enumerate(
    closest_df.iterrows(),
    start=1
):

    print(
        f"{i:02d}: "
        f"distance={row['target_distance']:.4f} | "
        f"SMILES={row['canonical_smiles']}"
    )


# ============================================================
# SAVE FULL EVALUATION
# ============================================================

evaluation_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_evaluation.csv"
)


valid_df.drop(
    columns=["mol"],
    errors="ignore"
).to_csv(
    evaluation_path,
    index=False
)


# ============================================================
# SAVE BEST MOLECULES
# ============================================================

best_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_best_target_matches.csv"
)


closest_df.drop(
    columns=["mol"],
    errors="ignore"
).to_csv(
    best_path,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

all_filters_of_requested = (
    100.0 *
    n_all_filters /
    requested_molecules
)


print()

print(
    "=" * 70
)

print(
    "EVALUATION FINISHED"
)

print(
    "=" * 70
)

print()


print(
    f"Requested molecules : "
    f"{requested_molecules}"
)

print(
    f"Valid molecules     : "
    f"{n_valid} "
    f"({validity:.2f}%)"
)

print(
    f"Unique molecules    : "
    f"{n_unique} "
    f"({uniqueness:.2f}% of valid)"
)

print(
    f"All filters         : "
    f"{n_all_filters} / {n_total_valid} "
    f"({100*n_all_filters/n_total_valid:.2f}% of valid)"
)

print(
    f"All filters / total : "
    f"{n_all_filters} / {requested_molecules} "
    f"({all_filters_of_requested:.2f}% of generated)"
)

print()

print(
    f"Mean target distance   : "
    f"{mean_target_distance:.4f}"
)

print(
    f"Median target distance : "
    f"{median_target_distance:.4f}"
)

print(
    f"Best target distance   : "
    f"{min_target_distance:.4f}"
)

print()


print(
    f"Full evaluation saved : "
    f"{evaluation_path}"
)

print(
    f"Best target matches   : "
    f"{best_path}"
)

print()


print(
    "✓ EMA generated molecules loaded"
)

print(
    "✓ Descriptor calculation completed"
)

print(
    "✓ RDKit validity completed"
)

print(
    "✓ Canonicalization completed"
)

print(
    "✓ Lipinski filtering completed"
)

print(
    "✓ Veber filtering completed"
)

print(
    "✓ PAINS filtering completed"
)

print(
    "✓ Target-distance calculation completed"
)

print(
    "✓ Target-distance statistics completed"
)

print(
    "✓ Evaluation files saved"
)

In [ ]:
# ============================================================
# 51. NOVELTY AND MOLECULAR DIVERSITY
# ============================================================

print("=" * 70)
print("NOVELTY AND MOLECULAR DIVERSITY")
print("=" * 70)
print()


# ============================================================
# SETTINGS
# ============================================================

NOVELTY_SAMPLE_SIZE = min(
    5000,
    len(valid_df) if "valid_df" in globals() else 0
)

DIVERSITY_SAMPLE_SIZE = min(
    1000,
    len(valid_df) if "valid_df" in globals() else 0
)

NOVEL_DIVERSITY_SAMPLE_SIZE = 500

FP_RADIUS_EVAL = 2
FP_BITS_EVAL = 2048

SEED = 42


# ============================================================
# CHECK INPUT
# ============================================================

if "valid_df" not in globals():

    raise RuntimeError(
        "Переменная valid_df не найдена.\n"
        "Сначала запустите Cell 50."
    )


if "canonical_smiles" not in valid_df.columns:

    raise RuntimeError(
        "В valid_df отсутствует canonical_smiles.\n"
        "Сначала запустите Cell 50."
    )


if "df_vae" not in globals():

    raise RuntimeError(
        "Переменная df_vae не найдена.\n"
        "Необходимо загрузить dataset ZINC250K."
    )


if "train_idx" not in globals():

    raise RuntimeError(
        "Переменная train_idx не найдена.\n"
        "Необходимо выполнить train/validation/test split."
    )


# ============================================================
# VALID GENERATED MOLECULES
# ============================================================

generated_canonical = (
    valid_df["canonical_smiles"]
    .dropna()
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# Уникальные generated molecules
# ------------------------------------------------------------

unique_generated_smiles = list(
    dict.fromkeys(
        generated_canonical
    )
)


print(
    f"Valid generated molecules : "
    f"{len(generated_canonical)}"
)

print(
    f"Unique generated molecules: "
    f"{len(unique_generated_smiles)}"
)

print()


# ============================================================
# TRAINING SET CANONICAL SMILES
# ============================================================

print(
    "Canonicalizing training molecules..."
)

train_smiles = (
    df_vae.iloc[train_idx]["smiles"]
    .dropna()
)


train_canonical_smiles = set()


for smiles in tqdm(
    train_smiles,
    desc="Training canonicalization"
):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:
        continue

    canonical = Chem.MolToSmiles(
        mol,
        canonical=True
    )

    train_canonical_smiles.add(
        canonical
    )


print()

print(
    f"Training molecules        : "
    f"{len(train_smiles)}"
)

print(
    f"Unique canonical training : "
    f"{len(train_canonical_smiles)}"
)

print()


# ============================================================
# RANDOM GENERATOR
# ============================================================

rng = np.random.default_rng(
    SEED
)


# ============================================================
# NOVELTY
# ============================================================

novelty_sample_size = min(
    NOVELTY_SAMPLE_SIZE,
    len(unique_generated_smiles)
)


if novelty_sample_size > 0:

    if novelty_sample_size < len(
        unique_generated_smiles
    ):

        novelty_indices = rng.choice(
            len(unique_generated_smiles),
            size=novelty_sample_size,
            replace=False
        )

        generated_for_novelty = [
            unique_generated_smiles[i]
            for i in novelty_indices
        ]

    else:

        generated_for_novelty = (
            unique_generated_smiles.copy()
        )

else:

    generated_for_novelty = []


novel_generated = [

    smiles

    for smiles
    in generated_for_novelty

    if smiles not in train_canonical_smiles
]


n_novel = len(
    novel_generated
)


n_novel_total = len(
    generated_for_novelty
)


novelty_rate = (
    100.0 *
    n_novel /
    n_novel_total
    if n_novel_total > 0
    else 0.0
)


print(
    "=" * 70
)

print(
    "NOVELTY"
)

print(
    "=" * 70
)

print()

print(
    f"Unique generated sample : "
    f"{n_novel_total}"
)

print(
    f"Novel molecules         : "
    f"{n_novel} / {n_novel_total} "
    f"({novelty_rate:.2f}%)"
)

print(
    f"Training-like           : "
    f"{n_novel_total - n_novel} / "
    f"{n_novel_total} "
    f"({100.0 - novelty_rate:.2f}%)"
)

print()


# ============================================================
# ADD NOVELTY TO valid_df
# ============================================================

valid_df["Novel"] = (
    ~valid_df["canonical_smiles"].isin(
        train_canonical_smiles
    )
)


# ============================================================
# MORGAN FINGERPRINT FUNCTION
# ============================================================

def get_morgan_fp(
    smiles
):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:

        return None

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=FP_RADIUS_EVAL,
        nBits=FP_BITS_EVAL
    )


# ============================================================
# UNIQUE GENERATED FINGERPRINTS
# ============================================================

print(
    "Calculating Morgan fingerprints..."
)

print()


generated_fps = [

    get_morgan_fp(
        smiles
    )

    for smiles
    in tqdm(
        unique_generated_smiles,
        desc="Generated fingerprints"
    )
]


generated_fps = [
    fp
    for fp in generated_fps
    if fp is not None
]


print()


# ============================================================
# INTRA-GENERATION DIVERSITY
# ============================================================

diversity_sample_size = min(
    DIVERSITY_SAMPLE_SIZE,
    len(generated_fps)
)


if diversity_sample_size > 0:

    if diversity_sample_size < len(
        generated_fps
    ):

        diversity_indices = rng.choice(
            len(generated_fps),
            size=diversity_sample_size,
            replace=False
        )

        diversity_fps = [
            generated_fps[i]
            for i in diversity_indices
        ]

    else:

        diversity_fps = (
            generated_fps.copy()
        )

else:

    diversity_fps = []


pairwise_similarities = []


for i in tqdm(
    range(len(diversity_fps)),
    desc="Calculating diversity"
):

    if i == 0:

        continue


    similarities = (
        DataStructs.BulkTanimotoSimilarity(
            diversity_fps[i],
            diversity_fps[:i]
        )
    )


    pairwise_similarities.extend(
        similarities
    )


if pairwise_similarities:

    mean_similarity = float(
        np.mean(
            pairwise_similarities
        )
    )

    median_similarity = float(
        np.median(
            pairwise_similarities
        )
    )

    diversity = (
        1.0 -
        mean_similarity
    )

else:

    mean_similarity = np.nan

    median_similarity = np.nan

    diversity = np.nan


# ============================================================
# RESULTS — GENERATED DIVERSITY
# ============================================================

print()

print(
    "=" * 70
)

print(
    "MOLECULAR DIVERSITY"
)

print(
    "=" * 70
)

print()

print(
    f"Unique molecule sample : "
    f"{len(diversity_fps)}"
)

print(
    f"Mean pairwise Tanimoto : "
    f"{mean_similarity:.4f}"
)

print(
    f"Median pairwise Tanimoto: "
    f"{median_similarity:.4f}"
)

print(
    f"Fingerprint diversity  : "
    f"{diversity:.4f}"
)

print()


# ============================================================
# ALL NOVEL GENERATED MOLECULES
# ============================================================

novel_unique_smiles = [

    smiles

    for smiles
    in unique_generated_smiles

    if smiles not in train_canonical_smiles
]


print(
    f"All novel unique molecules: "
    f"{len(novel_unique_smiles)}"
)

print()


# ============================================================
# NOVEL MOLECULE DIVERSITY
# ============================================================

if len(
    novel_unique_smiles
) >= 2:

    novel_sample_size = min(
        NOVEL_DIVERSITY_SAMPLE_SIZE,
        len(novel_unique_smiles)
    )


    if novel_sample_size < len(
        novel_unique_smiles
    ):

        novel_indices = rng.choice(
            len(novel_unique_smiles),
            size=novel_sample_size,
            replace=False
        )

        novel_sample_smiles = [

            novel_unique_smiles[i]

            for i in novel_indices
        ]

    else:

        novel_sample_smiles = (
            novel_unique_smiles.copy()
        )


    novel_fps = [

        get_morgan_fp(
            smiles
        )

        for smiles
        in novel_sample_smiles
    ]


    novel_fps = [
        fp
        for fp in novel_fps
        if fp is not None
    ]


    novel_pairwise = []


    for i in tqdm(
        range(len(novel_fps)),
        desc="Novel diversity"
    ):

        if i == 0:

            continue


        similarities = (
            DataStructs.BulkTanimotoSimilarity(
                novel_fps[i],
                novel_fps[:i]
            )
        )


        novel_pairwise.extend(
            similarities
        )


    if novel_pairwise:

        novel_mean_similarity = float(
            np.mean(
                novel_pairwise
            )
        )

        novel_median_similarity = float(
            np.median(
                novel_pairwise
            )
        )

        novel_diversity = (
            1.0 -
            novel_mean_similarity
        )

    else:

        novel_mean_similarity = np.nan

        novel_median_similarity = np.nan

        novel_diversity = np.nan

else:

    novel_sample_size = 0

    novel_mean_similarity = np.nan

    novel_median_similarity = np.nan

    novel_diversity = np.nan


# ============================================================
# RESULTS — NOVEL DIVERSITY
# ============================================================

print()

print(
    "=" * 70
)

print(
    "NOVEL MOLECULE DIVERSITY"
)

print(
    "=" * 70
)

print()

print(
    f"Novel unique sample    : "
    f"{novel_sample_size}"
)

print(
    f"Mean pairwise Tanimoto : "
    f"{novel_mean_similarity:.4f}"
)

print(
    f"Median pairwise Tanimoto: "
    f"{novel_median_similarity:.4f}"
)

print(
    f"Fingerprint diversity  : "
    f"{novel_diversity:.4f}"
)

print()


# ============================================================
# UPDATE EVALUATION DATAFRAME
# ============================================================

evaluation_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_evaluation.csv"
)


valid_df.to_csv(
    evaluation_path,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

novelty_summary = pd.DataFrame({

    "metric": [

        "Valid generated molecules",

        "Unique generated molecules",

        "Training unique molecules",

        "Novelty sample size",

        "Novel molecules",

        "Novelty rate (%)",

        "Diversity sample size",

        "Mean pairwise Tanimoto",

        "Median pairwise Tanimoto",

        "Fingerprint diversity",

        "Novel unique molecules",

        "Novel diversity sample size",

        "Novel mean pairwise Tanimoto",

        "Novel median pairwise Tanimoto",

        "Novel fingerprint diversity"
    ],

    "value": [

        len(
            generated_canonical
        ),

        len(
            unique_generated_smiles
        ),

        len(
            train_canonical_smiles
        ),

        n_novel_total,

        n_novel,

        novelty_rate,

        len(
            diversity_fps
        ),

        mean_similarity,

        median_similarity,

        diversity,

        len(
            novel_unique_smiles
        ),

        novel_sample_size,

        novel_mean_similarity,

        novel_median_similarity,

        novel_diversity
    ]
})


print(
    novelty_summary.to_string(
        index=False
    )
)


# ============================================================
# SAVE SUMMARY
# ============================================================

novelty_summary_path = (
    CHECKPOINT_DIR /
    "generated_5000_diffusion_ema_novelty_diversity.csv"
)


novelty_summary.to_csv(
    novelty_summary_path,
    index=False
)


print()

print(
    "=" * 70
)

print(
    "NOVELTY AND DIVERSITY FINISHED"
)

print(
    "=" * 70
)

print()

print(
    "Updated evaluation saved:"
)

print(
    evaluation_path
)

print()

print(
    "Novelty summary saved:"
)

print(
    novelty_summary_path
)

print()

print(
    "✓ Canonical training SMILES calculated"
)

print(
    "✓ Canonical generated SMILES calculated"
)

print(
    "✓ Novelty calculated"
)

print(
    "✓ Morgan fingerprints calculated"
)

print(
    "✓ Intra-generation diversity calculated"
)

print(
    "✓ Novel molecule diversity calculated"
)

print(
    "✓ Evaluation files updated"
)



In [ ]:
# ============================================================
# 52. FINAL CONDITIONAL LATENT DIFFUSION REPORT
# ============================================================

print("=" * 70)
print("FINAL CONDITIONAL LATENT DIFFUSION REPORT")
print("=" * 70)
print()


# ============================================================
# CHECK REQUIRED VARIABLES
# ============================================================

required_variables = [
    "history",
    "valid_df",
    "novelty_rate",
    "mean_similarity",
    "median_similarity",
    "diversity",
    "novel_mean_similarity",
    "novel_median_similarity",
    "novel_diversity",
    "n_novel",
    "n_novel_total",
    "novel_sample_size",
    "diversity_sample_size",
    "novel_unique_smiles",
    "n_lipinski",
    "n_veber",
    "n_pains_free",
    "n_all_filters"
]


missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Не найдены необходимые переменные:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_variables
        )
        + "\n\n"
        "Сначала выполните Cells 49–51."
    )


# ============================================================
# GENERATION METRICS
# ============================================================

if "requested_molecules" in globals():

    n_generated = requested_molecules

elif "N_GENERATED" in globals():

    n_generated = N_GENERATED

elif "generation_df" in globals():

    n_generated = len(
        generation_df
    )

else:

    n_generated = len(
        valid_df
    )


n_valid_generated = len(
    valid_df
)


n_unique_generated = (
    valid_df[
        "canonical_smiles"
    ].nunique()
)


generation_validity = (
    100.0 *
    n_valid_generated /
    n_generated
    if n_generated > 0
    else 0.0
)


generation_uniqueness = (
    100.0 *
    n_unique_generated /
    n_valid_generated
    if n_valid_generated > 0
    else 0.0
)


# ============================================================
# TRAINING METRICS
# ============================================================

if len(
    history["val_loss"]
) > 0:

    best_idx = int(
        np.argmin(
            history["val_loss"]
        )
    )

    best_diffusion_val_loss = (
        history["val_loss"][best_idx]
    )

    best_diffusion_epoch = (
        history["epoch"][best_idx]
    )

    best_train_loss = (
        history["train_loss"][best_idx]
    )

    best_val_diffusion_loss = (
        history[
            "val_diffusion_loss"
        ][best_idx]
    )

    best_val_reconstruction_loss = (
        history[
            "val_reconstruction_loss"
        ][best_idx]
    )

    best_val_x0_loss = (
        history[
            "val_x0_loss"
        ][best_idx]
        if "val_x0_loss" in history
        else np.nan
    )

else:

    best_diffusion_val_loss = np.nan
    best_diffusion_epoch = np.nan
    best_train_loss = np.nan
    best_val_diffusion_loss = np.nan
    best_val_reconstruction_loss = np.nan
    best_val_x0_loss = np.nan


# ============================================================
# FINAL EPOCH METRICS
# ============================================================

if len(
    history["epoch"]
) > 0:

    final_epoch = (
        history["epoch"][-1]
    )

    final_train_loss = (
        history["train_loss"][-1]
    )

    final_val_loss = (
        history["val_loss"][-1]
    )

    final_train_diffusion_loss = (
        history[
            "train_diffusion_loss"
        ][-1]
    )

    final_val_diffusion_loss = (
        history[
            "val_diffusion_loss"
        ][-1]
    )

    final_train_x0_loss = (
        history[
            "train_x0_loss"
        ][-1]
        if "train_x0_loss" in history
        else np.nan
    )

    final_val_x0_loss = (
        history[
            "val_x0_loss"
        ][-1]
        if "val_x0_loss" in history
        else np.nan
    )

    final_train_reconstruction_loss = (
        history[
            "train_reconstruction_loss"
        ][-1]
    )

    final_val_reconstruction_loss = (
        history[
            "val_reconstruction_loss"
        ][-1]
    )

    final_x0_std = (
        history[
            "x0_std"
        ][-1]
    )

    final_x0_norm_std = (
        history[
            "x0_norm_std"
        ][-1]
    )

    final_latent_rms = (
        history[
            "latent_rms"
        ][-1]
    )

else:

    final_epoch = np.nan
    final_train_loss = np.nan
    final_val_loss = np.nan
    final_train_diffusion_loss = np.nan
    final_val_diffusion_loss = np.nan
    final_train_x0_loss = np.nan
    final_val_x0_loss = np.nan
    final_train_reconstruction_loss = np.nan
    final_val_reconstruction_loss = np.nan
    final_x0_std = np.nan
    final_x0_norm_std = np.nan
    final_latent_rms = np.nan


# ============================================================
# CURRENT EMA LATENT RMS
# ============================================================

if "model" in globals():

    current_latent_rms = float(
        model.latent_rms.detach()
        .cpu()
        .item()
    )

else:

    current_latent_rms = (
        final_latent_rms
    )


# ============================================================
# TARGET PROPERTY ERRORS
# ============================================================

target_mae = {}


for feature in FEATURE_NAMES:

    error_column = (
        f"{feature}_error"
    )

    if error_column in valid_df.columns:

        target_mae[feature] = float(
            valid_df[
                error_column
            ].mean()
        )

    else:

        target_mae[feature] = np.nan


# ============================================================
# TARGET DISTANCE
# ============================================================

if "target_distance" in valid_df.columns:

    target_distance_mean = float(
        valid_df[
            "target_distance"
        ].mean()
    )

    target_distance_median = float(
        valid_df[
            "target_distance"
        ].median()
    )

    target_distance_min = float(
        valid_df[
            "target_distance"
        ].min()
    )

    target_distance_max = float(
        valid_df[
            "target_distance"
        ].max()
    )

else:

    target_distance_mean = np.nan
    target_distance_median = np.nan
    target_distance_min = np.nan
    target_distance_max = np.nan


# ============================================================
# MOLECULAR FILTER PERCENTAGES
# ============================================================

n_total_valid = len(
    valid_df
)


if n_total_valid > 0:

    lipinski_rate = (
        100.0 *
        n_lipinski /
        n_total_valid
    )

    veber_rate = (
        100.0 *
        n_veber /
        n_total_valid
    )

    pains_rate = (
        100.0 *
        n_pains_free /
        n_total_valid
    )

    all_filters_rate = (
        100.0 *
        n_all_filters /
        n_total_valid
    )

else:

    lipinski_rate = np.nan
    veber_rate = np.nan
    pains_rate = np.nan
    all_filters_rate = np.nan


# ============================================================
# FILTER PERCENTAGES OF ALL REQUESTED MOLECULES
# ============================================================

if n_generated > 0:

    all_filters_generation_rate = (
        100.0 *
        n_all_filters /
        n_generated
    )

else:

    all_filters_generation_rate = np.nan


# ============================================================
# NORMALIZATION SETTINGS
# ============================================================

latent_normalization = (
    "EMA_RMS"
)


initial_latent_rms = (
    INITIAL_LATENT_RMS
    if "INITIAL_LATENT_RMS" in globals()
    else np.nan
)


latent_rms_momentum = (
    LATENT_RMS_MOMENTUM
    if "LATENT_RMS_MOMENTUM" in globals()
    else np.nan
)


# ============================================================
# DIFFUSION / LOSS SETTINGS
# ============================================================

recon_weight = (
    RECON_WEIGHT
    if "RECON_WEIGHT" in globals()
    else np.nan
)


x0_loss_weight = (
    X0_LOSS_WEIGHT
    if "X0_LOSS_WEIGHT" in globals()
    else np.nan
)


snr_gamma = (
    SNR_GAMMA
    if "SNR_GAMMA" in globals()
    else np.nan
)


min_diffusion_weight = (
    MIN_DIFFUSION_WEIGHT
    if "MIN_DIFFUSION_WEIGHT" in globals()
    else np.nan
)


# ============================================================
# SUMMARY TABLE
# ============================================================

summary_rows = [

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    (
        "Model",
        "Conditional Latent Diffusion"
    ),

    (
        "Representation",
        "SELFIES + Morgan Fingerprint + Molecular Descriptors"
    ),

    (
        "Architecture",
        "Multimodal Encoder → raw x₀ → "
        "EMA-RMS normalization → Diffusion Transformer"
    ),

    (
        "Latent dimension",
        EMBED_DIM
    ),

    (
        "Fingerprint bits",
        FP_BITS
    ),

    (
        "Fingerprint radius",
        FP_RADIUS
    ),

    (
        "Property dimension",
        PROPERTY_DIM
    ),

    (
        "SELFIES max length",
        MAX_LEN
    ),

    # --------------------------------------------------------
    # Latent normalization
    # --------------------------------------------------------

    (
        "Latent normalization",
        latent_normalization
    ),

    (
        "Initial latent RMS",
        initial_latent_rms
    ),

    (
        "EMA momentum",
        latent_rms_momentum
    ),

    (
        "Final latent RMS",
        current_latent_rms
    ),

    (
        "Final raw x₀ std",
        final_x0_std
    ),

    (
        "Final normalized x₀ std",
        final_x0_norm_std
    ),

    # --------------------------------------------------------
    # Diffusion
    # --------------------------------------------------------

    (
        "Diffusion steps",
        NUM_DIFFUSION_STEPS
    ),

    (
        "Beta start",
        BETA_START
    ),

    (
        "Beta end",
        BETA_END
    ),

    (
        "Diffusion schedule",
        DIFFUSION_SCHEDULE_TYPE
    ),

    (
        "Min-SNR gamma",
        snr_gamma
    ),

    (
        "Minimum diffusion weight",
        min_diffusion_weight
    ),

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    (
        "Diffusion loss",
        "Min-SNR weighted epsilon MSE"
    ),

    (
        "X₀ auxiliary loss",
        "MSE(predicted x₀, true x₀)"
    ),

    (
        "X₀ loss weight",
        x0_loss_weight
    ),

    (
        "Reconstruction loss",
        "SELFIES cross-entropy"
    ),

    (
        "Reconstruction weight",
        recon_weight
    ),

    (
        "Total loss",
        "L_diffusion + "
        "X0_WEIGHT × L_x0 + "
        "RECON_WEIGHT × L_reconstruction"
    ),

    (
        "Latent KL loss",
        "Not used"
    ),

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    (
        "Training epochs",
        final_epoch
    ),

    (
        "Best validation epoch",
        best_diffusion_epoch
    ),

    (
        "Best validation total loss",
        best_diffusion_val_loss
    ),

    (
        "Best validation diffusion loss",
        best_val_diffusion_loss
    ),

    (
        "Best validation X₀ loss",
        best_val_x0_loss
    ),

    (
        "Best validation reconstruction CE",
        best_val_reconstruction_loss
    ),

    (
        "Final train loss",
        final_train_loss
    ),

    (
        "Final validation loss",
        final_val_loss
    ),

    (
        "Final train diffusion loss",
        final_train_diffusion_loss
    ),

    (
        "Final validation diffusion loss",
        final_val_diffusion_loss
    ),

    (
        "Final train X₀ loss",
        final_train_x0_loss
    ),

    (
        "Final validation X₀ loss",
        final_val_x0_loss
    ),

    (
        "Final train reconstruction loss",
        final_train_reconstruction_loss
    ),

    (
        "Final validation reconstruction loss",
        final_val_reconstruction_loss
    ),

    # --------------------------------------------------------
    # Generation
    # --------------------------------------------------------

    (
        "Requested molecules",
        n_generated
    ),

    (
        "Valid SMILES",
        n_valid_generated
    ),

    (
        "Validity (%)",
        generation_validity
    ),

    (
        "Unique valid molecules",
        n_unique_generated
    ),

    (
        "Uniqueness (%)",
        generation_uniqueness
    ),

    # --------------------------------------------------------
    # Novelty / diversity
    # --------------------------------------------------------

    (
        "Novelty sample size",
        n_novel_total
    ),

    (
        "Novel molecules",
        n_novel
    ),

    (
        "Novelty (%)",
        novelty_rate
    ),

    (
        "Diversity sample size",
        diversity_sample_size
    ),

    (
        "Mean pairwise Tanimoto",
        mean_similarity
    ),

    (
        "Median pairwise Tanimoto",
        median_similarity
    ),

    (
        "Fingerprint diversity",
        diversity
    ),

    (
        "Novel unique molecules",
        len(novel_unique_smiles)
    ),

    (
        "Novel diversity sample size",
        novel_sample_size
    ),

    (
        "Novel mean Tanimoto",
        novel_mean_similarity
    ),

    (
        "Novel median Tanimoto",
        novel_median_similarity
    ),

    (
        "Novel fingerprint diversity",
        novel_diversity
    ),

    # --------------------------------------------------------
    # Target distance
    # --------------------------------------------------------

    (
        "Mean target distance",
        target_distance_mean
    ),

    (
        "Median target distance",
        target_distance_median
    ),

    (
        "Minimum target distance",
        target_distance_min
    ),

    (
        "Maximum target distance",
        target_distance_max
    ),

    # --------------------------------------------------------
    # Filters
    # --------------------------------------------------------

    (
        "Lipinski (%)",
        lipinski_rate
    ),

    (
        "Veber (%)",
        veber_rate
    ),

    (
        "PAINS-free (%)",
        pains_rate
    ),

    (
        "All filters (%) of valid",
        all_filters_rate
    ),

    (
        "All filters (%) of requested",
        all_filters_generation_rate
    )
]


summary_df = pd.DataFrame(
    summary_rows,
    columns=[
        "Metric",
        "Value"
    ]
)


# ============================================================
# PRINT SUMMARY
# ============================================================

print(
    summary_df.to_string(
        index=False
    )
)


# ============================================================
# TARGET DESCRIPTOR TABLE
# ============================================================

target_report_rows = []


for feature in FEATURE_NAMES:

    if feature not in valid_df.columns:

        continue


    generated_mean = float(
        valid_df[
            feature
        ].mean()
    )

    generated_std = float(
        valid_df[
            feature
        ].std()
    )

    generated_median = float(
        valid_df[
            feature
        ].median()
    )

    target = TARGET_VALUES[
        feature
    ]

    mae = target_mae[
        feature
    ]

    target_report_rows.append({

        "Descriptor":
            feature,

        "Target":
            target,

        "Generated Mean":
            generated_mean,

        "Generated Median":
            generated_median,

        "Generated Std":
            generated_std,

        "MAE":
            mae,

        "Mean Error":
            generated_mean - target
    })


target_report_df = pd.DataFrame(
    target_report_rows
)


# ============================================================
# PRINT TARGET REPORT
# ============================================================

print()

print(
    "=" * 70
)

print(
    "TARGET DESCRIPTOR REPORT"
)

print(
    "=" * 70
)

print()

print(
    target_report_df.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary_path = (
    CHECKPOINT_DIR /
    "diffusion_final_summary.csv"
)


target_report_path = (
    CHECKPOINT_DIR /
    "diffusion_target_descriptor_report.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


target_report_df.to_csv(
    target_report_path,
    index=False
)


# ============================================================
# TEXT REPORT
# ============================================================

text_report_path = (
    CHECKPOINT_DIR /
    "diffusion_final_report.txt"
)


with open(
    text_report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "FINAL CONDITIONAL LATENT DIFFUSION REPORT\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        summary_df.to_string(
            index=False
        )
    )

    f.write(
        "\n\n"
    )

    # --------------------------------------------------------
    # TARGET DESCRIPTOR REPORT
    # --------------------------------------------------------

    f.write(
        "TARGET DESCRIPTOR REPORT\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        target_report_df.to_string(
            index=False
        )
    )

    f.write(
        "\n\n"
    )

    # --------------------------------------------------------
    # GENERATION SUMMARY
    # --------------------------------------------------------

    f.write(
        "GENERATION SUMMARY\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        f"Requested molecules : "
        f"{n_generated}\n"
    )

    f.write(
        f"Valid molecules     : "
        f"{n_valid_generated}\n"
    )

    f.write(
        f"Validity            : "
        f"{generation_validity:.2f}%\n"
    )

    f.write(
        f"Unique molecules    : "
        f"{n_unique_generated}\n"
    )

    f.write(
        f"Uniqueness          : "
        f"{generation_uniqueness:.2f}%\n"
    )

    f.write(
        f"Novelty sample size : "
        f"{n_novel_total}\n"
    )

    f.write(
        f"Novel molecules     : "
        f"{n_novel}\n"
    )

    f.write(
        f"Novelty             : "
        f"{novelty_rate:.2f}%\n"
    )

    f.write(
        f"Diversity sample    : "
        f"{diversity_sample_size}\n"
    )

    f.write(
        f"Mean Tanimoto       : "
        f"{mean_similarity:.4f}\n"
    )

    f.write(
        f"Median Tanimoto     : "
        f"{median_similarity:.4f}\n"
    )

    f.write(
        f"Fingerprint diversity: "
        f"{diversity:.4f}\n"
    )

    f.write(
        f"Novel unique molecules: "
        f"{len(novel_unique_smiles)}\n"
    )

    f.write(
        f"Novel diversity sample: "
        f"{novel_sample_size}\n"
    )

    f.write(
        f"Novel mean Tanimoto : "
        f"{novel_mean_similarity:.4f}\n"
    )

    f.write(
        f"Novel median Tanimoto: "
        f"{novel_median_similarity:.4f}\n"
    )

    f.write(
        f"Novel diversity     : "
        f"{novel_diversity:.4f}\n"
    )

    f.write(
        "\n"
    )

    # --------------------------------------------------------
    # TARGET DISTANCE
    # --------------------------------------------------------

    f.write(
        "TARGET DISTANCE\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        f"Mean target distance   : "
        f"{target_distance_mean:.6f}\n"
    )

    f.write(
        f"Median target distance : "
        f"{target_distance_median:.6f}\n"
    )

    f.write(
        f"Minimum target distance: "
        f"{target_distance_min:.6f}\n"
    )

    f.write(
        f"Maximum target distance: "
        f"{target_distance_max:.6f}\n"
    )

    f.write(
        "\n"
    )

    # --------------------------------------------------------
    # MODEL ARCHITECTURE
    # --------------------------------------------------------

    f.write(
        "MODEL ARCHITECTURE\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        "ZINC250K\n"
    )

    f.write(
        "   ↓\n"
    )

    f.write(
        "SELFIES + Morgan Fingerprint + Descriptors\n"
    )

    f.write(
        "   ↓\n"
    )

    f.write(
        "Multimodal Fusion Encoder\n"
    )

    f.write(
        "   ↓\n"
    )

    f.write(
        "raw latent x₀ (128-D)\n"
    )

    f.write(
        "   ├──────────────→ SELFIES Decoder\n"
    )

    f.write(
        "   │                    ↓\n"
    )

    f.write(
        "   │                 SELFIES\n"
    )

    f.write(
        "   │\n"
    )

    f.write(
        "   └→ EMA-RMS normalization\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "       normalized x₀\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "      Forward Diffusion\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "             xₜ\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        " Conditional Diffusion Transformer\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "       Reverse Diffusion\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "     generated normalized x₀\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "      × saved latent RMS\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "       generated raw x₀\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "        SELFIES Decoder\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "            SELFIES\n"
    )

    f.write(
        "             ↓\n"
    )

    f.write(
        "            SMILES\n"
    )

    f.write(
        "\n"
    )

    # --------------------------------------------------------
    # LOSS FUNCTION
    # --------------------------------------------------------

    f.write(
        "LOSS FUNCTION\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        "L_total = L_diffusion + "
        f"{x0_loss_weight:.4f} × L_x0 + "
        f"{recon_weight:.4f} × L_reconstruction\n"
    )

    f.write(
        "L_diffusion = Min-SNR weighted "
        "MSE(predicted_noise, true_noise)\n"
    )

    f.write(
        "SNR gamma = "
        f"{snr_gamma:.4f}\n"
    )

    f.write(
        "Minimum diffusion weight = "
        f"{min_diffusion_weight:.4f}\n"
    )

    f.write(
        "L_x0 = MSE(predicted_x₀, true_x₀)\n"
    )

    f.write(
        "L_reconstruction = SELFIES cross-entropy\n"
    )

    f.write(
        "No VAE KL loss and no latent sampling are used.\n"
    )

    f.write(
        "\n"
    )

    # --------------------------------------------------------
    # NOVELTY / DIVERSITY
    # --------------------------------------------------------

    f.write(
        "NOVELTY / DIVERSITY\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        "Novelty is measured against the canonicalized "
        "training set.\n"
    )

    f.write(
        "Diversity is calculated from Morgan fingerprints "
        f"(radius={FP_RADIUS_EVAL}, bits={FP_BITS_EVAL}).\n"
    )

    f.write(
        "Fingerprint diversity = "
        "1 - mean pairwise Tanimoto similarity.\n"
    )

    f.write(
        "\n"
    )

    # --------------------------------------------------------
    # FILTERS
    # --------------------------------------------------------

    f.write(
        "MOLECULAR FILTERS\n"
    )

    f.write(
        "=" * 70 +
        "\n\n"
    )

    f.write(
        f"Lipinski: "
        f"{n_lipinski} / {n_total_valid} "
        f"({lipinski_rate:.2f}%)\n"
    )

    f.write(
        f"Veber: "
        f"{n_veber} / {n_total_valid} "
        f"({veber_rate:.2f}%)\n"
    )

    f.write(
        f"PAINS-free: "
        f"{n_pains_free} / {n_total_valid} "
        f"({pains_rate:.2f}%)\n"
    )

    f.write(
        f"All filters: "
        f"{n_all_filters} / {n_total_valid} "
        f"valid molecules "
        f"({all_filters_rate:.2f}%)\n"
    )

    f.write(
        f"All filters of requested: "
        f"{n_all_filters} / {n_generated} "
        f"({all_filters_generation_rate:.2f}%)\n"
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print()

print(
    "=" * 70
)

print(
    "REPORT FILES"
)

print(
    "=" * 70
)

print()

print(
    f"Summary CSV       : "
    f"{summary_path}"
)

print(
    f"Target report CSV : "
    f"{target_report_path}"
)

print(
    f"Text report       : "
    f"{text_report_path}"
)

print()

print(
    "=" * 70
)

print(
    "EXPERIMENT COMPLETED"
)

print(
    "=" * 70
)

print()

print(
    "Architecture:"
)

print(
    "  SELFIES + Fingerprint + Descriptors"
)

print(
    "              ↓"
)

print(
    "       Multimodal Encoder"
)

print(
    "              ↓"
)

print(
    "          raw x₀"
)

print(
    "          ↙    ↘"
)

print(
    " SELFIES       EMA-RMS"
)

print(
    " Decoder       normalization"
)

print(
    "                 ↓"
)

print(
    "        Forward Diffusion"
)

print(
    "                 ↓"
)

print(
    "                xₜ"
)

print(
    "                 ↓"
)

print(
    "  Conditional Diffusion Transformer"
)

print(
    "                 ↓"
)

print(
    "         Reverse Diffusion"
)

print(
    "                 ↓"
)

print(
    "        generated normalized x₀"
)

print(
    "                 ↓"
)

print(
    "         saved latent RMS"
)

print(
    "                 ↓"
)

print(
    "          generated raw x₀"
)

print(
    "                 ↓"
)

print(
    "           SELFIES Decoder"
)

print(
    "                 ↓"
)

print(
    "              SELFIES"
)

print(
    "                 ↓"
)

print(
    "              SMILES"
)

print()

print(
    "✓ Diffusion training results collected"
)

print(
    "✓ Min-SNR diffusion configuration collected"
)

print(
    "✓ X₀ auxiliary loss collected"
)

print(
    "✓ Joint reconstruction results collected"
)

print(
    "✓ EMA latent normalization collected"
)

print(
    "✓ Generation metrics collected"
)

print(
    "✓ Target descriptor metrics collected"
)

print(
    "✓ Target distance metrics collected"
)

print(
    "✓ Novelty collected"
)

print(
    "✓ Diversity collected"
)

print(
    "✓ Molecular filters collected"
)

print(
    "✓ Final report saved"
)

            logP target
                 ↓
          Conditional VAE
                 ↓
          1000 molecules
                 ↓
       насколько изменился logP?

VAE:

latent space
     ↓
выбираем z
     ↓
Decoder
     ↓
molecule

Можно экспериментировать с:

μ
σ
шумом
temperature
stochastic decoding
latent interpolation
latent optimization
выбором latent вокруг конкретной молекулы.

Temperature Decoder

Это мы у твоей старой VAE практически не использовали как отдельный систематический эксперимент.

Вместо:

token = logits.argmax(dim=-1)

можно делать sampling:

probs = torch.softmax(logits / temperature, dim=-1)
token = torch.multinomial(probs, 1)

Например:

T = 0.0  → greedy
T = 0.2
T = 0.4
T = 0.6
T = 0.8

Diffusion:

noise xT
   ↓
reverse diffusion
   ↓
x0
   ↓
Decoder
   ↓
molecule

Главное — temperature Decoder

В ячейке 48 сейчас:

DECODER_TEMPERATURE = 0.0

Это фактически жадная генерация: на каждом шаге выбирается наиболее вероятный SELFIES-токен.

Я бы проверил:

temperature = 0.0
temperature = 0.2
temperature = 0.4
temperature = 0.6
temperature = 0.8

Обычно здесь будет компромисс:

низкая temperature
    ↓
более стабильные молекулы
но меньше разнообразия

высокая temperature
    ↓
больше разнообразия
но больше ошибок / отклонения свойств

Проверить число reverse diffusion steps

Это очень интересный эксперимент.

Модель обучалась на:

NUM_DIFFUSION_STEPS = 1000

Но при генерации можно попробовать ускоренные варианты, если sampler корректно реализован под подмножество timestep'ов:

1000 steps  ← baseline
500 steps
250 steps
100 steps

Но здесь есть важный момент: нельзя просто заменить range(999, -1, -1) на 500 шагов. Нужно сделать корректный DDIM/DDPM subsampling sampler.

Я бы это делал вторым этапом.

переходить на v-prediction или SNR weighting.

Проблема заключалась в слишком малом масштабе латентного представления относительно масштаба шума diffusion, что приводило к нестабильности обратного процесса. Для исправления латентный вектор перед diffusion был масштабирован примерно в 5,6 раза, а raw-вектор оставлен для декодирования молекулы.